# 7BZ Dev Cloud Game

Controller-first Tesla T4 cloud gaming and development console on Kaggle.

### Architecture & Storefront Suite
- **Unprivileged Execution**: Storefronts and games execute under standard user `gamer` for credential protection and clean user space.
- **Steam (Valve Native)**: Full multiarch client running with software CEF flags. License agreement presented directly to the user in the GUI.
- **Epic Games**: Heroic GUI and Legendary CLI are available for testing. Launch each installed title to verify compatibility.
- **Wine Runner**: Heroic settings select a complete Wine runner, disable UMU/DXVK/VKD3D for this container, and back up existing JSON before repair.
- **GOG Galaxy (Minigalaxy & Heroic)**: DRM-free client for GOG games. Supports native Linux builds and Wine.
- **Lutris Gaming Hub**: Multi-storefront aggregator installed from Ubuntu multiverse.
- **EA App (experimental)**: Wine installer opens; login and game execution need interactive validation.
- **7BZ Console Dashboard**: Card library and storefront hub with mouse, keyboard, and full gamepad navigation.
- **Streaming HUD**: English glass overlay with WebSocket telemetry, adaptive bitrate, browser controls, and emergency game exit.
- **Desktop Window Management**: Openbox window manager and tint2 taskbar panel for smooth multi-window multitasking.
- **Selkies 2.1.0 Streaming**: Tesla T4 NVENC hardware H.264 video encoding at 60 FPS, Opus stereo audio, and 4 virtual Xbox gamepad sockets over Kaggle signed Jupyter Server proxy.

### Hardware Reality & Constraints
- **NVENC Hardware Acceleration**: Encoding is hardware-accelerated on Tesla T4 via NVENC.
- **Desktop 3D Rendering**: VirtualGL `egl0` passes native Linux probes. For Wolfenstein, the verified Wine OpenGL path is direct NVIDIA Xorg `:1` with its own Selkies stream on port `8081`. Vulkan/DXVK remains capability-gated.

### Instructions
1. Set **Accelerator: GPU T4 x2** and **Internet: On** in the Kaggle sidebar.
2. Run all cells in an active interactive session.
3. Copy the **VSCode Compatible URL** from **Run → Kaggle Jupyter Server**, paste it into the connection cell, and connect!
4. The connection cell opens the direct NVIDIA desktop on port `8081` when available. Openbox, Legendary, and games share that display. The Xvfb desktop on port `8080` remains a fallback.


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time

BENCH = Path('/tmp/kaggle-game-dev-bench')
BUILDS = Path('/tmp/game-builds')
LOGS = BENCH / 'logs'
EA_STORE = Path('/tmp/ea-store')
REPO_URL = 'https://github.com/Mohammedsbastawy/kaggle-gaming-bench.git'

BUILDS.mkdir(parents=True, exist_ok=True)
EA_STORE.mkdir(parents=True, exist_ok=True)

# 1. Automatic recovery for apt/dpkg locks (prevents RuntimeError exit 100 on interactive sessions)
print('▶ Clearing stale apt/dpkg locks and recovering package manager...', flush=True)
for lock in ['/var/lib/dpkg/lock-frontend', '/var/lib/dpkg/lock', '/var/lib/apt/lists/lock', '/var/cache/apt/archives/lock']:
    subprocess.run(['fuser', '-k', lock], check=False)
    try:
        os.remove(lock)
    except OSError:
        pass
subprocess.run(['dpkg', '--configure', '-a'], check=False, env=dict(os.environ, DEBIAN_FRONTEND='noninteractive'))

# 2. Clone or update the ready Git repository
setup_sh = BENCH / 'setup.sh'
if not setup_sh.exists():
    shutil.rmtree(str(BENCH), ignore_errors=True)
    print(f'▶ Cloning ready gaming bench suite from {REPO_URL}...', flush=True)
    res = subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(BENCH)], timeout=60)
    if res.returncode != 0 or not setup_sh.exists():
        print('Notice: Git clone unavailable, using embedded setup suite.', flush=True)
        BENCH.mkdir(parents=True, exist_ok=True)
        setup_sh.write_text('#!/bin/bash\nset -e\n\nBENCH="/tmp/kaggle-game-dev-bench"\nLOGS="/home/gamer/logs"\nBUILDS="/tmp/game-builds"\nmkdir -p "$BENCH/logs" "$BUILDS" /tmp/ea-store\n\nexport DEBIAN_FRONTEND=noninteractive\nexport PATH="/usr/games:/usr/local/bin:/usr/bin:$PATH"\n\necho "=========================================================="\necho "    7BZ DEV CLOUD GAME BOOTSTRAPPER                       "\necho "=========================================================="\n\n# 1. Kill stale apt/dpkg locks if running in dirty interactive session\necho "▶ Checking and clearing stale apt locks..."\nfuser -k /var/lib/dpkg/lock-frontend 2>/dev/null || true\nfuser -k /var/lib/dpkg/lock 2>/dev/null || true\nfuser -k /var/lib/apt/lists/lock 2>/dev/null || true\nfuser -cache/apt/archives/lock 2>/dev/null || true\nrm -f /var/lib/dpkg/lock* /var/lib/apt/lists/lock* /var/cache/apt/archives/lock* 2>/dev/null || true\ndpkg --configure -a 2>/dev/null || true\n\n# Helper function for resilient apt install with timeout and retries\napt_install() {\n    local label="$1"\n    shift\n    echo "▶ Installing $label..."\n    local attempts=0\n    while [ $attempts -lt 3 ]; do\n        attempts=$((attempts + 1))\n        if apt-get install -y --no-install-recommends \\\n            -o DPkg::Lock::Timeout=120 \\\n            -o APT::Get::Assume-Yes=true \\\n            "$@"; then\n            return 0\n        fi\n        echo "Notice: apt install for $label failed (attempt $attempts/3). Retrying in 4s..."\n        sleep 4\n        fuser -k /var/lib/dpkg/lock-frontend 2>/dev/null || true\n        rm -f /var/lib/dpkg/lock* 2>/dev/null || true\n        dpkg --configure -a 2>/dev/null || true\n    done\n    echo "Warning: Could not install some packages for $label, continuing..."\n    return 0\n}\n\n# 2. Multiarch & repositories setup\ndpkg --add-architecture i386 || true\nif [ -f /etc/apt/sources.list.d/ubuntu.sources ]; then\n    sed -i \'s/Components: main universe/Components: main universe multiverse/g\' /etc/apt/sources.list.d/ubuntu.sources || true\nfi\n\n# Resilient apt update\napt-get update -o DPkg::Lock::Timeout=120 -qq || true\n\n# 3. Core desktop, X11, audio, and compression utilities\napt_install "core desktop" \\\n    xvfb openbox tint2 xterm pulseaudio pulseaudio-utils xauth dbus-x11 \\\n    libgl1-mesa-dri libgbm1 libpixman-1-0 libxkbcommon0 mesa-utils x11-utils vulkan-tools \\\n    xdotool wmctrl wget curl cabextract p7zip-full zstd winetricks python3-pygame python3-aiohttp\n\n# Build tools (isolated so multilib issues don\'t block core)\napt_install "build tools" git make gcc\napt-get install -y --no-install-recommends -o DPkg::Lock::Timeout=120 gcc-multilib 2>/dev/null || true\n\n# 4. Wine and complete 32-bit/64-bit graphics runtime\napt_install "wine runtime" wine wine64 winbind\napt_install "32-bit graphics runtime" \\\n    wine32:i386 libgl1:i386 libglx0:i386 libglx-mesa0:i386 libgl1-mesa-dri:i386 \\\n    mesa-vulkan-drivers:i386 libx11-6:i386 libxext6:i386 libxfixes3:i386 \\\n    libxcomposite1:i386 libxrandr2:i386 libxinerama1:i386 libxi6:i386 libasound2:i386 \\\n    libpulse0:i386 libgnutls30:i386 libvulkan1:i386\n\n# 4b. NVIDIA 3D Hardware Userspace Driver & Vulkan Provisioner\nif [ -c /dev/nvidiactl ] && [ -c /dev/nvidia0 ]; then\n    echo "▶ Initializing NVIDIA Tesla T4 userspace driver and Vulkan ICD..."\n    if [ -f "$BENCH/probe_nvidia_userspace.sh" ]; then\n        bash "$BENCH/probe_nvidia_userspace.sh" || true\n    fi\n    work="$BENCH/nvidia-userspace-probe"\n    if [ -f "$work/env.sh" ]; then\n        # shellcheck disable=SC1090\n        . "$work/env.sh"\n    fi\nfi\n\n# 4c. VirtualGL 3.1.5 installation\nif [ -c /dev/nvidiactl ] && [ -c /dev/nvidia0 ]; then\n    if ! [ -x /opt/VirtualGL/bin/vglrun ]; then\n        VGL64=/tmp/virtualgl_3.1.5_amd64.deb\n        VGL32=/tmp/virtualgl32_3.1.5_amd64.deb\n        wget -q https://github.com/VirtualGL/virtualgl/releases/download/3.1.5/virtualgl_3.1.5_amd64.deb -O "$VGL64" || true\n        wget -q https://github.com/VirtualGL/virtualgl/releases/download/3.1.5/virtualgl32_3.1.5_amd64.deb -O "$VGL32" || true\n        if echo \'df3f7788ce41b182a47c0d298e5cd6d2d63579522cb41825970b7726e825485e  /tmp/virtualgl_3.1.5_amd64.deb\' | sha256sum -c - && \\\n           echo \'7dc63ecb47d7ab9cb486bb21a9a968dee853380da760cc7f36f616d09ec62b9e  /tmp/virtualgl32_3.1.5_amd64.deb\' | sha256sum -c -; then\n            apt-get install -y --no-install-recommends "$VGL64" "$VGL32" || true\n        else\n            echo \'VirtualGL download or checksum failed; GPU GL bridge unavailable.\'\n        fi\n    fi\nelse\n    echo \'NVIDIA GPU devices unavailable; GPU 3D cannot be enabled inside this container.\'\nfi\nif [ -c /dev/nvidia-modeset ] && [ -c /dev/nvidia0 ]; then\n    # The Xorg binary alone cannot load NVIDIA GLX. Install the matching\n    # display driver from the same repository as the NVIDIA userspace stack.\n    nvidia_userspace_version=$(dpkg-query -W -f=\'${Version}\' libnvidia-gl-580 2>/dev/null || true)\n    if [ -n "$nvidia_userspace_version" ]; then\n        apt_install "NVIDIA Xorg GLX server support" xserver-xorg-core \\\n            "xserver-xorg-video-nvidia-580=$nvidia_userspace_version" \\\n            "libnvidia-cfg1-580=$nvidia_userspace_version"\n    else\n        echo \'NVIDIA GL userspace is missing; skipping the optional Xorg backend.\'\n    fi\nfi\n\n# 4d. DXVK 2.6.2 (DirectX 9/10/11 -> Vulkan) installation\nDXVK_TAR=/tmp/dxvk-2.6.2.tar.gz\nif [ ! -d /opt/gamelab-dxvk/dxvk-2.6.2 ]; then\n    echo "▶ Downloading and extracting DXVK 2.6.2 Vulkan translation layer..."\n    wget -q https://github.com/doitsujin/dxvk/releases/download/v2.6.2/dxvk-2.6.2.tar.gz -O "$DXVK_TAR" || true\n    if echo \'17761876556afd55736cb895d184f5a1c55d43350f1b1e3b129f8d28706d7992  /tmp/dxvk-2.6.2.tar.gz\' | sha256sum -c -; then\n        mkdir -p /opt/gamelab-dxvk\n        tar -xzf "$DXVK_TAR" -C /opt/gamelab-dxvk || true\n    else\n        echo \'Warning: DXVK download or checksum failed; DXVK unavailable.\'\n    fi\nfi\n\n# 4e. VKD3D-Proton 2.14.1 (DirectX 12 -> Vulkan) installation for modern AAA games\nVKD3D_TAR=/tmp/vkd3d-proton-2.14.1.tar.zst\nif [ ! -d /opt/gamelab-vkd3d ]; then\n    echo "▶ Downloading VKD3D-Proton (DirectX 12) translation layer for modern AAA games..."\n    wget -q https://github.com/HansKristian-Work/vkd3d-proton/releases/download/v2.14.1/vkd3d-proton-2.14.1.tar.zst -O "$VKD3D_TAR" || true\n    if [ -f "$VKD3D_TAR" ] && [ $(stat -c%s "$VKD3D_TAR" 2>/dev/null || echo 0) -gt 1000000 ]; then\n        mkdir -p /opt/gamelab-vkd3d\n        tar --zstd -xf "$VKD3D_TAR" -C /opt/gamelab-vkd3d 2>/dev/null || \\\n        tar -xf "$VKD3D_TAR" -C /opt/gamelab-vkd3d 2>/dev/null || true\n    fi\nfi\n\n# 5. Extra libraries for Lutris and Wine Gecko\napt_install "extra libraries" python3-tk python3.12 python3-dbus python3-gi python3-yaml python3-lxml python3-requests python3-pil\napt-get install -y --no-install-recommends -o DPkg::Lock::Timeout=120 wine-gecko 2>/dev/null || true\n\n# 6. Python streaming, interface and gaming packages\npython3 -m pip install --no-cache-dir \\\n    \'selkies[jupyter]==2.1.0\' \'pygame==2.6.1\' \'psutil\' \'legendary-gl\'\n\n# 7. Steam installation (Valve official deb)\nSTEAM_DEB="/tmp/steam_latest.deb"\nif [ ! -f "$STEAM_DEB" ] || [ $(stat -c%s "$STEAM_DEB" 2>/dev/null || echo 0) -lt 1000000 ]; then\n    wget -q https://repo.steampowered.com/steam/archive/stable/steam_latest.deb -O "$STEAM_DEB" || true\nfi\nif [ -f "$STEAM_DEB" ] && [ $(stat -c%s "$STEAM_DEB") -gt 1000000 ]; then\n    apt_install "steam deb" "$STEAM_DEB"\nelse\n    apt_install "steam apt" steam\nfi\n\n# 8. Heroic Games Launcher\nHEROIC_DEB="/tmp/heroic.deb"\nif [ ! -f "$HEROIC_DEB" ] || [ $(stat -c%s "$HEROIC_DEB" 2>/dev/null || echo 0) -lt 10000000 ]; then\n    wget -q https://github.com/Heroic-Games-Launcher/HeroicGamesLauncher/releases/download/v2.22.4/Heroic-2.22.4-linux-amd64.deb -O "$HEROIC_DEB" || true\nfi\nif [ -f "$HEROIC_DEB" ] && [ $(stat -c%s "$HEROIC_DEB") -gt 10000000 ]; then\n    dpkg -i "$HEROIC_DEB" || apt-get install -f -y -o DPkg::Lock::Timeout=120 || true\nfi\n\n# 9. Lutris & Minigalaxy\napt_install "lutris" lutris\napt_install "minigalaxy" minigalaxy\n\n# 10. EA App installer download\nEA_EXE="/tmp/ea-store/EAappInstaller.exe"\nif [ ! -f "$EA_EXE" ] || [ $(stat -c%s "$EA_EXE" 2>/dev/null || echo 0) -lt 1000000 ]; then\n    wget -q https://origin-a.akamaihd.net/EA-Desktop-Client-Download/installer-releases/EAappInstaller.exe -O "$EA_EXE" || true\nfi\n\n# 11. Configure unprivileged gamer user\nid -u gamer >/dev/null 2>&1 || useradd -m -s /bin/bash -G audio,video gamer\nfor sub in .config .local .steam game-builds logs ea-prefix; do\n    mkdir -p "/home/gamer/$sub"\ndone\n\n# 12. Wine-GE runner installation for Heroic & Lutris\nWINE_GE_DIR="/home/gamer/.config/heroic/tools/wine"\nmkdir -p "$WINE_GE_DIR"\nWINE_GE_TAR="/tmp/wine-lutris-GE-Proton8-26-x86_64.tar.xz"\nif [ ! -d "$WINE_GE_DIR/Wine-GE-Proton8-26" ] && [ ! -d "$WINE_GE_DIR/lutris-GE-Proton8-26-x86_64" ]; then\n    echo "▶ Downloading Wine-GE-Proton8-26 compatibility runner..."\n    wget -q -c https://github.com/GloriousEggroll/wine-ge-custom/releases/download/GE-Proton8-26/wine-lutris-GE-Proton8-26-x86_64.tar.xz -O "$WINE_GE_TAR" || true\n    if [ -f "$WINE_GE_TAR" ] && [ $(stat -c%s "$WINE_GE_TAR") -gt 50000000 ]; then\n        echo "▶ Extracting Wine-GE-Proton8-26..."\n        tar -xf "$WINE_GE_TAR" -C "$WINE_GE_DIR" || true\n        for extracted in "$WINE_GE_DIR"/*; do\n            if [ -d "$extracted" ] && [ -f "$extracted/bin/wine" ]; then\n                ln -sf "$extracted" "$WINE_GE_DIR/Wine-GE-Proton8-26" || true\n                mkdir -p "/home/gamer/.local/share/lutris/runners/wine"\n                ln -sf "$extracted" "/home/gamer/.local/share/lutris/runners/wine/Wine-GE-Proton8-26" || true\n                if [ ! -f /usr/bin/wine ]; then\n                    ln -sf "$extracted/bin/wine" /usr/bin/wine || true\n                    ln -sf "$(dirname "$extracted/bin/wine")/wineserver" /usr/bin/wineserver || true\n                    ln -sf "$(dirname "$extracted/bin/wine")/wineboot" /usr/bin/wineboot || true\n                fi\n                if [ ! -f /usr/local/bin/wine ]; then\n                    ln -sf "$extracted/bin/wine" /usr/local/bin/wine || true\n                fi\n                break\n            fi\n        done\n    fi\nfi\n\n# 13. Pre-configure Heroic default settings\npython3 "$BENCH/fix_heroic_config.py" || true\n\n# 14. Pre-initialize clean 64-bit Wine prefix with Wine-GE\nif [ -x "$WINE_GE_DIR/Wine-GE-Proton8-26/bin/wineboot" ]; then\n    echo "▶ Initializing clean default Wine prefix (/home/gamer/.wine)..."\n    export WINEPREFIX="/home/gamer/.wine"\n    export WINEARCH=win64\n    export WINEDLLOVERRIDES="mscoree,mshtml=d"\n    runuser -u gamer -- env WINEPREFIX="/home/gamer/.wine" WINEARCH=win64 WINEDLLOVERRIDES="mscoree,mshtml=d" \\\n        "$WINE_GE_DIR/Wine-GE-Proton8-26/bin/wineboot" --init 2>/dev/null || true\n\n    # Pre-install DXVK and VKD3D DirectX-to-Vulkan libraries into Wine prefix\n    echo "▶ Installing DXVK and VKD3D DirectX-to-Vulkan libraries into Wine prefix..."\n    if [ -d /opt/gamelab-dxvk/dxvk-2.6.2 ]; then\n        for arch_pair in "x64:system32" "x32:syswow64"; do\n            IFS=\':\' read -r darch dsub <<< "$arch_pair"\n            for dll in d3d9.dll d3d10core.dll d3d11.dll dxgi.dll; do\n                src="/opt/gamelab-dxvk/dxvk-2.6.2/$darch/$dll"\n                dst="/home/gamer/.wine/drive_c/windows/$dsub/$dll"\n                [ -f "$src" ] && cp -f "$src" "$dst" 2>/dev/null || true\n            done\n        done\n        echo "✓ DXVK installed into Wine prefix."\n    fi\n    for vkd3d_dir in /opt/gamelab-vkd3d/vkd3d-proton-*; do\n        if [ -d "$vkd3d_dir" ]; then\n            for arch_pair in "x64:system32" "x86:syswow64"; do\n                IFS=\':\' read -r darch dsub <<< "$arch_pair"\n                for dll in d3d12.dll d3d12core.dll; do\n                    src="$vkd3d_dir/$darch/$dll"\n                    dst="/home/gamer/.wine/drive_c/windows/$dsub/$dll"\n                    [ -f "$src" ] && cp -f "$src" "$dst" 2>/dev/null || true\n                done\n            done\n            echo "✓ VKD3D-Proton (DirectX 12) installed into Wine prefix."\n            break\n        fi\n    done\n\n    # Pre-install Microsoft Visual C++ 2015-2022 runtimes (x64 and x86) into Wine prefix\n    echo "▶ Pre-installing Microsoft Visual C++ runtimes into Wine prefix..."\n    REDIST_DIR="/tmp/vc_redist"\n    mkdir -p "$REDIST_DIR"\n    wget -q -c "https://aka.ms/vs/17/release/vc_redist.x64.exe" -O "$REDIST_DIR/vc_redist.x64.exe" || true\n    wget -q -c "https://aka.ms/vs/17/release/vc_redist.x86.exe" -O "$REDIST_DIR/vc_redist.x86.exe" || true\n\n    if [ -f "$REDIST_DIR/vc_redist.x64.exe" ]; then\n        runuser -u gamer -- env WINEPREFIX="/home/gamer/.wine" WINEARCH=win64 WINEDLLOVERRIDES="mscoree,mshtml=d" \\\n            "$WINE_GE_DIR/Wine-GE-Proton8-26/bin/wine" "$REDIST_DIR/vc_redist.x64.exe" /install /quiet /norestart 2>/dev/null || true\n    fi\n    if [ -f "$REDIST_DIR/vc_redist.x86.exe" ]; then\n        runuser -u gamer -- env WINEPREFIX="/home/gamer/.wine" WINEARCH=win64 WINEDLLOVERRIDES="mscoree,mshtml=d" \\\n            "$WINE_GE_DIR/Wine-GE-Proton8-26/bin/wine" "$REDIST_DIR/vc_redist.x86.exe" /install /quiet /norestart 2>/dev/null || true\n    fi\n    echo "▶ Installing the Unreal Engine Visual C++ prerequisite..."\n    runuser -u gamer -- env WINEPREFIX="/home/gamer/.wine" WINEARCH=win64 WINEDLLOVERRIDES="mscoree,mshtml=d" \\\n        winetricks -q vcrun2019 2>/dev/null || true\n\n    echo "▶ Applying verified DirectInput, AMD FSR, and zero-acceleration mouse settings..."\n    for reg_entry in \\\n        \'HKEY_CURRENT_USER\\Software\\Wine\\DirectInput|MouseWarpOverride|force\' \\\n        \'HKEY_CURRENT_USER\\Software\\Wine\\X11 Driver|GrabPointer|Y\' \\\n        \'HKEY_CURRENT_USER\\Software\\Wine\\X11 Driver|UseTakeFocus|N\' \\\n        \'HKEY_CURRENT_USER\\Control Panel\\Mouse|MouseSpeed|0\' \\\n        \'HKEY_CURRENT_USER\\Control Panel\\Mouse|MouseThreshold1|0\' \\\n        \'HKEY_CURRENT_USER\\Control Panel\\Mouse|MouseThreshold2|0\' \\\n        \'HKEY_CURRENT_USER\\Control Panel\\Mouse|MouseSensitivity|10\' \\\n        \'HKEY_CURRENT_USER\\Software\\Wine\\Direct3D|WINE_FULLSCREEN_FSR|1\' \\\n        \'HKEY_CURRENT_USER\\Software\\Wine\\Direct3D|WINE_FULLSCREEN_FSR_STRENGTH|2\' \\\n        \'HKEY_CURRENT_USER\\Software\\Wine\\DllOverrides|d3d11|native,builtin\' \\\n        \'HKEY_CURRENT_USER\\Software\\Wine\\DllOverrides|d3d12|native,builtin\' \\\n        \'HKEY_CURRENT_USER\\Software\\Wine\\DllOverrides|dxgi|native,builtin\'; do\n        IFS=\'|\' read -r rkey rname rval <<< "$reg_entry"\n        runuser -u gamer -- env WINEPREFIX="/home/gamer/.wine" WINEARCH=win64 \\\n            "$WINE_GE_DIR/Wine-GE-Proton8-26/bin/wine" reg add "$rkey" \\\n            /v "$rname" /t REG_SZ /d "$rval" /f 2>/dev/null || true\n    done\n    echo "✓ Wine prefix fully initialized with DXVK, VKD3D-Proton, and FSR upscaling."\nfi\n\n# 15. Inject keyboard lock into Selkies HTML (prevents Esc from exiting browser fullscreen)\npython3 -c "\nimport sys; sys.path.insert(0, \'$BENCH\')\ntry:\n    from auto_optimize import inject_keyboard_lock\n    result = inject_keyboard_lock()\n    if result: print(\'Keyboard lock injected into:\', result)\n    else: print(\'No Selkies HTML found to patch for keyboard lock.\')\nexcept Exception as e: print(\'Keyboard lock injection skipped:\', e)\n" || true\n\n# 16. Compile Selkies gamepad interposers if not present\nif [ ! -f "$BENCH/lib/x86_64-linux-gnu/selkies_input_interposer.so" ]; then\n    SOURCE="$BENCH/selkies-source"\n    if [ ! -d "$SOURCE" ]; then\n        git clone --depth 1 --branch 2.1.0 https://github.com/selkies-project/selkies.git "$SOURCE" || true\n    fi\n    if [ -d "$SOURCE/addons/input-interposer" ]; then\n        make -C "$SOURCE/addons/input-interposer" PREFIX="$BENCH" install || true\n        make -C "$SOURCE/addons/input-interposer" PREFIX="$BENCH" install32 2>/dev/null || true\n        make -C "$SOURCE/addons/fake-udev" || true\n        make -C "$SOURCE/addons/fake-udev" all32 2>/dev/null || true\n        mkdir -p "$BENCH/lib/x86_64-linux-gnu" "$BENCH/lib/i386-linux-gnu"\n        cp -f "$SOURCE/addons/fake-udev/libudev.so.1" "$BENCH/lib/x86_64-linux-gnu/selkies_fake_udev.so" 2>/dev/null || true\n        cp -f "$SOURCE/addons/fake-udev/libudev_x86.so.1" "$BENCH/lib/i386-linux-gnu/selkies_fake_udev.so" 2>/dev/null || true\n    fi\n    ln -sf "$BENCH/lib/x86_64-linux-gnu" "$BENCH/lib64" 2>/dev/null || true\n    ln -sf "$BENCH/lib/i386-linux-gnu" "$BENCH/lib32" 2>/dev/null || true\n    cp -f "$BENCH/lib/x86_64-linux-gnu/"*.so /usr/lib/x86_64-linux-gnu/ 2>/dev/null || true\n    cp -f "$BENCH/lib/i386-linux-gnu/"*.so /usr/lib/i386-linux-gnu/ 2>/dev/null || true\nfi\n\n# 17. Prepare /dev/input gamepad device nodes and permissions\nmkdir -p /dev/input 2>/dev/null || true\nfor i in 0 1 2 3; do touch /dev/input/js$i 2>/dev/null || true; done\nfor i in 1000 1001 1002 1003; do touch /dev/input/event$i 2>/dev/null || true; done\nchmod -R 666 /dev/input 2>/dev/null || true\n\n# 18. Set permissions and ownership\nchmod -R 755 "$BENCH"/*.sh 2>/dev/null || true\nchmod -R 777 "$BENCH" 2>/dev/null || true\nchown -R gamer:gamer /home/gamer 2>/dev/null || true\nchmod -R 777 /home/gamer 2>/dev/null || true\n\necho "=========================================================="\necho "    BENCH ENVIRONMENT SETUP COMPLETED SUCCESSFULLY!       "\necho "=========================================================="\n', encoding='utf-8')
else:
    print('▶ Updating bench repository from git...', flush=True)
    subprocess.run(['git', '-C', str(BENCH), 'reset', '--hard', 'HEAD'], check=False, timeout=30)
    subprocess.run(['git', '-C', str(BENCH), 'clean', '-fd'], check=False, timeout=30)
    subprocess.run(['git', '-C', str(BENCH), 'pull', '--force'], check=False, timeout=30)

# Use the setup bundled with this notebook, including its GPU driver probes.
# The public repository can lag behind a newly uploaded notebook version.
setup_sh.write_text('#!/bin/bash\nset -e\n\nBENCH="/tmp/kaggle-game-dev-bench"\nLOGS="/home/gamer/logs"\nBUILDS="/tmp/game-builds"\nmkdir -p "$BENCH/logs" "$BUILDS" /tmp/ea-store\n\nexport DEBIAN_FRONTEND=noninteractive\nexport PATH="/usr/games:/usr/local/bin:/usr/bin:$PATH"\n\necho "=========================================================="\necho "    7BZ DEV CLOUD GAME BOOTSTRAPPER                       "\necho "=========================================================="\n\n# 1. Kill stale apt/dpkg locks if running in dirty interactive session\necho "▶ Checking and clearing stale apt locks..."\nfuser -k /var/lib/dpkg/lock-frontend 2>/dev/null || true\nfuser -k /var/lib/dpkg/lock 2>/dev/null || true\nfuser -k /var/lib/apt/lists/lock 2>/dev/null || true\nfuser -cache/apt/archives/lock 2>/dev/null || true\nrm -f /var/lib/dpkg/lock* /var/lib/apt/lists/lock* /var/cache/apt/archives/lock* 2>/dev/null || true\ndpkg --configure -a 2>/dev/null || true\n\n# Helper function for resilient apt install with timeout and retries\napt_install() {\n    local label="$1"\n    shift\n    echo "▶ Installing $label..."\n    local attempts=0\n    while [ $attempts -lt 3 ]; do\n        attempts=$((attempts + 1))\n        if apt-get install -y --no-install-recommends \\\n            -o DPkg::Lock::Timeout=120 \\\n            -o APT::Get::Assume-Yes=true \\\n            "$@"; then\n            return 0\n        fi\n        echo "Notice: apt install for $label failed (attempt $attempts/3). Retrying in 4s..."\n        sleep 4\n        fuser -k /var/lib/dpkg/lock-frontend 2>/dev/null || true\n        rm -f /var/lib/dpkg/lock* 2>/dev/null || true\n        dpkg --configure -a 2>/dev/null || true\n    done\n    echo "Warning: Could not install some packages for $label, continuing..."\n    return 0\n}\n\n# 2. Multiarch & repositories setup\ndpkg --add-architecture i386 || true\nif [ -f /etc/apt/sources.list.d/ubuntu.sources ]; then\n    sed -i \'s/Components: main universe/Components: main universe multiverse/g\' /etc/apt/sources.list.d/ubuntu.sources || true\nfi\n\n# Resilient apt update\napt-get update -o DPkg::Lock::Timeout=120 -qq || true\n\n# 3. Core desktop, X11, audio, and compression utilities\napt_install "core desktop" \\\n    xvfb openbox tint2 xterm pulseaudio pulseaudio-utils xauth dbus-x11 \\\n    libgl1-mesa-dri libgbm1 libpixman-1-0 libxkbcommon0 mesa-utils x11-utils vulkan-tools \\\n    xdotool wmctrl wget curl cabextract p7zip-full zstd winetricks python3-pygame python3-aiohttp\n\n# Build tools (isolated so multilib issues don\'t block core)\napt_install "build tools" git make gcc\napt-get install -y --no-install-recommends -o DPkg::Lock::Timeout=120 gcc-multilib 2>/dev/null || true\n\n# 4. Wine and complete 32-bit/64-bit graphics runtime\napt_install "wine runtime" wine wine64 winbind\napt_install "32-bit graphics runtime" \\\n    wine32:i386 libgl1:i386 libglx0:i386 libglx-mesa0:i386 libgl1-mesa-dri:i386 \\\n    mesa-vulkan-drivers:i386 libx11-6:i386 libxext6:i386 libxfixes3:i386 \\\n    libxcomposite1:i386 libxrandr2:i386 libxinerama1:i386 libxi6:i386 libasound2:i386 \\\n    libpulse0:i386 libgnutls30:i386 libvulkan1:i386\n\n# 4b. NVIDIA 3D Hardware Userspace Driver & Vulkan Provisioner\nif [ -c /dev/nvidiactl ] && [ -c /dev/nvidia0 ]; then\n    echo "▶ Initializing NVIDIA Tesla T4 userspace driver and Vulkan ICD..."\n    if [ -f "$BENCH/probe_nvidia_userspace.sh" ]; then\n        bash "$BENCH/probe_nvidia_userspace.sh" || true\n    fi\n    work="$BENCH/nvidia-userspace-probe"\n    if [ -f "$work/env.sh" ]; then\n        # shellcheck disable=SC1090\n        . "$work/env.sh"\n    fi\nfi\n\n# 4c. VirtualGL 3.1.5 installation\nif [ -c /dev/nvidiactl ] && [ -c /dev/nvidia0 ]; then\n    if ! [ -x /opt/VirtualGL/bin/vglrun ]; then\n        VGL64=/tmp/virtualgl_3.1.5_amd64.deb\n        VGL32=/tmp/virtualgl32_3.1.5_amd64.deb\n        wget -q https://github.com/VirtualGL/virtualgl/releases/download/3.1.5/virtualgl_3.1.5_amd64.deb -O "$VGL64" || true\n        wget -q https://github.com/VirtualGL/virtualgl/releases/download/3.1.5/virtualgl32_3.1.5_amd64.deb -O "$VGL32" || true\n        if echo \'df3f7788ce41b182a47c0d298e5cd6d2d63579522cb41825970b7726e825485e  /tmp/virtualgl_3.1.5_amd64.deb\' | sha256sum -c - && \\\n           echo \'7dc63ecb47d7ab9cb486bb21a9a968dee853380da760cc7f36f616d09ec62b9e  /tmp/virtualgl32_3.1.5_amd64.deb\' | sha256sum -c -; then\n            apt-get install -y --no-install-recommends "$VGL64" "$VGL32" || true\n        else\n            echo \'VirtualGL download or checksum failed; GPU GL bridge unavailable.\'\n        fi\n    fi\nelse\n    echo \'NVIDIA GPU devices unavailable; GPU 3D cannot be enabled inside this container.\'\nfi\nif [ -c /dev/nvidia-modeset ] && [ -c /dev/nvidia0 ]; then\n    # The Xorg binary alone cannot load NVIDIA GLX. Install the matching\n    # display driver from the same repository as the NVIDIA userspace stack.\n    nvidia_userspace_version=$(dpkg-query -W -f=\'${Version}\' libnvidia-gl-580 2>/dev/null || true)\n    if [ -n "$nvidia_userspace_version" ]; then\n        apt_install "NVIDIA Xorg GLX server support" xserver-xorg-core \\\n            "xserver-xorg-video-nvidia-580=$nvidia_userspace_version" \\\n            "libnvidia-cfg1-580=$nvidia_userspace_version"\n    else\n        echo \'NVIDIA GL userspace is missing; skipping the optional Xorg backend.\'\n    fi\nfi\n\n# 4d. DXVK 2.6.2 (DirectX 9/10/11 -> Vulkan) installation\nDXVK_TAR=/tmp/dxvk-2.6.2.tar.gz\nif [ ! -d /opt/gamelab-dxvk/dxvk-2.6.2 ]; then\n    echo "▶ Downloading and extracting DXVK 2.6.2 Vulkan translation layer..."\n    wget -q https://github.com/doitsujin/dxvk/releases/download/v2.6.2/dxvk-2.6.2.tar.gz -O "$DXVK_TAR" || true\n    if echo \'17761876556afd55736cb895d184f5a1c55d43350f1b1e3b129f8d28706d7992  /tmp/dxvk-2.6.2.tar.gz\' | sha256sum -c -; then\n        mkdir -p /opt/gamelab-dxvk\n        tar -xzf "$DXVK_TAR" -C /opt/gamelab-dxvk || true\n    else\n        echo \'Warning: DXVK download or checksum failed; DXVK unavailable.\'\n    fi\nfi\n\n# 4e. VKD3D-Proton 2.14.1 (DirectX 12 -> Vulkan) installation for modern AAA games\nVKD3D_TAR=/tmp/vkd3d-proton-2.14.1.tar.zst\nif [ ! -d /opt/gamelab-vkd3d ]; then\n    echo "▶ Downloading VKD3D-Proton (DirectX 12) translation layer for modern AAA games..."\n    wget -q https://github.com/HansKristian-Work/vkd3d-proton/releases/download/v2.14.1/vkd3d-proton-2.14.1.tar.zst -O "$VKD3D_TAR" || true\n    if [ -f "$VKD3D_TAR" ] && [ $(stat -c%s "$VKD3D_TAR" 2>/dev/null || echo 0) -gt 1000000 ]; then\n        mkdir -p /opt/gamelab-vkd3d\n        tar --zstd -xf "$VKD3D_TAR" -C /opt/gamelab-vkd3d 2>/dev/null || \\\n        tar -xf "$VKD3D_TAR" -C /opt/gamelab-vkd3d 2>/dev/null || true\n    fi\nfi\n\n# 5. Extra libraries for Lutris and Wine Gecko\napt_install "extra libraries" python3-tk python3.12 python3-dbus python3-gi python3-yaml python3-lxml python3-requests python3-pil\napt-get install -y --no-install-recommends -o DPkg::Lock::Timeout=120 wine-gecko 2>/dev/null || true\n\n# 6. Python streaming, interface and gaming packages\npython3 -m pip install --no-cache-dir \\\n    \'selkies[jupyter]==2.1.0\' \'pygame==2.6.1\' \'psutil\' \'legendary-gl\'\n\n# 7. Steam installation (Valve official deb)\nSTEAM_DEB="/tmp/steam_latest.deb"\nif [ ! -f "$STEAM_DEB" ] || [ $(stat -c%s "$STEAM_DEB" 2>/dev/null || echo 0) -lt 1000000 ]; then\n    wget -q https://repo.steampowered.com/steam/archive/stable/steam_latest.deb -O "$STEAM_DEB" || true\nfi\nif [ -f "$STEAM_DEB" ] && [ $(stat -c%s "$STEAM_DEB") -gt 1000000 ]; then\n    apt_install "steam deb" "$STEAM_DEB"\nelse\n    apt_install "steam apt" steam\nfi\n\n# 8. Heroic Games Launcher\nHEROIC_DEB="/tmp/heroic.deb"\nif [ ! -f "$HEROIC_DEB" ] || [ $(stat -c%s "$HEROIC_DEB" 2>/dev/null || echo 0) -lt 10000000 ]; then\n    wget -q https://github.com/Heroic-Games-Launcher/HeroicGamesLauncher/releases/download/v2.22.4/Heroic-2.22.4-linux-amd64.deb -O "$HEROIC_DEB" || true\nfi\nif [ -f "$HEROIC_DEB" ] && [ $(stat -c%s "$HEROIC_DEB") -gt 10000000 ]; then\n    dpkg -i "$HEROIC_DEB" || apt-get install -f -y -o DPkg::Lock::Timeout=120 || true\nfi\n\n# 9. Lutris & Minigalaxy\napt_install "lutris" lutris\napt_install "minigalaxy" minigalaxy\n\n# 10. EA App installer download\nEA_EXE="/tmp/ea-store/EAappInstaller.exe"\nif [ ! -f "$EA_EXE" ] || [ $(stat -c%s "$EA_EXE" 2>/dev/null || echo 0) -lt 1000000 ]; then\n    wget -q https://origin-a.akamaihd.net/EA-Desktop-Client-Download/installer-releases/EAappInstaller.exe -O "$EA_EXE" || true\nfi\n\n# 11. Configure unprivileged gamer user\nid -u gamer >/dev/null 2>&1 || useradd -m -s /bin/bash -G audio,video gamer\nfor sub in .config .local .steam game-builds logs ea-prefix; do\n    mkdir -p "/home/gamer/$sub"\ndone\n\n# 12. Wine-GE runner installation for Heroic & Lutris\nWINE_GE_DIR="/home/gamer/.config/heroic/tools/wine"\nmkdir -p "$WINE_GE_DIR"\nWINE_GE_TAR="/tmp/wine-lutris-GE-Proton8-26-x86_64.tar.xz"\nif [ ! -d "$WINE_GE_DIR/Wine-GE-Proton8-26" ] && [ ! -d "$WINE_GE_DIR/lutris-GE-Proton8-26-x86_64" ]; then\n    echo "▶ Downloading Wine-GE-Proton8-26 compatibility runner..."\n    wget -q -c https://github.com/GloriousEggroll/wine-ge-custom/releases/download/GE-Proton8-26/wine-lutris-GE-Proton8-26-x86_64.tar.xz -O "$WINE_GE_TAR" || true\n    if [ -f "$WINE_GE_TAR" ] && [ $(stat -c%s "$WINE_GE_TAR") -gt 50000000 ]; then\n        echo "▶ Extracting Wine-GE-Proton8-26..."\n        tar -xf "$WINE_GE_TAR" -C "$WINE_GE_DIR" || true\n        for extracted in "$WINE_GE_DIR"/*; do\n            if [ -d "$extracted" ] && [ -f "$extracted/bin/wine" ]; then\n                ln -sf "$extracted" "$WINE_GE_DIR/Wine-GE-Proton8-26" || true\n                mkdir -p "/home/gamer/.local/share/lutris/runners/wine"\n                ln -sf "$extracted" "/home/gamer/.local/share/lutris/runners/wine/Wine-GE-Proton8-26" || true\n                if [ ! -f /usr/bin/wine ]; then\n                    ln -sf "$extracted/bin/wine" /usr/bin/wine || true\n                    ln -sf "$(dirname "$extracted/bin/wine")/wineserver" /usr/bin/wineserver || true\n                    ln -sf "$(dirname "$extracted/bin/wine")/wineboot" /usr/bin/wineboot || true\n                fi\n                if [ ! -f /usr/local/bin/wine ]; then\n                    ln -sf "$extracted/bin/wine" /usr/local/bin/wine || true\n                fi\n                break\n            fi\n        done\n    fi\nfi\n\n# 13. Pre-configure Heroic default settings\npython3 "$BENCH/fix_heroic_config.py" || true\n\n# 14. Pre-initialize clean 64-bit Wine prefix with Wine-GE\nif [ -x "$WINE_GE_DIR/Wine-GE-Proton8-26/bin/wineboot" ]; then\n    echo "▶ Initializing clean default Wine prefix (/home/gamer/.wine)..."\n    export WINEPREFIX="/home/gamer/.wine"\n    export WINEARCH=win64\n    export WINEDLLOVERRIDES="mscoree,mshtml=d"\n    runuser -u gamer -- env WINEPREFIX="/home/gamer/.wine" WINEARCH=win64 WINEDLLOVERRIDES="mscoree,mshtml=d" \\\n        "$WINE_GE_DIR/Wine-GE-Proton8-26/bin/wineboot" --init 2>/dev/null || true\n\n    # Pre-install DXVK and VKD3D DirectX-to-Vulkan libraries into Wine prefix\n    echo "▶ Installing DXVK and VKD3D DirectX-to-Vulkan libraries into Wine prefix..."\n    if [ -d /opt/gamelab-dxvk/dxvk-2.6.2 ]; then\n        for arch_pair in "x64:system32" "x32:syswow64"; do\n            IFS=\':\' read -r darch dsub <<< "$arch_pair"\n            for dll in d3d9.dll d3d10core.dll d3d11.dll dxgi.dll; do\n                src="/opt/gamelab-dxvk/dxvk-2.6.2/$darch/$dll"\n                dst="/home/gamer/.wine/drive_c/windows/$dsub/$dll"\n                [ -f "$src" ] && cp -f "$src" "$dst" 2>/dev/null || true\n            done\n        done\n        echo "✓ DXVK installed into Wine prefix."\n    fi\n    for vkd3d_dir in /opt/gamelab-vkd3d/vkd3d-proton-*; do\n        if [ -d "$vkd3d_dir" ]; then\n            for arch_pair in "x64:system32" "x86:syswow64"; do\n                IFS=\':\' read -r darch dsub <<< "$arch_pair"\n                for dll in d3d12.dll d3d12core.dll; do\n                    src="$vkd3d_dir/$darch/$dll"\n                    dst="/home/gamer/.wine/drive_c/windows/$dsub/$dll"\n                    [ -f "$src" ] && cp -f "$src" "$dst" 2>/dev/null || true\n                done\n            done\n            echo "✓ VKD3D-Proton (DirectX 12) installed into Wine prefix."\n            break\n        fi\n    done\n\n    # Pre-install Microsoft Visual C++ 2015-2022 runtimes (x64 and x86) into Wine prefix\n    echo "▶ Pre-installing Microsoft Visual C++ runtimes into Wine prefix..."\n    REDIST_DIR="/tmp/vc_redist"\n    mkdir -p "$REDIST_DIR"\n    wget -q -c "https://aka.ms/vs/17/release/vc_redist.x64.exe" -O "$REDIST_DIR/vc_redist.x64.exe" || true\n    wget -q -c "https://aka.ms/vs/17/release/vc_redist.x86.exe" -O "$REDIST_DIR/vc_redist.x86.exe" || true\n\n    if [ -f "$REDIST_DIR/vc_redist.x64.exe" ]; then\n        runuser -u gamer -- env WINEPREFIX="/home/gamer/.wine" WINEARCH=win64 WINEDLLOVERRIDES="mscoree,mshtml=d" \\\n            "$WINE_GE_DIR/Wine-GE-Proton8-26/bin/wine" "$REDIST_DIR/vc_redist.x64.exe" /install /quiet /norestart 2>/dev/null || true\n    fi\n    if [ -f "$REDIST_DIR/vc_redist.x86.exe" ]; then\n        runuser -u gamer -- env WINEPREFIX="/home/gamer/.wine" WINEARCH=win64 WINEDLLOVERRIDES="mscoree,mshtml=d" \\\n            "$WINE_GE_DIR/Wine-GE-Proton8-26/bin/wine" "$REDIST_DIR/vc_redist.x86.exe" /install /quiet /norestart 2>/dev/null || true\n    fi\n    echo "▶ Installing the Unreal Engine Visual C++ prerequisite..."\n    runuser -u gamer -- env WINEPREFIX="/home/gamer/.wine" WINEARCH=win64 WINEDLLOVERRIDES="mscoree,mshtml=d" \\\n        winetricks -q vcrun2019 2>/dev/null || true\n\n    echo "▶ Applying verified DirectInput, AMD FSR, and zero-acceleration mouse settings..."\n    for reg_entry in \\\n        \'HKEY_CURRENT_USER\\Software\\Wine\\DirectInput|MouseWarpOverride|force\' \\\n        \'HKEY_CURRENT_USER\\Software\\Wine\\X11 Driver|GrabPointer|Y\' \\\n        \'HKEY_CURRENT_USER\\Software\\Wine\\X11 Driver|UseTakeFocus|N\' \\\n        \'HKEY_CURRENT_USER\\Control Panel\\Mouse|MouseSpeed|0\' \\\n        \'HKEY_CURRENT_USER\\Control Panel\\Mouse|MouseThreshold1|0\' \\\n        \'HKEY_CURRENT_USER\\Control Panel\\Mouse|MouseThreshold2|0\' \\\n        \'HKEY_CURRENT_USER\\Control Panel\\Mouse|MouseSensitivity|10\' \\\n        \'HKEY_CURRENT_USER\\Software\\Wine\\Direct3D|WINE_FULLSCREEN_FSR|1\' \\\n        \'HKEY_CURRENT_USER\\Software\\Wine\\Direct3D|WINE_FULLSCREEN_FSR_STRENGTH|2\' \\\n        \'HKEY_CURRENT_USER\\Software\\Wine\\DllOverrides|d3d11|native,builtin\' \\\n        \'HKEY_CURRENT_USER\\Software\\Wine\\DllOverrides|d3d12|native,builtin\' \\\n        \'HKEY_CURRENT_USER\\Software\\Wine\\DllOverrides|dxgi|native,builtin\'; do\n        IFS=\'|\' read -r rkey rname rval <<< "$reg_entry"\n        runuser -u gamer -- env WINEPREFIX="/home/gamer/.wine" WINEARCH=win64 \\\n            "$WINE_GE_DIR/Wine-GE-Proton8-26/bin/wine" reg add "$rkey" \\\n            /v "$rname" /t REG_SZ /d "$rval" /f 2>/dev/null || true\n    done\n    echo "✓ Wine prefix fully initialized with DXVK, VKD3D-Proton, and FSR upscaling."\nfi\n\n# 15. Inject keyboard lock into Selkies HTML (prevents Esc from exiting browser fullscreen)\npython3 -c "\nimport sys; sys.path.insert(0, \'$BENCH\')\ntry:\n    from auto_optimize import inject_keyboard_lock\n    result = inject_keyboard_lock()\n    if result: print(\'Keyboard lock injected into:\', result)\n    else: print(\'No Selkies HTML found to patch for keyboard lock.\')\nexcept Exception as e: print(\'Keyboard lock injection skipped:\', e)\n" || true\n\n# 16. Compile Selkies gamepad interposers if not present\nif [ ! -f "$BENCH/lib/x86_64-linux-gnu/selkies_input_interposer.so" ]; then\n    SOURCE="$BENCH/selkies-source"\n    if [ ! -d "$SOURCE" ]; then\n        git clone --depth 1 --branch 2.1.0 https://github.com/selkies-project/selkies.git "$SOURCE" || true\n    fi\n    if [ -d "$SOURCE/addons/input-interposer" ]; then\n        make -C "$SOURCE/addons/input-interposer" PREFIX="$BENCH" install || true\n        make -C "$SOURCE/addons/input-interposer" PREFIX="$BENCH" install32 2>/dev/null || true\n        make -C "$SOURCE/addons/fake-udev" || true\n        make -C "$SOURCE/addons/fake-udev" all32 2>/dev/null || true\n        mkdir -p "$BENCH/lib/x86_64-linux-gnu" "$BENCH/lib/i386-linux-gnu"\n        cp -f "$SOURCE/addons/fake-udev/libudev.so.1" "$BENCH/lib/x86_64-linux-gnu/selkies_fake_udev.so" 2>/dev/null || true\n        cp -f "$SOURCE/addons/fake-udev/libudev_x86.so.1" "$BENCH/lib/i386-linux-gnu/selkies_fake_udev.so" 2>/dev/null || true\n    fi\n    ln -sf "$BENCH/lib/x86_64-linux-gnu" "$BENCH/lib64" 2>/dev/null || true\n    ln -sf "$BENCH/lib/i386-linux-gnu" "$BENCH/lib32" 2>/dev/null || true\n    cp -f "$BENCH/lib/x86_64-linux-gnu/"*.so /usr/lib/x86_64-linux-gnu/ 2>/dev/null || true\n    cp -f "$BENCH/lib/i386-linux-gnu/"*.so /usr/lib/i386-linux-gnu/ 2>/dev/null || true\nfi\n\n# 17. Prepare /dev/input gamepad device nodes and permissions\nmkdir -p /dev/input 2>/dev/null || true\nfor i in 0 1 2 3; do touch /dev/input/js$i 2>/dev/null || true; done\nfor i in 1000 1001 1002 1003; do touch /dev/input/event$i 2>/dev/null || true; done\nchmod -R 666 /dev/input 2>/dev/null || true\n\n# 18. Set permissions and ownership\nchmod -R 755 "$BENCH"/*.sh 2>/dev/null || true\nchmod -R 777 "$BENCH" 2>/dev/null || true\nchown -R gamer:gamer /home/gamer 2>/dev/null || true\nchmod -R 777 /home/gamer 2>/dev/null || true\n\necho "=========================================================="\necho "    BENCH ENVIRONMENT SETUP COMPLETED SUCCESSFULLY!       "\necho "=========================================================="\n', encoding='utf-8')
probe_sh = BENCH / 'probe_nvidia_userspace.sh'
probe_sh.write_text('#!/usr/bin/env bash\n# High-Performance NVIDIA Userspace 3D Driver & Vulkan ICD Provisioner for Kaggle\n# Offloads 3D rendering to NVIDIA GPU hardware (OpenGL & Vulkan) without modifying the kernel.\nset -uo pipefail\n\nbench=/tmp/kaggle-game-dev-bench\nwork="$bench/nvidia-userspace-probe"\nmkdir -p "$work" "$work/lib64" "$work/lib32"\n\necho "=========================================================="\necho "    NVIDIA 3D HARDWARE ACCELERATION PROVISIONER           "\necho "=========================================================="\n\nif ! command -v nvidia-smi >/dev/null 2>&1; then\n    echo "Notice: nvidia-smi not found. Running in CPU-only environment."\n    exit 0\nfi\n\nversion=$(nvidia-smi --query-gpu=driver_version --format=csv,noheader 2>/dev/null | head -n 1 | tr -d \'[:space:]\')\nif [[ ! "$version" =~ ^[0-9]+(\\.[0-9]+)+$ ]]; then\n    echo "Notice: Cannot determine loaded NVIDIA driver version ($version)."\n    exit 0\nfi\n\nmajor=$(echo "$version" | cut -d. -f1)\necho "▶ Host NVIDIA Driver Version detected: $version (Major Branch: $major)"\n\ninstalled_ok=false\n\n# Tier 1: Fast APT package installation (~10 seconds)\necho "▶ Tier 1: Checking official NVIDIA 3D graphics packages for driver $major..."\nif [ "$(id -u)" -eq 0 ]; then\n    apt-get update -o DPkg::Lock::Timeout=60 -qq 2>/dev/null || true\n    for pkg in "libnvidia-gl-$major" "libnvidia-gl-${major}-server"; do\n        if apt-cache show "$pkg" >/dev/null 2>&1; then\n            echo "▶ Found matching repository package: $pkg. Installing userspace libraries..."\n            apt-get install -y --no-install-recommends \\\n                -o DPkg::Lock::Timeout=120 -o APT::Get::Assume-Yes=true \\\n                "$pkg" "${pkg}:i386" vulkan-tools libgl1-mesa-dri mesa-utils 2>/dev/null || \\\n            apt-get install -y --no-install-recommends \\\n                -o DPkg::Lock::Timeout=120 -o APT::Get::Assume-Yes=true \\\n                "$pkg" vulkan-tools libgl1-mesa-dri mesa-utils 2>/dev/null || true\n            break\n        fi\n    done\nfi\n\n# Verify if 64-bit GLX/EGL library is now present in standard paths\nif ldconfig -p | grep -q "libGLX_nvidia.so.0" || [ -f "/usr/lib/x86_64-linux-gnu/libGLX_nvidia.so.0" ]; then\n    echo "✓ NVIDIA 3D graphics libraries verified via system package manager."\n    installed_ok=true\nfi\n\n# Tier 2: Direct matching NVIDIA archive extraction if APT package was unavailable\nif [ "$installed_ok" = false ]; then\n    echo "▶ Tier 2: Fetching matching NVIDIA userspace binary archive for $version..."\n    runfile="$work/NVIDIA-Linux-x86_64-$version.run"\n    if [ ! -f "$runfile" ] || [ $(stat -c%s "$runfile" 2>/dev/null || echo 0) -lt 10000000 ]; then\n        urls=(\n            "https://us.download.nvidia.com/tesla/$version/NVIDIA-Linux-x86_64-$version.run"\n            "https://us.download.nvidia.com/XFree86/Linux-x86_64/$version/NVIDIA-Linux-x86_64-$version.run"\n            "https://download.nvidia.com/XFree86/Linux-x86_64/$version/NVIDIA-Linux-x86_64-$version.run"\n        )\n        for u in "${urls[@]}"; do\n            echo "Attempting download from: $u"\n            if curl -fL --connect-timeout 10 --max-time 120 "$u" -o "$runfile" 2>/dev/null; then\n                if [ -f "$runfile" ] && [ $(stat -c%s "$runfile") -gt 10000000 ]; then\n                    echo "✓ Downloaded matching NVIDIA installer: $(stat -c%s "$runfile") bytes"\n                    break\n                fi\n            fi\n            rm -f "$runfile"\n        done\n    fi\n\n    driver="$work/NVIDIA-Linux-x86_64-$version"\n    if [ -f "$runfile" ] && [ ! -d "$driver" ]; then\n        echo "▶ Extracting userspace graphics components from archive..."\n        (cd "$work" && sh "$runfile" --extract-only) 2>/dev/null || true\n    fi\n\n    if [ -d "$driver" ]; then\n        link_libraries() {\n            local src=$1 dst=$2\n            mkdir -p "$dst"\n            find "$src" -maxdepth 1 -type f \\( -name \'libnvidia-*.so*\' -o -name \'libGLX_nvidia.so*\' -o -name \'libEGL_nvidia.so*\' \\) -exec cp -sf {} "$dst/" \\; 2>/dev/null || true\n        }\n        link_libraries "$driver" "$work/lib64"\n        [ -d "$driver/32" ] && link_libraries "$driver/32" "$work/lib32"\n\n        for libdir in "$work/lib64" "$work/lib32"; do\n            for name in libGLX_nvidia libEGL_nvidia; do\n                vlib=$(find "$libdir" -name "${name}.so.*" 2>/dev/null | head -n1)\n                [ -n "$vlib" ] && ln -sfn "$(basename "$vlib")" "$libdir/${name}.so.0" 2>/dev/null || true\n            done\n        done\n        installed_ok=true\n    fi\nfi\n\n# Step 3: Register GLVND and Vulkan ICD manifests\nmkdir -p /usr/share/glvnd/egl_vendor.d /etc/vulkan/icd.d /usr/share/vulkan/icd.d "$work"\n\ncat > "$work/10_nvidia.json" <<\'EOF\'\n{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}\nEOF\n\ncat > "$work/nvidia_icd.json" <<\'EOF\'\n{"file_format_version":"1.0.0","ICD":{"library_path":"libGLX_nvidia.so.0","api_version":"1.3.0"}}\nEOF\n\ncat > "$work/nvidia_egl_icd.json" <<\'EOF\'\n{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0","api_version":"1.3.0"}}\nEOF\n\ncp -f "$work/10_nvidia.json" /usr/share/glvnd/egl_vendor.d/10_nvidia.json 2>/dev/null || true\ncp -f "$work/nvidia_icd.json" /etc/vulkan/icd.d/nvidia_icd.json 2>/dev/null || true\ncp -f "$work/nvidia_icd.json" /usr/share/vulkan/icd.d/nvidia_icd.json 2>/dev/null || true\n\n# Step 4: Register libraries with dynamic linker\nif [ -d "$work/lib64" ] && [ "$(ls -A "$work/lib64" 2>/dev/null)" ]; then\n    echo "$work/lib64" > /etc/ld.so.conf.d/nvidia-gpu.conf\n    [ -d "$work/lib32" ] && [ "$(ls -A "$work/lib32" 2>/dev/null)" ] && echo "$work/lib32" >> /etc/ld.so.conf.d/nvidia-gpu.conf\nfi\nldconfig 2>/dev/null || true\n\n# Step 5: Write environment exports\n{\n    printf \'export __EGL_VENDOR_LIBRARY_FILENAMES="/usr/share/glvnd/egl_vendor.d/10_nvidia.json"\\n\'\n    printf \'export VK_DRIVER_FILES="/etc/vulkan/icd.d/nvidia_icd.json"\\n\'\n    printf \'export VK_ICD_FILENAMES="/etc/vulkan/icd.d/nvidia_icd.json"\\n\'\n    if [ -d "$work/lib64" ] && [ "$(ls -A "$work/lib64" 2>/dev/null)" ]; then\n        printf \'export LD_LIBRARY_PATH="%s:%s${LD_LIBRARY_PATH:+:$LD_LIBRARY_PATH}"\\n\' "$work/lib64" "$work/lib32"\n        printf \'export GAMELAB_NVIDIA_LIB32="%s"\\n\' "$work/lib32"\n    fi\n} > "$work/env.sh"\n\necho "✓ NVIDIA userspace graphics bridge successfully configured."\necho "  Vulkan ICD: /etc/vulkan/icd.d/nvidia_icd.json"\necho "  EGL Vendor: /usr/share/glvnd/egl_vendor.d/10_nvidia.json"\n', encoding='utf-8')
probe_sh.chmod(0o755)
(BENCH / 'gpu_runtime.py').write_text('"""Choose a measured graphics path for Legendary/Wine games.\n\nNVENC and nvidia-smi are deliberately not treated as proof of 3D rendering.\nRun ``python3 gpu_runtime.py diagnose`` inside the live desktop session.\n"""\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport re\nimport shutil\nimport subprocess\nimport sys\n\n\ndef run(command, env=None, timeout=12):\n    try:\n        result = subprocess.run(command, env=env, text=True, errors="replace",\n                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT,\n                                timeout=timeout, check=False)\n        return result.returncode, result.stdout\n    except (OSError, subprocess.TimeoutExpired) as exc:\n        return 1, str(exc)\n\n\ndef clean_env():\n    env = os.environ.copy()\n    for key in ("LIBGL_ALWAYS_SOFTWARE", "GALLIUM_DRIVER", "MESA_LOADER_DRIVER_OVERRIDE",\n                "MESA_GL_VERSION_OVERRIDE", "LIBGL_DRIVERS_PATH",\n                "__GLX_VENDOR_LIBRARY_NAME"):\n        env.pop(key, None)\n    # The Selkies input interposer must not affect graphics probes.\n    env.pop("LD_PRELOAD", None)\n    return env\n\n\ndef nvidia_renderer(output):\n    vendor = re.search(r"OpenGL vendor string:\\s*(.+)", output, re.I)\n    renderer = re.search(r"OpenGL renderer string:\\s*(.+)", output, re.I)\n    # Tesla renderers often say "Tesla T4/PCIe/SSE2", without "NVIDIA".\n    return bool(vendor and "NVIDIA" in vendor.group(1).upper() and renderer\n                and not re.search(r"llvmpipe|softpipe|software", renderer.group(1), re.I))\n\n\ndef has_32bit_library(name):\n    injected = os.environ.get("GAMELAB_NVIDIA_LIB32", "")\n    if injected:\n        candidate = Path(injected) / name\n        try:\n            with candidate.open("rb") as library:\n                if library.read(5) == b"\\x7fELF\\x01":\n                    return True\n        except OSError:\n            pass\n    rc, out = run(["ldconfig", "-p"], timeout=4)\n    return rc == 0 and any(f"{name} (libc6)" in line\n                           for line in out.splitlines())\n\n\ndef dxvk_available():\n    root = Path("/opt/gamelab-dxvk/dxvk-2.6.2")\n    return all((root / arch / (name + ".dll")).is_file()\n               for arch in ("x32", "x64")\n               for name in ("d3d9", "d3d10core", "d3d11", "dxgi"))\n\n\ndef egl_device_probe(env):\n    """Test the NVIDIA EGL device directly, independent of Xorg and Xvfb."""\n    eglinfo = Path("/opt/VirtualGL/bin/eglinfo")\n    if not eglinfo.is_file():\n        return False, "VirtualGL eglinfo is not installed", ""\n    rc, listing = run([str(eglinfo), "-e"], env, timeout=8)\n    candidates = list(dict.fromkeys(re.findall(r"\\begl\\d+\\b", listing)))[:8] if rc == 0 else []\n    if not candidates:\n        candidates = ["egl0"]\n    last_result = listing[-240:]\n    for target in candidates:\n        rc, out = run([str(eglinfo), target], env, timeout=12)\n        renderer = next((line.strip() for line in out.splitlines()\n                         if line.strip().startswith("OpenGL renderer string:")), "")\n        vendor = next((line.strip() for line in out.splitlines()\n                       if line.strip().startswith("OpenGL vendor string:")), "")\n        if (rc == 0 and "NVIDIA" in vendor.upper() and renderer\n                and not re.search(r"llvmpipe|softpipe|software", renderer, re.I)):\n            return True, renderer, target\n        last_result = renderer or out[-240:]\n    return False, last_result, ""\n\n\ndef cpu_quota_count():\n    """Return the cgroup CPU quota as a whole-core budget, if one exists."""\n    try:\n        quota, period = Path("/sys/fs/cgroup/cpu.max").read_text().split()[:2]\n        if quota != "max":\n            return max(1, math.floor(int(quota) / int(period)))\n    except (OSError, ValueError, ZeroDivisionError):\n        pass\n    try:\n        quota = int(Path("/sys/fs/cgroup/cpu/cpu.cfs_quota_us").read_text())\n        period = int(Path("/sys/fs/cgroup/cpu/cpu.cfs_period_us").read_text())\n        if quota > 0:\n            return max(1, math.floor(quota / period))\n    except (OSError, ValueError, ZeroDivisionError):\n        pass\n    return None\n\n\ndef software_threads():\n    override = os.environ.get("LP_NUM_THREADS", "")\n    if override.isdigit() and 1 <= int(override) <= 32:\n        return int(override)\n    try:\n        cores = len(os.sched_getaffinity(0))\n    except AttributeError:\n        cores = os.cpu_count() or 1\n    quota = cpu_quota_count()\n    if quota is not None:\n        cores = min(cores, quota)\n    reserve = 2 if cores >= 4 else 1\n    return max(1, min(8, cores - reserve))\n\n\ndef lower_software_resolution(env):\n    """Best-effort temporary Xvfb mode change; game settings may differ."""\n    requested = os.environ.get("GAMELAB_SOFTWARE_RESOLUTION", "854x480")\n    if requested == "off" or not shutil.which("xrandr") or not env.get("DISPLAY"):\n        return None\n    if requested not in ("640x360", "854x480", "1024x576", "1280x720"):\n        print(f"Unsupported GAMELAB_SOFTWARE_RESOLUTION={requested!r}; keeping desktop mode.",\n              file=sys.stderr)\n        return None\n    rc, out = run(["xrandr", "--current"], env, timeout=5)\n    match = re.search(r"\\bcurrent (\\d+) x (\\d+)\\b", out)\n    if rc != 0 or not match:\n        return None\n    previous = f"{match.group(1)}x{match.group(2)}"\n    if previous == requested:\n        return None\n    rc, out = run(["xrandr", "-s", requested], env, timeout=5)\n    if rc != 0:\n        print(f"Could not set {requested} desktop mode: {out[-200:]}", file=sys.stderr)\n        return None\n    print(f"Software desktop mode: {requested} (restore {previous} on exit)", flush=True)\n    return previous\n\n\ndef install_dxvk(prefix):\n    """Install both bitnesses without following Wine DLL symlinks."""\n    root = Path("/opt/gamelab-dxvk/dxvk-2.6.2")\n    for arch, subdir in (("x64", "system32"), ("x32", "syswow64")):\n        destination = Path(prefix) / "drive_c/windows" / subdir\n        if not destination.is_dir():\n            raise RuntimeError(f"Wine prefix is incomplete: {destination}")\n        for name in ("d3d9", "d3d10core", "d3d11", "dxgi"):\n            target = destination / (name + ".dll")\n            backup = destination / (name + ".dll.gamelab-original")\n            temporary = destination / (name + ".dll.gamelab-new")\n            shutil.copyfile(root / arch / target.name, temporary)\n            if (target.exists() or target.is_symlink()) and not backup.exists():\n                os.replace(target, backup)\n            os.replace(temporary, target)\n\n\ndef install_vkd3d(prefix):\n    """Install VKD3D-Proton (DirectX 12) if available."""\n    candidates = list(Path("/opt/gamelab-vkd3d").glob("vkd3d-proton-*"))\n    if not candidates:\n        return\n    root = candidates[0]\n    for arch, subdir in (("x64", "system32"), ("x86", "syswow64")):\n        src_dir = root / arch\n        if not src_dir.is_dir():\n            continue\n        dst_dir = Path(prefix) / "drive_c/windows" / subdir\n        if not dst_dir.is_dir():\n            continue\n        for name in ("d3d12.dll", "d3d12core.dll"):\n            src_file = src_dir / name\n            if src_file.is_file():\n                shutil.copyfile(src_file, dst_dir / name)\n\n\ndef diagnose():\n    env = clean_env()\n    facts = {\n        "display": env.get("DISPLAY", ""),\n        "nvidia_control": Path("/dev/nvidiactl").exists(),\n        "nvidia_gpu": Path("/dev/nvidia0").exists(),\n        "nvidia_modeset": Path("/dev/nvidia-modeset").exists(),\n        "nvidia_icd": any(Path(p).exists() for p in (\n            "/etc/vulkan/icd.d/nvidia_icd.json",\n            "/usr/share/vulkan/icd.d/nvidia_icd.json")) or any(\n                Path(p).is_file() and "nvidia" in Path(p).name.lower()\n                for p in (env.get("VK_DRIVER_FILES") or env.get("VK_ICD_FILENAMES") or "").split(":") if p),\n        "direct_gl": False, "virtualgl": False, "virtualgl_display": "",\n        "vulkan_device": False,\n        "vulkan_window": False,\n        "nvidia_32bit_gl": has_32bit_library("libGLX_nvidia.so.0"),\n        "vulkan_32bit_loader": has_32bit_library("libvulkan.so.1"),\n        "dxvk_files": dxvk_available(),\n    }\n    facts["egl_device"], facts["egl_renderer"], facts["egl_device_display"] = egl_device_probe(env)\n    if shutil.which("glxinfo") and env.get("DISPLAY"):\n        rc, out = run(["glxinfo", "-B"], env)\n        facts["direct_gl"] = rc == 0 and nvidia_renderer(out)\n        facts["gl_renderer"] = next((line.strip() for line in out.splitlines()\n                                     if "OpenGL renderer string:" in line), out[-240:])\n        vgl = shutil.which("vglrun") or "/opt/VirtualGL/bin/vglrun"\n        if Path(vgl).is_file() and facts["nvidia_control"] and facts["nvidia_gpu"]:\n            candidates = [facts["egl_device_display"]] if facts["egl_device"] else ["egl0"]\n            if Path("/tmp/kaggle-game-dev-bench/gpu-xorg-ready").exists():\n                candidates.append(":1")\n            for target in candidates:\n                rc, out = run([vgl, "-d", target, "glxinfo", "-B"], env)\n                if rc == 0 and nvidia_renderer(out):\n                    facts["virtualgl"] = True\n                    facts["virtualgl_display"] = target\n                    facts["vgl_renderer"] = next((line.strip() for line in out.splitlines()\n                                                   if "OpenGL renderer string:" in line), out[-240:])\n                    break\n    if facts["nvidia_control"] and facts["nvidia_gpu"] and shutil.which("vulkaninfo"):\n        rc, out = run(["vulkaninfo", "--summary"], env)\n        facts["vulkan_device"] = rc == 0 and bool(re.search(\n            r"deviceName\\s*=.*(?:NVIDIA|Tesla T4)", out, re.I))\n        facts["vulkan_summary"] = out[-500:]\n    if facts["vulkan_device"] and env.get("DISPLAY"):\n        rc, out = run([sys.executable, "-m", "selkies.gpu_probe", "--vulkan-window"], env)\n        facts["vulkan_window"] = rc == 0 and out.strip() == "true"\n    return facts\n\n\ndef legendary_install_path(app):\n    """Return an installed Epic title\'s directory from Legendary metadata."""\n    home = Path(os.environ.get("KAGGLE_GAMER_HOME", "/home/gamer"))\n    candidates = (\n        home / ".config/heroic/legendaryConfig/legendary/installed.json",\n        home / ".config/legendary/installed.json",\n    )\n    for installed in candidates:\n        try:\n            record = json.loads(installed.read_text(encoding="utf-8")).get(app, {})\n            path = Path(record.get("install_path") or record.get("installPath") or "")\n            if path.is_dir():\n                return path\n        except (OSError, ValueError, AttributeError):\n            continue\n    return None\n\n\ndef unreal_shipping_binary(app):\n    """Find a UE Shipping executable while excluding prerequisite launchers."""\n    install = legendary_install_path(app)\n    if install is None:\n        return None\n    matches = []\n    for path in install.rglob("*-Win64-Shipping.exe"):\n        lowered = str(path).lower()\n        if "prereq" not in lowered and "redist" not in lowered and "crashreport" not in lowered:\n            matches.append(path)\n    matches.sort(key=lambda path: ("binaries\\\\win64" not in str(path).lower().replace("/", "\\\\"), len(str(path))))\n    return matches[0] if matches else None\n\n\ndef find_game_binary(app):\n    """Find the game\'s actual executable binary (supporting UE, id Tech, Unity, custom engines)."""\n    ue_shipping = unreal_shipping_binary(app)\n    if ue_shipping and ue_shipping.is_file():\n        return ue_shipping\n\n    home = Path(os.environ.get("KAGGLE_GAMER_HOME", "/home/gamer"))\n    install_candidates = []\n\n    # Check installed.json metadata (exact key match, then fuzzy/case-insensitive match)\n    for installed_file in (\n        home / ".config/heroic/legendaryConfig/legendary/installed.json",\n        home / ".config/legendary/installed.json",\n    ):\n        try:\n            data = json.loads(installed_file.read_text(encoding="utf-8"))\n            if app in data:\n                record = data[app]\n                ip = Path(record.get("install_path") or record.get("installPath") or "")\n                ex = record.get("executable") or ""\n                if ip.is_dir():\n                    if ex and (ip / ex).is_file():\n                        return ip / ex\n                    for p in ip.rglob("*.exe"):\n                        low = p.name.lower()\n                        if any(skip in low for skip in ("prereq", "redist", "crash", "unins", "setup", "dxsetup", "vcredist", "epicgameslauncher")):\n                            continue\n                        if ex and p.name.lower() == Path(ex).name.lower():\n                            return p\n                        if "x64" in low or "wolf" in low:\n                            return p\n                    install_candidates.append(ip)\n            for key, record in data.items():\n                if not isinstance(record, dict):\n                    continue\n                k_low = key.lower()\n                a_low = app.lower()\n                t_low = str(record.get("title", "")).lower()\n                if a_low in k_low or k_low in a_low or a_low in t_low:\n                    ip = Path(record.get("install_path") or record.get("installPath") or "")\n                    ex = record.get("executable") or ""\n                    if ip.is_dir():\n                        if ex and (ip / ex).is_file():\n                            return ip / ex\n                        for p in ip.rglob("*.exe"):\n                            low = p.name.lower()\n                            if any(skip in low for skip in ("prereq", "redist", "crash", "unins", "setup", "dxsetup", "vcredist", "epicgameslauncher")):\n                                continue\n                            if ex and p.name.lower() == Path(ex).name.lower():\n                                return p\n                            if "x64" in low or "wolf" in low:\n                                return p\n                        install_candidates.append(ip)\n        except Exception:\n            pass\n\n    install_candidates.extend([\n        home / "game-builds",\n        home / ".wine/drive_c/Program Files",\n        home / ".wine/drive_c/Program Files (x86)",\n        Path("/kaggle/working"),\n    ])\n\n    found_exes = []\n    for root in install_candidates:\n        if not root.is_dir():\n            continue\n        try:\n            for p in root.rglob("*.exe"):\n                low = p.name.lower()\n                if any(skip in low for skip in ("prereq", "redist", "crash", "unins", "setup", "dxsetup", "vcredist", "epicgameslauncher")):\n                    continue\n                found_exes.append(p)\n        except Exception:\n            pass\n\n    if found_exes:\n        a_low = app.lower()\n        found_exes.sort(key=lambda p: (\n            0 if a_low in p.name.lower() else 1,\n            0 if ("x64" in p.name.lower() or "64" in p.name.lower()) else 1,\n            len(p.parts)\n        ))\n        return found_exes[0]\n    return None\n\n\ndef get_input_mode() -> str:\n    """Read the active input mode (mouse_keyboard or gamepad)."""\n    mode_file = Path("/tmp/kaggle-game-dev-bench/input-mode.json")\n    try:\n        if mode_file.is_file():\n            data = json.loads(mode_file.read_text(encoding="utf-8"))\n            return data.get("mode", "mouse_keyboard")\n    except Exception:\n        pass\n    return os.environ.get("SEVEN_BZ_INPUT_MODE", "mouse_keyboard")\n\n\ndef configure_mouse(wine_bin, env):\n    """Apply verified Wine DirectInput, X11 pointer capture, and raw sensitivity."""\n    if not wine_bin:\n        return\n    values = (\n        ("HKEY_CURRENT_USER\\\\Software\\\\Wine\\\\DirectInput", "MouseWarpOverride", "force"),\n        ("HKEY_CURRENT_USER\\\\Software\\\\Wine\\\\X11 Driver", "GrabPointer", "Y"),\n        ("HKEY_CURRENT_USER\\\\Software\\\\Wine\\\\X11 Driver", "UseTakeFocus", "N"),\n        ("HKEY_CURRENT_USER\\\\Control Panel\\\\Mouse", "MouseSpeed", "0"),\n        ("HKEY_CURRENT_USER\\\\Control Panel\\\\Mouse", "MouseThreshold1", "0"),\n        ("HKEY_CURRENT_USER\\\\Control Panel\\\\Mouse", "MouseThreshold2", "0"),\n        ("HKEY_CURRENT_USER\\\\Control Panel\\\\Mouse", "MouseSensitivity", "10"),\n    )\n    for key, name, value in values:\n        try:\n            subprocess.run([wine_bin, "reg", "add", key, "/v", name, "/t", "REG_SZ",\n                            "/d", value, "/f"], env=env, check=False,\n                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=8)\n        except (OSError, subprocess.TimeoutExpired):\n            pass\n\n\ndef select_graphics_backend(facts, preference="auto", is_64bit=True):\n    """Select only paths whose required presentation probe succeeded."""\n    if preference not in ("auto", "dxvk", "wined3d"):\n        raise ValueError("GAMELAB_GRAPHICS_BACKEND must be auto, dxvk, or wined3d")\n    dxvk_ready = (facts["vulkan_window"] and (facts["vulkan_32bit_loader"] or is_64bit)\n                  and facts["dxvk_files"])\n    gl_ready = facts["nvidia_32bit_gl"] or is_64bit\n    if preference == "dxvk":\n        if not dxvk_ready:\n            raise RuntimeError("DXVK requires a working Vulkan X11 swapchain, 32-bit loader, and both DLL sets")\n        return "dxvk"\n    if preference == "auto" and dxvk_ready:\n        return "dxvk"\n    if facts["direct_gl"] and gl_ready:\n        return "direct_gl"\n    if facts["virtualgl"] and gl_ready:\n        return "virtualgl"\n    if preference == "wined3d":\n        if facts["virtualgl"] and gl_ready:\n            return "virtualgl"\n        raise RuntimeError("WineD3D requires verified NVIDIA GL and 32-bit GL libraries")\n    return "software"\n\n\ndef launch(app, wine):\n    facts = diagnose()\n    env = clean_env()\n    direct_xorg = (env.get("DISPLAY", "").split(".", 1)[0] == ":1" and\n                   facts["direct_gl"] and\n                   os.environ.get("GAMELAB_DIRECT_XORG") == "1")\n    env["WINEPREFIX"] = os.environ.get("WINEPREFIX", "/home/gamer/.wine")\n    env["WINEDLLOVERRIDES"] = "d3d11,dxgi,d3d9,d3d10core=b;mscoree,mshtml=d"\n    env["SDL_AUDIODRIVER"] = "pulse"\n    env["HOME"] = os.environ.get("KAGGLE_GAMER_HOME", "/home/gamer")\n    env["__GL_SYNC_TO_VBLANK"] = "0"\n\n    # Dedicate GPU 0 exclusively to 3D rendering and physics simulation\n    env["CUDA_VISIBLE_DEVICES"] = "0"\n    env["DRI_PRIME"] = "0"\n    if not direct_xorg:\n        env["VGL_DISPLAY"] = "egl0"\n    env["DXVK_FILTER_DEVICE_NAME"] = "Tesla T4"\n    env["WINE_FULLSCREEN_FSR"] = "1"\n    env["WINE_FULLSCREEN_FSR_STRENGTH"] = "2"\n    env["VKD3D_CONFIG"] = "dxr11,dxr"\n    env["VKD3D_FEATURE_LEVEL"] = "12_1"\n\n    # Guarantee full OpenGL 4.5 Compatibility Profile & WGL extensions for Wine/idTech\n    if not direct_xorg:\n        env["MESA_GL_VERSION_OVERRIDE"] = "4.5COMPAT"\n        env["MESA_GLSL_VERSION_OVERRIDE"] = "450"\n        env["__NV_PRIME_RENDER_OFFLOAD"] = "1"\n        env["__GLX_VENDOR_LIBRARY_NAME"] = "nvidia"\n    if not env.get("DISPLAY"):\n        env["DISPLAY"] = ":0"\n\n    # Propagate hardware profile to Legendary configuration\n    try:\n        leg_cfg_path = Path(env["HOME"]) / ".config/legendary/config.ini"\n        leg_cfg_path.parent.mkdir(parents=True, exist_ok=True)\n        current_cfg = leg_cfg_path.read_text(encoding="utf-8") if leg_cfg_path.is_file() else ""\n        if "[default.env]" not in current_cfg:\n            with leg_cfg_path.open("a", encoding="utf-8") as f:\n                f.write("\\n[default.env]\\nVGL_DISPLAY = egl0\\nMESA_GL_VERSION_OVERRIDE = 4.5COMPAT\\nMESA_GLSL_VERSION_OVERRIDE = 450\\n__NV_PRIME_RENDER_OFFLOAD = 1\\n__GLX_VENDOR_LIBRARY_NAME = nvidia\\n")\n    except Exception:\n        pass\n\n    if not env.get("PULSE_SERVER"):\n        for candidate in ("/run/user/1000/pulse/native", "/tmp/pulse-PKdhtXMmr18n/native"):\n            if Path(candidate).exists():\n                env["PULSE_SERVER"] = f"unix:{candidate}"\n                break\n        if not env.get("PULSE_SERVER"):\n            try:\n                found = list(Path("/tmp").glob("pulse-*/native")) + list(Path("/run/user").glob("*/pulse/native"))\n                if found:\n                    env["PULSE_SERVER"] = f"unix:{found[0]}"\n            except Exception:\n                pass\n\n    wine_bin = wine or shutil.which("wine")\n    configure_mouse(wine_bin, env)\n\n    input_mode = get_input_mode()\n    print(f"Active input mode: {input_mode}", flush=True)\n    if input_mode == "mouse_keyboard":\n        # Pure Mouse & Keyboard: Remove interposer and disable Wine virtual joystick\n        # to prevent ghost controller conflict (locks Pitch / Yaw fighting).\n        env.pop("LD_PRELOAD", None)\n        env["SDL_JOYSTICK_DISABLE_UDEV"] = "1"\n        env["SDL_JOYSTICK_DEVICE"] = ""\n        env["WINEDLLOVERRIDES"] = env.get("WINEDLLOVERRIDES", "") + ";winebus.sys=d"\n    else:\n        # Gamepad Mode: Bind Xbox 360 controller sockets via interposers\n        env["SELKIES_JS_SOCKET_PATH"] = ("/tmp/kaggle-game-dev-bench/direct-stream-sockets"\n                                         if direct_xorg else "/tmp")\n        env["SDL_JOYSTICK_DISABLE_UDEV"] = "1"\n        env["SDL_JOYSTICK_DEVICE"] = "/dev/input/event1000:/dev/input/event1001:/dev/input/event1002:/dev/input/event1003"\n        interposer = "/tmp/kaggle-game-dev-bench/$LIB/selkies_input_interposer.so:/tmp/kaggle-game-dev-bench/$LIB/selkies_fake_udev.so"\n        env["LD_PRELOAD"] = os.environ.get("LD_PRELOAD", interposer)\n\n    target_bin = find_game_binary(app)\n    is_64bit = True\n    if target_bin and "32" in target_bin.name.lower() and "64" not in target_bin.name.lower():\n        is_64bit = False\n    launch_cwd = str(target_bin.parent) if target_bin else None\n\n    # Switch to 64-bit Wine binary for 64-bit binaries to prevent 32-bit wrapper dropping 64-bit VGL faker\n    if is_64bit and wine_bin:\n        wine64_candidate = Path(wine_bin).with_name("wine64")\n        if wine64_candidate.is_file():\n            wine_bin = str(wine64_candidate)\n\n    vgl = shutil.which("vglrun") or "/opt/VirtualGL/bin/vglrun"\n    vgl_ready = (not direct_xorg and Path(vgl).is_file() and\n                 (facts.get("virtualgl") or facts.get("egl_device")))\n    vgl_display = facts.get("virtualgl_display") or facts.get("egl_device_display") or "egl0"\n\n    # vglrun enables its dlopen interposer by default.  -dl is not a valid\n    # option in current VirtualGL and is interpreted as the executable.\n    wrap_script = Path("/tmp/kaggle-game-dev-bench/vgl-wrap.sh")\n    try:\n        wrap_script.write_text(f"#!/bin/bash\\nexec {vgl} -d {vgl_display} \\"$@\\"\\n", encoding="utf-8")\n        wrap_script.chmod(0o755)\n    except Exception:\n        pass\n\n    if vgl_ready:\n        env["VGL_DISPLAY"] = vgl_display\n        env.update(VGL_READBACK="pbo", VGL_SPOIL="1", VGL_SYNC="0", VGL_COMPRESS="proxy")\n        vgl_fps = os.environ.get("GAMELAB_VGL_FPS", "60")\n        try:\n            if 0 <= float(vgl_fps) <= 240:\n                env["VGL_FPS"] = vgl_fps\n        except ValueError:\n            pass\n\n    if target_bin and wine_bin:\n        print(f"Target game binary detected: {target_bin}", flush=True)\n        command = [wine_bin, str(target_bin)]\n    else:\n        command = ["legendary", "launch", app, "--wine", wine]\n        if vgl_ready:\n            command.extend(["--wrapper", str(wrap_script)])\n\n    # Prepend vglrun so the process tree inherits its OpenGL interposers.\n    if vgl_ready:\n        command = [vgl, "-d", vgl_display, *command]\n\n    try:\n        selected = select_graphics_backend(\n            facts, os.environ.get("GAMELAB_GRAPHICS_BACKEND", "auto"), is_64bit=is_64bit)\n    except (ValueError, RuntimeError) as exc:\n        print(f"Graphics backend unavailable: {exc}", file=sys.stderr)\n        return 78\n    if selected == "dxvk":\n        try:\n            install_dxvk(env["WINEPREFIX"])\n        except (OSError, RuntimeError) as exc:\n            print(f"DXVK setup failed: {exc}", file=sys.stderr)\n            return 78\n        try:\n            install_vkd3d(env["WINEPREFIX"])\n        except Exception as exc:\n            print(f"VKD3D setup notice: {exc}", file=sys.stderr)\n        env["WINEDLLOVERRIDES"] = "d3d12,d3d12core,d3d11,dxgi,d3d9,d3d10core=n,b;mscoree,mshtml=d"\n        env["DXVK_HUD"] = "devinfo,fps"\n        backend = "NVIDIA Vulkan through DXVK & VKD3D + VirtualGL (GPU 0)"\n    elif selected == "direct_gl":\n        backend = "NVIDIA OpenGL"\n    elif selected == "virtualgl":\n        display = facts.get("virtualgl_display") or "egl0"\n        backend = f"NVIDIA OpenGL through VirtualGL ({display})"\n    elif os.environ.get("GAMELAB_ALLOW_SOFTWARE") == "1":\n        backend = "Mesa software (explicit opt-in)"\n        env["WINEDLLOVERRIDES"] = "d3d11,dxgi,d3d9,d3d10core=b;mscoree,mshtml=d"\n        env["LIBGL_ALWAYS_SOFTWARE"] = "1"\n        env["GALLIUM_DRIVER"] = "llvmpipe"\n        env["LP_NUM_THREADS"] = str(software_threads())\n        if shutil.which("nice"):\n            command = ["nice", "-n", "5", *command]\n    else:\n        backend = "Mesa software (fallback)"\n        env["WINEDLLOVERRIDES"] = "d3d11,dxgi,d3d9,d3d10core=b;mscoree,mshtml=d"\n        env["LIBGL_ALWAYS_SOFTWARE"] = "1"\n        env["GALLIUM_DRIVER"] = "llvmpipe"\n        env["LP_NUM_THREADS"] = str(software_threads())\n        if shutil.which("nice"):\n            command = ["nice", "-n", "5", *command]\n    print(f"Rendering backend: {backend}", flush=True)\n    if backend.startswith("Mesa software"):\n        print(f"llvmpipe worker threads: {env[\'LP_NUM_THREADS\']}", flush=True)\n        previous_resolution = lower_software_resolution(env)\n        try:\n            return subprocess.call(command, env=env, cwd=launch_cwd)\n        finally:\n            if previous_resolution:\n                run(["xrandr", "-s", previous_resolution], env, timeout=5)\n    return subprocess.call(command, env=env, cwd=launch_cwd)\n\n\nif __name__ == "__main__":\n    if len(sys.argv) == 2 and sys.argv[1] == "diagnose":\n        print(json.dumps(diagnose(), indent=2))\n    elif len(sys.argv) == 4 and sys.argv[1] == "launch":\n        sys.exit(launch(sys.argv[2], sys.argv[3]))\n    else:\n        print("Usage: gpu_runtime.py diagnose | launch APP WINE_BIN", file=sys.stderr)\n        sys.exit(2)\n', encoding='utf-8')
(BENCH / 'launch_idtech5.py').write_text('"""Launch a 64-bit id Tech OpenGL game on the Selkies X desktop.\n\nThe GLX extension must exist on the captured X server.  VirtualGL sends\nOpenGL work to the selected NVIDIA GLX server or EGL device and copies\ncompleted frames into the Wine window on DISPLAY. Selkies captures :0.\n"""\n\nimport argparse\nimport os\nfrom pathlib import Path\nimport re\nimport subprocess\nimport sys\n\n\nVGL = Path("/opt/VirtualGL/bin/vglrun")\n\n\ndef probe(args, env, timeout=15):\n    result = subprocess.run(args, env=env, text=True, errors="replace",\n                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,\n                            timeout=timeout, check=False)\n    return result.returncode, result.stdout\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("game", type=Path,\n                        help="Path to a 64-bit Windows OpenGL game executable")\n    parser.add_argument("--wine", type=Path, required=True,\n                        help="Path to Wine-GE\'s bin/wine or bin/wine64")\n    parser.add_argument("--vgl-display", default="auto",\n                        help="VirtualGL target: auto, :1, or egl0 (default: auto)")\n    parser.add_argument("--direct-xorg", action="store_true",\n                        help="Run directly on the NVIDIA Xorg display :1 (use its Selkies stream)")\n    parser.add_argument("--egl-device", default=None,\n                        help="Deprecated alias for --vgl-display")\n    parser.add_argument("--prefix", type=Path,\n                        default=Path("/home/gamer/.wine"))\n    parser.add_argument("game_args", nargs=argparse.REMAINDER)\n    args = parser.parse_args()\n\n    game = args.game.resolve()\n    wine = args.wine.resolve()\n    if not game.is_file() or not wine.is_file() or (not args.direct_xorg and not VGL.is_file()):\n        parser.error("game, Wine binary, and selected graphics launcher must exist")\n\n    env = os.environ.copy()\n    env["DISPLAY"] = ":1" if args.direct_xorg else (env.get("DISPLAY") or ":0")\n    env["WINEPREFIX"] = str(args.prefix)\n    env["VGL_COMPRESS"] = "proxy"\n    env["SDL_AUDIODRIVER"] = "pulse"\n    overrides = env.get("WINEDLLOVERRIDES", "")\n    env["WINEDLLOVERRIDES"] = (overrides + ";" if overrides else "") + "opengl32=b"\n    # These variables cannot add GLX to Xvfb or select the VirtualGL GPU.\n    for key in ("LIBGL_ALWAYS_SOFTWARE", "GALLIUM_DRIVER",\n                "MESA_LOADER_DRIVER_OVERRIDE", "MESA_GL_VERSION_OVERRIDE",\n                "MESA_GLSL_VERSION_OVERRIDE", "__NV_PRIME_RENDER_OFFLOAD",\n                "__GLX_VENDOR_LIBRARY_NAME"):\n        env.pop(key, None)\n\n    if args.direct_xorg:\n        # A direct GLX context and the captured window must live on the same\n        # NVIDIA X server. VirtualGL\'s cross-server drawable mapping is not\n        # reliable for this game\'s multi-context WGL path.\n        env.pop("VGL_DISPLAY", None)\n        for key in ("VGL_SYNC", "VGL_READBACK", "VGL_LOGO", "VGL_PROFILE"):\n            env.pop(key, None)\n        socket_dir = Path("/tmp/kaggle-game-dev-bench/direct-stream-sockets")\n        input_hook = Path("/tmp/kaggle-game-dev-bench/lib64/selkies_input_interposer.so")\n        if socket_dir.is_dir() and input_hook.is_file():\n            env["SELKIES_JS_SOCKET_PATH"] = str(socket_dir)\n            env["LD_PRELOAD"] = (str(input_hook) + ":" + env.get("LD_PRELOAD", "")).rstrip(":")\n            env["SDL_JOYSTICK_DISABLE_UDEV"] = "1"\n        try:\n            rc, info = probe(["glxinfo", "-B"], env)\n            if rc or not re.search(r"OpenGL vendor string:\\s*NVIDIA", info):\n                raise RuntimeError("NVIDIA GLX is unavailable on :1: " + info[-500:])\n        except (OSError, subprocess.TimeoutExpired, RuntimeError) as exc:\n            print(f"Graphics prerequisite failed: {exc}", file=sys.stderr)\n            return 78\n        wine_bin = wine.with_name("wine64") if wine.with_name("wine64").is_file() else wine\n        command = [str(wine_bin), str(game)]\n        command.extend(args.game_args or\n                       ["+set", "com_skipIntroVideo", "1",\n                        "+set", "com_skipBootSequence", "1",\n                        "+set", "r_fullscreen", "0", "+set", "r_mode", "-1",\n                        "+set", "r_windowedWidth", "1280",\n                        "+set", "r_windowedHeight", "720"])\n        print("OpenGL target: direct NVIDIA Xorg :1", flush=True)\n        print("Stream this display using start_direct_stream.sh", flush=True)\n        return subprocess.call(command, cwd=game.parent, env=env)\n\n    try:\n        rc, extensions = probe(["xdpyinfo", "-queryExtensions"], env)\n        if rc or not re.search(r"(?m)^\\s*GLX\\s+\\(opcode:", extensions):\n            raise RuntimeError(\n                f"{env[\'DISPLAY\']} does not advertise GLX. Wine\'s winex11.drv "\n                "will disable WGL. Restart the captured Xvfb with GLX enabled "\n                "or use a GLX-capable captured X server."\n            )\n\n        selected = args.egl_device or args.vgl_display\n        if selected == "auto":\n            candidates = ([":1"] if Path("/tmp/kaggle-game-dev-bench/gpu-xorg-ready").exists()\n                          else []) + ["egl0"]\n        else:\n            candidates = [selected]\n        failures = []\n        for candidate in candidates:\n            rc, info = probe([str(VGL), "-d", candidate, "glxinfo", "-B"], env)\n            if (not rc and re.search(r"OpenGL vendor string:\\s*NVIDIA", info)\n                    and not re.search(r"OpenGL renderer string:.*(?:llvmpipe|softpipe)", info, re.I)):\n                selected = candidate\n                break\n            failures.append(f"{candidate}: {info[-500:]}")\n        else:\n            raise RuntimeError("VirtualGL NVIDIA probe failed:\\n" + "\\n".join(failures))\n    except (OSError, subprocess.TimeoutExpired, RuntimeError) as exc:\n        print(f"Graphics prerequisite failed: {exc}", file=sys.stderr)\n        return 78\n\n    # vglrun already preloads libdlfaker and libvglfaker by default.\n    wine_bin = wine.with_name("wine64") if wine.with_name("wine64").is_file() else wine\n    env["VGL_DISPLAY"] = selected\n    command = [str(VGL), "-d", selected, str(wine_bin), str(game)]\n    if not args.game_args:\n        command.extend(["-window", "+com_skipIntroVideo", "1", "+r_fullscreen", "0"])\n    else:\n        command.extend(args.game_args)\n    print("OpenGL target:", selected, flush=True)\n    print("Wine window:", env["DISPLAY"], flush=True)\n    return subprocess.call(command, cwd=game.parent, env=env)\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n', encoding='utf-8')
(BENCH / 'start_direct_stream.sh').write_text('#!/bin/bash\n# Stream the NVIDIA Xorg display directly. Run as root after the notebook\'s\n# ordinary desktop session has created PulseAudio and gpu_xorg.sh has run.\nset -euo pipefail\nBENCH=/tmp/kaggle-game-dev-bench\nPORT="${DIRECT_STREAM_PORT:-8081}"\nWIDTH="${DIRECT_STREAM_WIDTH:-1280}"\nHEIGHT="${DIRECT_STREAM_HEIGHT:-720}"\nORIGIN="${SELKIES_ALLOWED_ORIGINS:-https://kkb-production.jupyter-proxy.kaggle.net}"\nif [ "$(id -u)" -ne 0 ]; then\n    echo \'This Xorg installation requires root for Selkies MIT-SHM capture.\' >&2\n    exit 1\nfi\nbash "$BENCH/gpu_xorg.sh"\ntest -e "$BENCH/gpu-xorg-ready"\nDISPLAY=:1 XAUTHORITY="$BENCH/nvidia-Xauthority" \\\n    glxinfo -B | grep -q \'OpenGL vendor string: NVIDIA\'\nif command -v xrandr >/dev/null; then\n    DISPLAY=:1 XAUTHORITY="$BENCH/nvidia-Xauthority" \\\n        xrandr --output DVI-D-0 --mode "${WIDTH}x${HEIGHT}"\nfi\nPULSE_SERVER="$(cat "$BENCH/pulse-server-path" 2>/dev/null || true)"\nexport DISPLAY=:1 XAUTHORITY="$BENCH/nvidia-Xauthority" PULSE_SERVER\nENCODER_DEVICE=0\nif [ -e /dev/nvidia1 ]; then ENCODER_DEVICE=1; fi\nexport CUDA_VISIBLE_DEVICES="$ENCODER_DEVICE"\nexport SELKIES_ALLOWED_ORIGINS="$ORIGIN"\nexport SELKIES_JS_SOCKET_PATH="$BENCH/direct-stream-sockets"\nmkdir -p "$SELKIES_JS_SOCKET_PATH" "$BENCH/logs"\ntest -f "$BENCH/web-root/index.html"\n# Keep the system Openbox settings, with a rule for the game\'s Wine window.\n# EWMH fullscreen removes both the title bar and tint2\'s reserved screen area.\npython3 - "$BENCH/direct-openbox-rc.xml" <<\'PY\'\nfrom pathlib import Path\nimport sys\n\nsource = Path(\'/etc/xdg/openbox/rc.xml\').read_text(encoding=\'utf-8\')\nrule = \'\'\'<applications>\n  <application class="wolfneworder_x64.exe">\n    <decor>no</decor>\n    <fullscreen>yes</fullscreen>\n    <focus>yes</focus>\n  </application>\'\'\'\nif source.count(\'<applications>\') != 1:\n    raise SystemExit(\'Cannot locate Openbox application rules\')\nPath(sys.argv[1]).write_text(source.replace(\'<applications>\', rule, 1), encoding=\'utf-8\')\nPY\nstart_desktop_program() {\n    local label="$1"\n    shift\n    local pidfile="$BENCH/direct-${label}.pid"\n    if [ -f "$pidfile" ] && kill -0 "$(cat "$pidfile")" 2>/dev/null; then\n        return\n    fi\n    runuser -u gamer -- env DISPLAY=:1 HOME=/home/gamer \\\n        PULSE_SERVER="$PULSE_SERVER" SDL_AUDIODRIVER=pulse \\\n        CUDA_VISIBLE_DEVICES=0 GAMELAB_DIRECT_XORG=1 \\\n        SELKIES_JS_SOCKET_PATH="$SELKIES_JS_SOCKET_PATH" \\\n        PYTHONPATH=/root/.local/lib/python3.13/site-packages:/usr/local/lib/python3.13/dist-packages \\\n        "$@" > "$BENCH/logs/direct-${label}.log" 2>&1 < /dev/null &\n    echo $! > "$pidfile"\n}\nstart_desktop() {\n    chmod 666 "$SELKIES_JS_SOCKET_PATH"/*.sock 2>/dev/null || true\n    start_desktop_program openbox openbox --config-file "$BENCH/direct-openbox-rc.xml"\n    if command -v tint2 >/dev/null; then\n        start_desktop_program tint2 tint2 -c "$BENCH/tint2rc"\n    fi\n    start_desktop_program dashboard python3 "$BENCH/dev_launcher.py"\n    start_desktop_program legendary python3 "$BENCH/legendary_gui.py"\n}\nif curl -fsS "http://127.0.0.1:${PORT}/api/health" >/dev/null 2>&1; then\n    start_desktop\n    echo "Direct desktop already streaming on port $PORT"\n    exit 0\nfi\n# This process runs as root because Xorg rejected the gamer\'s MIT-SHM attach.\n# Disable command execution and transfers on this privileged stream.\nnohup env SELKIES_ENCODER_DEVICE="$ENCODER_DEVICE" SELKIES_ENCODER=h264enc \\\n    python3 -m selkies --port="$PORT" --enable-basic-auth=false \\\n    --command-enabled=false --file-transfers=none --printing-enabled=false \\\n    --web-root="$BENCH/web-root" --ui-title=\'7BZ Dev Cloud Game\' \\\n    --ui-show-sidebar=true --ui-show-logo=false \\\n    --manual-resolution=true --manual-width="$WIDTH" --manual-height="$HEIGHT" \\\n    > "$BENCH/logs/direct-stream.log" 2>&1 < /dev/null &\necho $! > "$BENCH/direct-stream.pid"\nfor attempt in $(seq 1 20); do\n    if curl -fsS "http://127.0.0.1:${PORT}/api/health" >/dev/null 2>&1; then\n        start_desktop\n        echo "Direct NVIDIA stream ready on port $PORT"\n        exit 0\n    fi\n    sleep 0.5\ndone\necho "Direct stream did not start; see $BENCH/logs/direct-stream.log" >&2\nexit 1\n', encoding='utf-8')

# 3. Execute setup.sh with live output streaming
setup_sh.chmod(0o755)
print('▶ Running automated setup.sh...', flush=True)
proc = subprocess.Popen(['/bin/bash', str(setup_sh)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print(f'setup.sh completed with exit code: {proc.returncode}', flush=True)

# 4. Configure unprivileged gamer user
subprocess.run(['useradd', '-m', '-s', '/bin/bash', '-G', 'audio,video', 'gamer'], check=False)
gamer_home = Path('/home/gamer')
for sub in ['.config', '.local', '.steam', 'game-builds', 'logs', 'ea-prefix']:
    (gamer_home / sub).mkdir(parents=True, exist_ok=True)

# Grant open permissions and ensure gamer ownership recursively
subprocess.run(['chown', '-R', 'gamer:gamer', str(gamer_home)], check=False)
subprocess.run(['chmod', '-R', '777', str(gamer_home)], check=False)
subprocess.run(['chmod', '-R', '777', str(BENCH)], check=False)

# 5. Storefront verification audit
def check_paths(candidates):
    for c in candidates:
        if c and os.path.exists(c):
            return c
        p = shutil.which(c)
        if p:
            return p
    return None

wine_ge_dir = gamer_home / '.config' / 'heroic' / 'tools' / 'wine'
storefront_status = {
    'steam': check_paths(['/usr/games/steam', '/usr/bin/steam']),
    'heroic': check_paths(['/usr/bin/heroic', '/opt/Heroic/heroic']),
    'legendary': check_paths(['/usr/local/bin/legendary', '/usr/bin/legendary', 'legendary']),
    'minigalaxy': check_paths(['/usr/games/minigalaxy', '/usr/bin/minigalaxy']),
    'lutris': check_paths(['/usr/games/lutris', '/usr/bin/lutris']),
    'wine': check_paths(['/usr/bin/wine', 'wine']),
    'wine_ge': check_paths([str(wine_ge_dir / 'Wine-GE-Proton8-26/bin/wine'), *[str(p) for p in wine_ge_dir.glob('*/bin/wine')]]),
    'ea_installer': str(EA_STORE / 'EAappInstaller.exe') if (EA_STORE / 'EAappInstaller.exe').exists() else None,
    'interposer_64': str(BENCH / 'lib/x86_64-linux-gnu/selkies_input_interposer.so') if (BENCH / 'lib/x86_64-linux-gnu/selkies_input_interposer.so').exists() else None,
    'interposer_32': str(BENCH / 'lib/i386-linux-gnu/selkies_input_interposer.so') if (BENCH / 'lib/i386-linux-gnu/selkies_input_interposer.so').exists() else None
}

print('================ STOREFRONT INSTALLATION AUDIT ================')
for k, v in storefront_status.items():
    print(f'  {k:15}: {v if v else "NOT FOUND / PENDING"}')
print('================================================================')


In [ ]:
from pathlib import Path
import os, subprocess

BENCH = Path('/tmp/kaggle-game-dev-bench')
BENCH.mkdir(parents=True, exist_ok=True)
(BENCH / 'logs').mkdir(parents=True, exist_ok=True)
(BENCH / 'web_client').mkdir(parents=True, exist_ok=True)

# Write the console dashboard, catalog, emergency service, and streaming HUD
(BENCH / 'dev_launcher.py').write_text('"""Ten-foot console dashboard for 7BZ Dev Cloud Game."""\nimport hashlib\nimport os\nfrom pathlib import Path\nimport subprocess\nimport threading\nimport time\nimport urllib.request\nfrom typing import Any, Dict, List, Optional, Tuple\n\nimport pygame\n\nfrom game_catalog import STOREFRONTS, cache_path, discover_games\nfrom session_control import clear_active_game, write_active_game\n\n\nos.environ["PATH"] = f"/usr/games:/usr/local/bin:/usr/bin:{os.environ.get(\'PATH\', \'\')}"\nBENCH = Path("/tmp/kaggle-game-dev-bench")\nLOGS = Path("/home/gamer/logs")\nLOGS.mkdir(parents=True, exist_ok=True)\nWIDTH, HEIGHT = 1280, 720\nBG = (7, 10, 20)\nPANEL = (16, 22, 39)\nPANEL_LIGHT = (24, 33, 56)\nTEXT = (244, 247, 255)\nMUTED = (150, 162, 188)\nCYAN = (72, 215, 255)\nVIOLET = (143, 124, 255)\nGREEN = (78, 230, 168)\nTABS = ("HOME", "LIBRARY", "STORES", "BUILDS")\n\npygame.init()\npygame.display.set_caption("7BZ Dev Cloud Game")\ndisplay = pygame.display.set_mode((WIDTH, HEIGHT), pygame.RESIZABLE)\nscreen = pygame.Surface((WIDTH, HEIGHT)).convert()\nclock = pygame.time.Clock()\nfont_logo = pygame.font.SysFont("DejaVu Sans", 27, bold=True)\nfont_hero = pygame.font.SysFont("DejaVu Sans", 35, bold=True)\nfont_card = pygame.font.SysFont("DejaVu Sans", 17, bold=True)\nfont_body = pygame.font.SysFont("DejaVu Sans", 14)\nfont_small = pygame.font.SysFont("DejaVu Sans", 12)\nfont_tiny = pygame.font.SysFont("DejaVu Sans", 10, bold=True)\n\n\ndef pulse_server() -> str:\n    """Return the active PulseAudio socket for every child process."""\n    configured = os.environ.get("PULSE_SERVER", "")\n    if configured:\n        return configured\n    candidates = [Path("/run/user/1000/pulse/native")]\n    candidates.extend(Path("/tmp").glob("pulse-*/native"))\n    for candidate in candidates:\n        if candidate.exists():\n            return f"unix:{candidate}"\n    return ""\n\n\nINPUT_MODE_FILE = BENCH / "input-mode.json"\n\n\ndef get_launcher_input_mode() -> str:\n    try:\n        if INPUT_MODE_FILE.is_file():\n            data = json.loads(INPUT_MODE_FILE.read_text(encoding="utf-8"))\n            return data.get("mode", "mouse_keyboard")\n    except Exception:\n        pass\n    return "mouse_keyboard"\n\n\ndef toggle_launcher_input_mode() -> str:\n    curr = get_launcher_input_mode()\n    new_m = "gamepad" if curr == "mouse_keyboard" else "mouse_keyboard"\n    try:\n        INPUT_MODE_FILE.write_text(json.dumps({"mode": new_m, "updated": time.time()}), encoding="utf-8")\n    except Exception:\n        pass\n    return new_m\n\n\ndef game_env() -> Dict[str, str]:\n    """Build the common X11, controller, and PulseAudio environment."""\n    env = os.environ.copy()\n    env.update(DISPLAY=env.get("DISPLAY", ":0"), HOME="/home/gamer", SDL_AUDIODRIVER="pulse")\n    server = pulse_server()\n    if server:\n        env["PULSE_SERVER"] = server\n    mode = get_launcher_input_mode()\n    if mode == "mouse_keyboard":\n        env.pop("LD_PRELOAD", None)\n        env["SDL_JOYSTICK_DEVICE"] = ""\n        env["SDL_JOYSTICK_DISABLE_UDEV"] = "1"\n        env["WINEDLLOVERRIDES"] = env.get("WINEDLLOVERRIDES", "") + ";winebus.sys=d"\n    else:\n        env["SELKIES_JS_SOCKET_PATH"] = ("/tmp/kaggle-game-dev-bench/direct-stream-sockets"\n                                         if env.get("GAMELAB_DIRECT_XORG") == "1" else "/tmp")\n        env["SDL_JOYSTICK_DISABLE_UDEV"] = "1"\n        env["SDL_JOYSTICK_DEVICE"] = "/dev/input/event1000:/dev/input/event1001:/dev/input/event1002:/dev/input/event1003"\n        interposer = "/tmp/kaggle-game-dev-bench/$LIB/selkies_input_interposer.so:/tmp/kaggle-game-dev-bench/$LIB/selkies_fake_udev.so"\n        env["LD_PRELOAD"] = os.environ.get("LD_PRELOAD", interposer)\n    return env\n\n\ndef wine_runner() -> str:\n    candidates = [Path("/home/gamer/.config/heroic/tools/wine/Wine-GE-Proton8-26/bin/wine"),\n                  *Path("/home/gamer/.config/heroic/tools/wine").glob("*/bin/wine"), Path("/usr/bin/wine")]\n    return str(next((path for path in candidates if path.is_file()), Path("/usr/bin/wine")))\n\n\nclass Artwork:\n    """Download and cache game artwork without blocking the render loop."""\n\n    def __init__(self) -> None:\n        self.surfaces: Dict[str, pygame.Surface] = {}\n        self.pending = set()\n\n    def get(self, item: Dict[str, Any], size: Tuple[int, int]) -> Optional[pygame.Surface]:\n        key = f"{item[\'id\']}:{size[0]}x{size[1]}"\n        if key in self.surfaces:\n            return self.surfaces[key]\n        path = cache_path(item["id"])\n        if path.is_file():\n            try:\n                loaded = pygame.image.load(str(path)).convert()\n                self.surfaces[key] = pygame.transform.smoothscale(loaded, size)\n                return self.surfaces[key]\n            except pygame.error:\n                path.unlink(missing_ok=True)\n        url = item.get("art", "")\n        if url and item["id"] not in self.pending:\n            self.pending.add(item["id"])\n            threading.Thread(target=self._download, args=(item["id"], url), daemon=True).start()\n        return None\n\n    def _download(self, item_id: str, url: str) -> None:\n        path = cache_path(item_id)\n        try:\n            path.parent.mkdir(parents=True, exist_ok=True)\n            request = urllib.request.Request(url, headers={"User-Agent": "7BZ-Console/1.0"})\n            with urllib.request.urlopen(request, timeout=8) as response:\n                data = response.read(8 * 1024 * 1024)\n            if len(data) > 512:\n                path.write_bytes(data)\n        except (OSError, ValueError):\n            pass\n        finally:\n            self.pending.discard(item_id)\n\n\nartwork = Artwork()\ncatalog: List[Dict[str, Any]] = []\ntab = 0\nselected = 0\nlast_scan = 0.0\nmessage = "Choose a game and press A or Enter to play."\nactive: Optional[subprocess.Popen] = None\nactive_title = ""\naxis_ready = True\n\n\ndef storefront_items() -> List[Dict[str, Any]]:\n    return [{**item, "status": "Ready", "store": "Storefront", "subtitle": "Open your game library",\n             "action": "store", "art": ""} for item in STOREFRONTS]\n\n\ndef visible_items() -> List[Dict[str, Any]]:\n    if tab == 0:\n        return catalog[:8] + storefront_items()[:4] if catalog else storefront_items()\n    if tab == 1:\n        return catalog\n    if tab == 2:\n        return storefront_items()\n    return [item for item in catalog if item.get("action") == "build"]\n\n\ndef command_for(item: Dict[str, Any]) -> List[str]:\n    action = item.get("action")\n    if action == "legendary":\n        return ["python3", str(BENCH / "gpu_runtime.py"), "launch", item["app_name"], wine_runner()]\n    if action == "steam":\n        return ["bash", str(BENCH / "launch_steam.sh"), f"steam://rungameid/{item[\'app_name\']}"]\n    if action == "build":\n        path = item["install_path"]\n        return ["bash", path] if path.lower().endswith(".sh") else [path]\n    if action == "store":\n        return ["bash", item["script"]]\n    raise ValueError("This library item has no launch action")\n\n\ndef launch(item: Dict[str, Any]) -> str:\n    global active, active_title\n    if active is not None and active.poll() is None:\n        return f"{active_title} is already running. Press B to stop it first."\n    try:\n        command = command_for(item)\n        log = (LOGS / "dashboard-launch.log").open("a", encoding="utf-8", buffering=1)\n        log.write(f"\\n--- {time.strftime(\'%Y-%m-%d %H:%M:%S\')} · {item[\'title\']} ---\\n")\n        active = subprocess.Popen(command, env=game_env(), stdin=subprocess.DEVNULL,\n                                  stdout=log, stderr=subprocess.STDOUT, start_new_session=True)\n        active_title = item["title"]\n        write_active_game(active.pid, os.getpgid(active.pid), active_title, item.get("store", "7BZ"))\n        threading.Thread(target=watch_active, args=(active,), daemon=True).start()\n        return f"Launching {active_title}. GPU, audio, and mouse fixes are applied automatically."\n    except (OSError, ValueError) as exc:\n        return f"Launch failed: {str(exc)[:100]}"\n\n\ndef watch_active(process: subprocess.Popen) -> None:\n    process.wait()\n    clear_active_game(process.pid)\n\n\ndef stop_active() -> str:\n    if active is None or active.poll() is not None:\n        return "No active game is running."\n    try:\n        os.killpg(os.getpgid(active.pid), 15)\n        return f"Stopping {active_title}…"\n    except OSError as exc:\n        return f"Could not stop the game: {exc}"\n\n\ndef clipped(text: str, font: pygame.font.Font, width: int) -> str:\n    result = text\n    while result and font.size(result + "…")[0] > width:\n        result = result[:-1]\n    return result + ("…" if result != text else "")\n\n\ndef gradient(rect: pygame.Rect, top: Tuple[int, int, int], bottom: Tuple[int, int, int]) -> None:\n    for offset in range(rect.height):\n        ratio = offset / max(1, rect.height - 1)\n        color = tuple(int(top[index] + (bottom[index] - top[index]) * ratio) for index in range(3))\n        pygame.draw.line(screen, color, (rect.x, rect.y + offset), (rect.right, rect.y + offset))\n\n\ndef placeholder(item: Dict[str, Any], rect: pygame.Rect) -> None:\n    digest = hashlib.sha256(item["id"].encode()).digest()\n    color_a = (30 + digest[0] % 45, 35 + digest[1] % 55, 65 + digest[2] % 70)\n    gradient(rect, color_a, (12, 14, 27))\n    pygame.draw.circle(screen, CYAN, (rect.right - 35, rect.y + 30), 54, 2)\n    badge = item.get("store", item.get("badge", "7BZ")).upper()[:8]\n    screen.blit(font_tiny.render(badge, True, (220, 231, 255)), (rect.x + 13, rect.y + 12))\n\n\ndef draw_art(item: Dict[str, Any], rect: pygame.Rect) -> None:\n    image = artwork.get(item, rect.size)\n    if image:\n        screen.blit(image, rect)\n        shade = pygame.Surface(rect.size, pygame.SRCALPHA)\n        shade.fill((4, 8, 18, 50))\n        screen.blit(shade, rect)\n    else:\n        placeholder(item, rect)\n\n\ndef draw_header(width: int) -> None:\n    pygame.draw.rect(screen, (10, 15, 29), (0, 0, width, 88))\n    pygame.draw.rect(screen, CYAN, (0, 0, 5, 88))\n    pygame.draw.rect(screen, VIOLET, (5, 0, 3, 88))\n    screen.blit(font_logo.render("7BZ", True, TEXT), (30, 16))\n    screen.blit(font_body.render("DEV CLOUD GAME", True, CYAN), (94, 24))\n    x = 360\n    for index, label in enumerate(TABS):\n        color = TEXT if index == tab else MUTED\n        surface = font_body.render(label, True, color)\n        screen.blit(surface, (x, 25))\n        if index == tab:\n            pygame.draw.rect(screen, CYAN, (x, 52, surface.get_width(), 3), border_radius=2)\n        x += surface.get_width() + 38\n    has_dual_gpu = Path("/dev/nvidia1").exists() and Path("/dev/nvidia0").exists()\n    gpu_badge = "DUAL GPU: G0 RENDER · G1 STREAM" if has_dual_gpu else "NVIDIA GPU ACCELERATED"\n    screen.blit(font_tiny.render(gpu_badge, True, VIOLET), (width - 380, 26))\n    status = "GAME ACTIVE" if active is not None and active.poll() is None else "SYSTEM READY"\n    color = CYAN if status == "GAME ACTIVE" else GREEN\n    screen.blit(font_tiny.render(f"●  {status}", True, color), (width - 160, 26))\n    screen.blit(font_small.render(time.strftime("%H:%M"), True, MUTED), (width - 61, 48))\n\n\ndef draw_featured(item: Dict[str, Any], width: int) -> None:\n    rect = pygame.Rect(28, 105, width - 56, 214)\n    draw_art(item, rect)\n    overlay = pygame.Surface(rect.size, pygame.SRCALPHA)\n    for x in range(rect.width):\n        alpha = max(0, 215 - int(x / rect.width * 245))\n        pygame.draw.line(overlay, (5, 9, 19, alpha), (x, 0), (x, rect.height))\n    screen.blit(overlay, rect)\n    screen.blit(font_tiny.render(f"{item.get(\'status\', \'READY\').upper()}  ·  {item.get(\'store\', \'7BZ\').upper()}", True, GREEN), (52, 133))\n    screen.blit(font_hero.render(clipped(item["title"], font_hero, 590), True, TEXT), (50, 160))\n    screen.blit(font_body.render(item.get("subtitle", "Ready to play"), True, (210, 221, 244)), (52, 207))\n    play = pygame.Rect(52, 251, 126, 43)\n    pygame.draw.rect(screen, TEXT, play, border_radius=10)\n    screen.blit(font_card.render("PLAY   A", True, BG), (72, 262))\n    screen.blit(font_small.render("Mouse, keyboard, and gamepad ready", True, MUTED), (196, 267))\n\n\ndef draw_cards(items: List[Dict[str, Any]], width: int, chosen: int) -> List[Tuple[int, pygame.Rect]]:\n    cards: List[Tuple[int, pygame.Rect]] = []\n    screen.blit(font_tiny.render("YOUR CLOUD LIBRARY", True, MUTED), (30, 346))\n    gap, columns = 14, 4\n    card_width = (width - 60 - gap * (columns - 1)) // columns\n    page_start = (chosen // 8) * 8\n    for local_index, item in enumerate(items[page_start:page_start + 8]):\n        index = page_start + local_index\n        row, column = divmod(local_index, columns)\n        rect = pygame.Rect(30 + column * (card_width + gap), 372 + row * 137, card_width, 121)\n        cards.append((index, rect))\n        selected_now = index == chosen\n        pygame.draw.rect(screen, PANEL_LIGHT if selected_now else PANEL, rect, border_radius=13)\n        draw_art(item, pygame.Rect(rect.x, rect.y, rect.width, 72))\n        pygame.draw.rect(screen, CYAN if selected_now else (47, 59, 84), rect, 3 if selected_now else 1, border_radius=13)\n        screen.blit(font_card.render(clipped(item["title"], font_card, rect.width - 20), True, TEXT), (rect.x + 10, rect.y + 80))\n        screen.blit(font_tiny.render(item.get("store", "7BZ").upper(), True, CYAN if selected_now else MUTED), (rect.x + 10, rect.y + 103))\n    return cards\n\n\ndef draw_footer(width: int, height: int) -> None:\n    pygame.draw.rect(screen, (10, 15, 29), (0, height - 42, width, 42))\n    mode_text = "🖱️ Mouse & Keyboard" if get_launcher_input_mode() == "mouse_keyboard" else "🎮 Gamepad (Xbox 360)"\n    mode_label = f"[{mode_text}] (M/I to switch)"\n    screen.blit(font_small.render(clipped(f"{mode_label}  ·  {message}", font_small, width - 520), True, CYAN), (28, height - 28))\n    hint = "D-Pad Move   A Play   B Stop   M/Select Mode   LB/RB Tabs"\n    screen.blit(font_small.render(hint, True, (195, 205, 227)), (width - font_small.size(hint)[0] - 28, height - 28))\n\n\nrunning = True\nwhile running:\n    now = time.monotonic()\n    if now - last_scan > 4:\n        catalog = discover_games()\n        last_scan = now\n    items = visible_items()\n    selected = min(selected, max(0, len(items) - 1))\n    current = items[selected] if items else {"id": "empty", "title": "No games found", "store": "7BZ",\n                                             "status": "Library", "subtitle": "Open a storefront to install a game", "art": ""}\n    w, h = WIDTH, HEIGHT\n    for event in pygame.event.get():\n        if event.type == pygame.QUIT:\n            running = False\n        elif event.type == pygame.VIDEORESIZE:\n            display = pygame.display.set_mode((max(640, event.w), max(360, event.h)), pygame.RESIZABLE)\n        elif event.type == pygame.KEYDOWN:\n            if event.key in (pygame.K_LEFT, pygame.K_a): selected = max(0, selected - 1)\n            elif event.key in (pygame.K_RIGHT, pygame.K_d): selected = min(len(items) - 1, selected + 1)\n            elif event.key in (pygame.K_UP, pygame.K_w): selected = max(0, selected - 4)\n            elif event.key in (pygame.K_DOWN, pygame.K_s): selected = min(len(items) - 1, selected + 4)\n            elif event.key in (pygame.K_RETURN, pygame.K_SPACE) and items: message = launch(current)\n            elif event.key in (pygame.K_BACKSPACE, pygame.K_DELETE, pygame.K_ESCAPE): message = stop_active()\n            elif event.key in (pygame.K_m, pygame.K_i):\n                new_m = toggle_launcher_input_mode()\n                message = f"Input Mode: {\'🖱️ Mouse & Keyboard\' if new_m == \'mouse_keyboard\' else \'🎮 Controller (Xbox 360)\'}"\n            elif event.key == pygame.K_TAB:\n                tab = (tab + (-1 if pygame.key.get_mods() & pygame.KMOD_SHIFT else 1)) % len(TABS); selected = 0\n        elif event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:\n            mx = round(event.pos[0] * WIDTH / max(1, display.get_width()))\n            my = round(event.pos[1] * HEIGHT / max(1, display.get_height()))\n            if 345 <= mx <= 900 and 8 <= my <= 70:\n                tab_x = 360\n                for tab_index, label in enumerate(TABS):\n                    tab_width = font_body.size(label)[0] + 38\n                    if tab_x <= mx <= tab_x + tab_width:\n                        tab, selected = tab_index, 0\n                        break\n                    tab_x += tab_width\n            elif 52 <= mx <= 178 and 251 <= my <= 294 and items:\n                message = launch(current)\n            else:\n                cards = draw_cards(items, w, selected)\n                for index, rect in cards:\n                    if rect.collidepoint((mx, my)):\n                        if index == selected: message = launch(items[index])\n                        selected = index\n        elif event.type == pygame.JOYBUTTONDOWN:\n            if event.button == 0 and items: message = launch(current)\n            elif event.button == 1: message = stop_active()\n            elif event.button in (3, 8):\n                new_m = toggle_launcher_input_mode()\n                message = f"Input Mode: {\'🖱️ Mouse & Keyboard\' if new_m == \'mouse_keyboard\' else \'🎮 Controller (Xbox 360)\'}"\n            elif event.button in (4, 9): tab = (tab - 1) % len(TABS); selected = 0\n            elif event.button in (5, 10): tab = (tab + 1) % len(TABS); selected = 0\n            elif event.button in (11, 12): selected = max(0, selected - 4)\n            elif event.button in (13, 14): selected = min(len(items) - 1, selected + 4)\n        elif event.type == pygame.JOYAXISMOTION and event.axis in (0, 1):\n            if abs(event.value) < .35:\n                axis_ready = True\n            elif axis_ready and items:\n                selected = max(0, min(len(items) - 1, selected + ((1 if event.value > 0 else -1) if event.axis == 0 else (4 if event.value > 0 else -4))))\n                axis_ready = False\n\n    gradient(pygame.Rect(0, 0, w, h), BG, (13, 17, 31))\n    draw_header(w)\n    draw_featured(current, w)\n    if items:\n        draw_cards(items, w, selected)\n    else:\n        screen.blit(font_body.render("No items are available in this section yet.", True, MUTED), (31, 370))\n    draw_footer(w, h)\n    output = screen if display.get_size() == (WIDTH, HEIGHT) else pygame.transform.smoothscale(screen, display.get_size())\n    display.blit(output, (0, 0))\n    pygame.display.flip()\n    clock.tick(60)\n\npygame.quit()\n', encoding='utf-8')
(BENCH / 'game_catalog.py').write_text('"""Discover games and storefronts for the 7BZ console dashboard."""\nimport json\nimport os\nfrom pathlib import Path\nfrom typing import Any, Dict, Iterable, List, Optional\n\n\nHOME = Path(os.environ.get("KAGGLE_GAMER_HOME", "/home/gamer"))\nBUILDS = Path(os.environ.get("GAME_BUILDS_DIR", "/tmp/game-builds"))\nCACHE = HOME / ".cache" / "7bz-dev-cloud-game"\n\n\nSTOREFRONTS: List[Dict[str, Any]] = [\n    {"id": "legendary", "title": "Epic / Legendary", "badge": "EPIC", "color": (50, 56, 70),\n     "script": "/tmp/kaggle-game-dev-bench/launch_legendary.sh"},\n    {"id": "steam", "title": "Steam", "badge": "STEAM", "color": (24, 83, 122),\n     "script": "/tmp/kaggle-game-dev-bench/launch_steam.sh"},\n    {"id": "heroic", "title": "Heroic", "badge": "HEROIC", "color": (45, 111, 177),\n     "script": "/tmp/kaggle-game-dev-bench/launch_heroic.sh"},\n    {"id": "lutris", "title": "Lutris", "badge": "LUTRIS", "color": (225, 123, 44),\n     "script": "/tmp/kaggle-game-dev-bench/launch_lutris.sh"},\n]\n\n\ndef _read_json(path: Path) -> Any:\n    try:\n        return json.loads(path.read_text(encoding="utf-8"))\n    except (OSError, ValueError):\n        return None\n\n\ndef _image_from_metadata(value: Any) -> str:\n    """Return the best landscape image URL found in Legendary metadata."""\n    if not isinstance(value, dict):\n        return ""\n    images = value.get("keyImages") or value.get("key_images") or []\n    if not isinstance(images, list):\n        return ""\n    preferred = ("DieselGameBoxWide", "OfferImageWide", "Featured", "Thumbnail")\n    for image_type in preferred:\n        for entry in images:\n            if (isinstance(entry, dict) and entry.get("type") == image_type\n                    and isinstance(entry.get("url"), str)):\n                return entry["url"]\n    return next((entry.get("url", "") for entry in images\n                 if isinstance(entry, dict) and entry.get("url")), "")\n\n\ndef _legendary_metadata(app_name: str) -> Dict[str, Any]:\n    candidates = (\n        HOME / ".config" / "heroic" / "legendaryConfig" / "legendary" / "metadata" / f"{app_name}.json",\n        HOME / ".config" / "legendary" / "metadata" / f"{app_name}.json",\n    )\n    for path in candidates:\n        data = _read_json(path)\n        if isinstance(data, dict):\n            return data\n    return {}\n\n\ndef legendary_games() -> List[Dict[str, Any]]:\n    """Return installed Epic games from Heroic\'s or Legendary\'s database."""\n    candidates = (\n        HOME / ".config" / "heroic" / "legendaryConfig" / "legendary" / "installed.json",\n        HOME / ".config" / "legendary" / "installed.json",\n    )\n    raw: Any = None\n    for path in candidates:\n        raw = _read_json(path)\n        if raw:\n            break\n    if not isinstance(raw, dict):\n        return []\n    games: List[Dict[str, Any]] = []\n    for app_name, record in raw.items():\n        if not isinstance(record, dict):\n            continue\n        metadata = _legendary_metadata(str(app_name))\n        title = str(record.get("title") or metadata.get("app_title") or app_name)\n        games.append({\n            "id": f"epic:{app_name}", "title": title, "store": "Epic Games",\n            "status": "Installed", "app_name": str(app_name),\n            "install_path": str(record.get("install_path") or record.get("installPath") or ""),\n            "art": _image_from_metadata(metadata), "action": "legendary",\n            "subtitle": str(record.get("version") or "Ready to play"),\n        })\n    return games\n\n\ndef _steam_library_paths() -> Iterable[Path]:\n    yield HOME / ".steam" / "steam" / "steamapps"\n    yield HOME / ".local" / "share" / "Steam" / "steamapps"\n    library_file = HOME / ".steam" / "steam" / "steamapps" / "libraryfolders.vdf"\n    try:\n        for line in library_file.read_text(encoding="utf-8", errors="ignore").splitlines():\n            if \'"path"\' in line:\n                parts = line.split(\'"\')\n                if len(parts) >= 4:\n                    yield Path(parts[3].replace("\\\\\\\\", "\\\\")) / "steamapps"\n    except OSError:\n        return\n\n\ndef steam_games() -> List[Dict[str, Any]]:\n    """Return installed Steam games from local app manifests."""\n    games: List[Dict[str, Any]] = []\n    seen = set()\n    for steamapps in _steam_library_paths():\n        for manifest in steamapps.glob("appmanifest_*.acf") if steamapps.is_dir() else ():\n            text = manifest.read_text(encoding="utf-8", errors="ignore")\n            fields: Dict[str, str] = {}\n            for line in text.splitlines():\n                parts = line.strip().split(\'"\')\n                if len(parts) >= 4:\n                    fields[parts[1]] = parts[3]\n            app_id = fields.get("appid", manifest.stem.removeprefix("appmanifest_"))\n            if app_id in seen:\n                continue\n            seen.add(app_id)\n            games.append({\n                "id": f"steam:{app_id}", "title": fields.get("name", f"Steam {app_id}"),\n                "store": "Steam", "status": "Installed", "app_name": app_id,\n                "install_path": str(steamapps / "common" / fields.get("installdir", "")),\n                "art": f"https://cdn.cloudflare.steamstatic.com/steam/apps/{app_id}/header.jpg",\n                "action": "steam", "subtitle": "Ready in Steam",\n            })\n    return games\n\n\ndef custom_builds() -> List[Dict[str, Any]]:\n    """Return executable custom builds uploaded to the shared build directory."""\n    items: List[Dict[str, Any]] = []\n    if not BUILDS.is_dir():\n        return items\n    for path in sorted(BUILDS.iterdir(), key=lambda item: item.name.lower()):\n        if path.is_file() and (os.access(path, os.X_OK) or path.suffix.lower() in (".sh", ".appimage", ".x86_64", ".bin")):\n            items.append({\n                "id": f"build:{path.name}", "title": path.stem, "store": "Custom Build",\n                "status": "Ready", "app_name": path.name, "install_path": str(path),\n                "art": "", "action": "build", "subtitle": "Uploaded development build",\n            })\n    return items\n\n\ndef discover_games() -> List[Dict[str, Any]]:\n    """Return a stable, title-sorted catalog from every supported local source."""\n    games = legendary_games() + steam_games() + custom_builds()\n    games.sort(key=lambda item: item["title"].lower())\n    return games\n\n\ndef cache_path(item_id: str) -> Path:\n    """Return the local artwork path for a catalog item."""\n    safe = "".join(ch if ch.isalnum() else "_" for ch in item_id)[:100]\n    return CACHE / f"{safe}.img"\n\n\ndef find_game(item_id: str) -> Optional[Dict[str, Any]]:\n    """Find one game by its stable catalog identifier."""\n    return next((game for game in discover_games() if game["id"] == item_id), None)\n', encoding='utf-8')
(BENCH / 'session_control.py').write_text('"""Local control service for the 7BZ streaming HUD and dashboard."""\nimport asyncio\nimport json\nimport os\nfrom pathlib import Path\nimport signal\nimport subprocess\nimport time\nfrom typing import Any, Dict, List\n\nfrom aiohttp import web\n\nfrom game_catalog import discover_games\n\n\nBENCH = Path("/tmp/kaggle-game-dev-bench")\nSTATE = BENCH / "active-game.json"\nPORT = int(os.environ.get("SEVEN_BZ_CONTROL_PORT", "8766"))\n\n\ndef get_gpu_metrics() -> List[Dict[str, Any]]:\n    """Query live metrics for all available NVIDIA GPUs."""\n    try:\n        res = subprocess.run(\n            ["nvidia-smi", "--query-gpu=index,name,utilization.gpu,memory.used,memory.total",\n             "--format=csv,noheader,nounits"],\n            stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True, timeout=2\n        )\n        if res.returncode != 0:\n            return []\n        gpus = []\n        for line in res.stdout.strip().splitlines():\n            parts = [p.strip() for p in line.split(",")]\n            if len(parts) >= 5:\n                idx = int(parts[0])\n                role = "Render (Game)" if idx == 0 else "Encode (Stream)"\n                gpus.append({\n                    "index": idx,\n                    "name": parts[1],\n                    "role": role,\n                    "utilization": int(parts[2]),\n                    "memory_used": int(parts[3]),\n                    "memory_total": int(parts[4]),\n                })\n        return gpus\n    except Exception:\n        return []\n\n\ndef write_active_game(pid: int, pgid: int, title: str, source: str) -> None:\n    """Record a dashboard-launched process after validating its identity."""\n    STATE.write_text(json.dumps({"pid": pid, "pgid": pgid, "title": title,\n                                 "source": source, "started": time.time()}), encoding="utf-8")\n\n\ndef clear_active_game(pid: int = 0) -> None:\n    """Remove the active record when it still belongs to the given process."""\n    try:\n        current = json.loads(STATE.read_text(encoding="utf-8"))\n        if not pid or int(current.get("pid", 0)) == pid:\n            STATE.unlink(missing_ok=True)\n    except (OSError, ValueError, TypeError):\n        STATE.unlink(missing_ok=True)\n\n\ndef _state() -> Dict[str, Any]:\n    state_data: Dict[str, Any] = {"running": False, "title": "No active game"}\n    try:\n        data = json.loads(STATE.read_text(encoding="utf-8"))\n        pid = int(data.get("pid", 0))\n        if pid > 1 and Path(f"/proc/{pid}").exists():\n            data["running"] = True\n            state_data = data\n    except (OSError, ValueError, TypeError):\n        pass\n    state_data["gpus"] = get_gpu_metrics()\n    return state_data\n\n\ndef _owned_process(pid: int) -> bool:\n    """Confirm a recorded process exists and belongs to this service\'s user."""\n    try:\n        return Path(f"/proc/{pid}").stat().st_uid == os.getuid()\n    except OSError:\n        return False\n\n\nasync def _end_group(pgid: int, sig: signal.Signals) -> bool:\n    try:\n        os.killpg(pgid, sig)\n        return True\n    except (ProcessLookupError, PermissionError, OSError):\n        return False\n\n\nasync def reset_session() -> Dict[str, Any]:\n    """Stop the tracked game group, Wine servers, and restore the dashboard."""\n    state = _state()\n    stopped = False\n    pid = int(state.get("pid", 0) or 0)\n    pgid = int(state.get("pgid", 0) or 0)\n    if pid > 1 and pgid > 1 and _owned_process(pid):\n        stopped = await _end_group(pgid, signal.SIGTERM)\n        for _ in range(12):\n            if not Path(f"/proc/{pid}").exists():\n                break\n            await asyncio.sleep(0.25)\n        if Path(f"/proc/{pid}").exists():\n            await _end_group(pgid, signal.SIGKILL)\n    clear_active_game(pid)\n    subprocess.run(["wineserver", "-k"], check=False, stdout=subprocess.DEVNULL,\n                   stderr=subprocess.DEVNULL, timeout=5)\n    subprocess.run(["wmctrl", "-a", "7BZ Dev Cloud Game"], check=False,\n                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=3)\n    return {"ok": True, "stopped": stopped, "message": "Game stopped. Dashboard restored."}\n\n\nasync def health(_request: web.Request) -> web.Response:\n    return web.json_response({"ok": True, "service": "7bz-control"})\n\n\nasync def status(_request: web.Request) -> web.Response:\n    return web.json_response(_state())\n\n\nasync def catalog(_request: web.Request) -> web.Response:\n    return web.json_response({"games": discover_games()})\n\n\nINPUT_MODE_FILE = BENCH / "input-mode.json"\n\n\ndef _get_input_mode() -> str:\n    try:\n        if INPUT_MODE_FILE.is_file():\n            data = json.loads(INPUT_MODE_FILE.read_text(encoding="utf-8"))\n            return data.get("mode", "mouse_keyboard")\n    except Exception:\n        pass\n    return "mouse_keyboard"\n\n\ndef _set_input_mode(mode: str) -> str:\n    valid = "gamepad" if mode == "gamepad" else "mouse_keyboard"\n    try:\n        INPUT_MODE_FILE.write_text(json.dumps({"mode": valid, "updated": time.time()}), encoding="utf-8")\n    except Exception:\n        pass\n    return valid\n\n\nasync def input_mode_get(_request: web.Request) -> web.Response:\n    return web.json_response({"mode": _get_input_mode()})\n\n\nasync def input_mode_post(request: web.Request) -> web.Response:\n    if request.headers.get("X-7BZ-Action") != "hud":\n        raise web.HTTPForbidden(text="Missing HUD action header")\n    try:\n        body = await request.json()\n        target_mode = body.get("mode", "")\n    except Exception:\n        target_mode = ""\n    if not target_mode:\n        current = _get_input_mode()\n        target_mode = "gamepad" if current == "mouse_keyboard" else "mouse_keyboard"\n    new_mode = _set_input_mode(target_mode)\n    return web.json_response({"ok": True, "mode": new_mode, "message": f"Input mode set to {new_mode}"})\n\n\nasync def reset(request: web.Request) -> web.Response:\n    if request.headers.get("X-7BZ-Action") != "hud":\n        raise web.HTTPForbidden(text="Missing HUD action header")\n    return web.json_response(await reset_session())\n\n\nasync def gpus_handler(_request: web.Request) -> web.Response:\n    return web.json_response({"gpus": get_gpu_metrics()})\n\n\ndef main() -> None:\n    """Serve narrow session actions on the loopback-only control port."""\n    app = web.Application(client_max_size=16 * 1024)\n    app.add_routes([web.get("/health", health), web.get("/status", status),\n                    web.get("/catalog", catalog), web.get("/gpus", gpus_handler),\n                    web.post("/reset", reset),\n                    web.get("/input-mode", input_mode_get), web.post("/input-mode", input_mode_post)])\n    web.run_app(app, host="127.0.0.1", port=PORT, access_log=None)\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
(BENCH / 'install_web_client.py').write_text('"""Create a branded Selkies web root with the 7BZ streaming HUD."""\nfrom importlib import resources\nfrom pathlib import Path\nimport shutil\n\n\nBENCH = Path("/tmp/kaggle-game-dev-bench")\nWEB_ROOT = BENCH / "web-root"\nASSETS = BENCH / "web_client"\nMARKER = "7BZ Dev Cloud Game HUD"\n\n\ndef _source_root() -> Path:\n    import selkies\n    direct = Path(selkies.__file__).resolve().parent / "selkies_web"\n    if direct.is_dir():\n        return direct\n    package = resources.files("selkies.selkies_web")\n    with resources.as_file(package) as extracted:\n        if extracted.is_dir():\n            temporary = BENCH / "selkies-web-source"\n            shutil.copytree(extracted, temporary, dirs_exist_ok=True)\n            return temporary\n    raise RuntimeError("Selkies web assets were not found")\n\n\ndef install() -> Path:\n    """Copy the installed client, add branded assets, and return its web root."""\n    import re, time\n    v = int(time.time())\n    source = _source_root()\n    shutil.copytree(source, WEB_ROOT, dirs_exist_ok=True)\n    shutil.copy2(ASSETS / "hud.css", WEB_ROOT / "7bz-hud.css")\n    shutil.copy2(ASSETS / "hud.js", WEB_ROOT / "7bz-hud.js")\n    index = WEB_ROOT / "index.html"\n    text = index.read_text(encoding="utf-8")\n    if MARKER not in text:\n        head = f\'<!-- {MARKER} -->\\n<link rel="stylesheet" href="7bz-hud.css?v={v}">\\n\'\n        script = f\'<script defer src="7bz-hud.js?v={v}"></script>\\n\'\n        text = text.replace("</head>", head + "</head>")\n        text = text.replace("</body>", script + "</body>")\n    else:\n        text = re.sub(r\'7bz-hud\\.css(\\?v=\\d+)?\', f\'7bz-hud.css?v={v}\', text)\n        text = re.sub(r\'7bz-hud\\.js(\\?v=\\d+)?\', f\'7bz-hud.js?v={v}\', text)\n    text = text.replace("<title>Selkies</title>", "<title>7BZ Dev Cloud Game</title>")\n    index.write_text(text, encoding="utf-8")\n    return WEB_ROOT\n\n\nif __name__ == "__main__":\n    print(install())\n', encoding='utf-8')
(BENCH / 'web_client' / 'hud.css').write_text(':root {\n  --seven-bg: rgba(7, 12, 25, .72);\n  --seven-panel: rgba(17, 25, 47, .76);\n  --seven-line: rgba(137, 183, 255, .18);\n  --seven-text: #f7f9ff;\n  --seven-muted: #9aa8c4;\n  --seven-cyan: #48d7ff;\n  --seven-violet: #8f7cff;\n  --seven-danger: #ff526b;\n  --seven-good: #4ee6a8;\n}\n\n#seven-bz-hud, #seven-bz-hud * { box-sizing: border-box; }\n\n#seven-bz-hud {\n  position: fixed;\n  z-index: 2147483000;\n  top: 14px;\n  left: 50%;\n  width: min(1080px, calc(100vw - 28px));\n  transform: translateX(-50%);\n  color: var(--seven-text);\n  font-family: Inter, ui-sans-serif, system-ui, -apple-system, "Segoe UI", sans-serif;\n  pointer-events: none;\n}\n\n.seven-shell {\n  pointer-events: auto;\n  overflow: hidden;\n  border: 1px solid var(--seven-line);\n  border-radius: 18px;\n  background: linear-gradient(120deg, rgba(12, 20, 40, .88), var(--seven-bg));\n  box-shadow: 0 18px 55px rgba(0, 0, 0, .45), inset 0 1px rgba(255,255,255,.08);\n  backdrop-filter: blur(22px) saturate(150%);\n  -webkit-backdrop-filter: blur(22px) saturate(150%);\n}\n\n.seven-topbar { display: flex; align-items: center; min-height: 48px; padding: 8px 12px 8px 16px; gap: 14px; }\n.seven-brand { display: flex; align-items: center; gap: 10px; min-width: 215px; font-weight: 760; letter-spacing: .02em; }\n.seven-mark { width: 24px; height: 24px; border-radius: 8px; background: linear-gradient(140deg, var(--seven-cyan), var(--seven-violet)); box-shadow: 0 0 22px rgba(72,215,255,.32); }\n.seven-live { display: inline-flex; align-items: center; gap: 6px; color: var(--seven-good); font-size: 11px; font-weight: 700; letter-spacing: .12em; }\n.seven-live::before { content: ""; width: 7px; height: 7px; border-radius: 50%; background: currentColor; box-shadow: 0 0 12px currentColor; }\n.seven-quick { display: flex; flex: 1; align-items: center; justify-content: center; gap: 7px; }\n.seven-chip { min-width: 86px; padding: 6px 9px; border-radius: 10px; background: rgba(255,255,255,.055); border: 1px solid rgba(255,255,255,.055); text-align: center; }\n.seven-chip b { display: block; font-size: 13px; line-height: 14px; font-weight: 720; }\n.seven-chip span { color: var(--seven-muted); font-size: 9px; text-transform: uppercase; letter-spacing: .1em; }\n\n.seven-button { appearance: none; border: 1px solid rgba(255,255,255,.1); border-radius: 10px; padding: 8px 11px; background: rgba(255,255,255,.07); color: var(--seven-text); font: inherit; font-size: 12px; font-weight: 650; cursor: pointer; transition: .16s ease; white-space: nowrap; }\n.seven-button:hover, .seven-button:focus-visible { transform: translateY(-1px); border-color: rgba(72,215,255,.55); background: rgba(72,215,255,.14); outline: none; }\n.seven-button.is-active { color: #07111f; background: linear-gradient(120deg, var(--seven-cyan), #75f1d0); border-color: transparent; }\n.seven-button.seven-danger { color: #ffeef1; border-color: rgba(255,82,107,.38); background: rgba(255,82,107,.15); }\n.seven-button.seven-danger:hover { background: rgba(255,82,107,.3); border-color: var(--seven-danger); }\n.seven-icon-button { width: 34px; height: 34px; padding: 0; font-size: 17px; }\n\n.seven-details { display: grid; grid-template-columns: 1fr 1fr; gap: 10px; max-height: 340px; padding: 0 12px 12px; transition: max-height .22s ease, padding .22s ease, opacity .18s ease; }\n.seven-details.is-collapsed { max-height: 0; padding-top: 0; padding-bottom: 0; opacity: 0; pointer-events: none; }\n.seven-card { border: 1px solid rgba(255,255,255,.07); border-radius: 14px; background: var(--seven-panel); padding: 12px; }\n.seven-card-title { display: flex; justify-content: space-between; align-items: center; margin-bottom: 10px; color: var(--seven-muted); font-size: 10px; font-weight: 740; letter-spacing: .12em; text-transform: uppercase; }\n.seven-metrics { display: grid; grid-template-columns: repeat(4, 1fr); gap: 7px; }\n.seven-metric { padding: 9px; border-radius: 11px; background: rgba(255,255,255,.04); }\n.seven-metric strong { display: block; font-size: 16px; }\n.seven-metric small { color: var(--seven-muted); font-size: 9px; text-transform: uppercase; letter-spacing: .08em; }\n.seven-actions { display: flex; flex-wrap: wrap; gap: 7px; }\n.seven-optimize { display: flex; align-items: center; justify-content: space-between; gap: 12px; margin-bottom: 10px; }\n.seven-optimize-copy b { display: block; font-size: 14px; }\n.seven-optimize-copy span { color: var(--seven-muted); font-size: 11px; }\n.seven-notice { grid-column: 1 / -1; display: flex; align-items: center; justify-content: space-between; gap: 12px; min-height: 34px; padding: 7px 12px; border: 1px solid rgba(143,124,255,.2); border-radius: 11px; background: rgba(143,124,255,.09); color: #dcd7ff; font-size: 11px; }\n.seven-toast { position: absolute; top: calc(100% + 8px); right: 8px; max-width: 380px; padding: 10px 14px; border-radius: 11px; background: rgba(7,12,25,.94); border: 1px solid var(--seven-line); box-shadow: 0 12px 30px rgba(0,0,0,.36); font-size: 12px; opacity: 0; transform: translateY(-5px); transition: .18s ease; pointer-events: none; }\n.seven-toast.is-visible { opacity: 1; transform: none; }\n\n@media (max-width: 850px) {\n  .seven-brand { min-width: auto; }\n  .seven-brand-name { display: none; }\n  .seven-quick .seven-chip:nth-child(n+4) { display: none; }\n  .seven-details { grid-template-columns: 1fr; }\n  .seven-notice { grid-column: 1; }\n}\n\n@media (max-width: 560px) {\n  #seven-bz-hud { top: 6px; width: calc(100vw - 12px); }\n  .seven-topbar { gap: 6px; padding: 6px 8px; }\n  .seven-live { display: none; }\n  .seven-quick { justify-content: flex-start; overflow: hidden; }\n  .seven-chip { min-width: 67px; padding: 5px; }\n  .seven-quick .seven-chip:nth-child(n+3) { display: none; }\n  .seven-metrics { grid-template-columns: repeat(2, 1fr); }\n}\n', encoding='utf-8')
(BENCH / 'web_client' / 'hud.js').write_text('/** 7BZ Dev Cloud Game streaming HUD. */\n(() => {\n  "use strict";\n  if (window.__sevenBzHud) return;\n  window.__sevenBzHud = true;\n\n  const post = (message) => window.postMessage(message, window.location.origin);\n  const value = (input, fallback = 0) => Number.isFinite(Number(input)) ? Number(input) : fallback;\n  const format = (number, digits = 0) => value(number).toFixed(digits);\n  const state = {\n    expanded: false, auto: false, level: 2, lastLost: 0, lastTune: 0,\n    rtts: [], audio: true, pointer: false, esc: false, toastTimer: 0,\n    hasGamepad: false, inputMode: "mouse_keyboard",\n  };\n  const levels = [\n    { bitrate: 2000, width: 854, height: 480, fps: 30, name: "Recovery" },\n    { bitrate: 4000, width: 1024, height: 576, fps: 45, name: "Balanced" },\n    { bitrate: 8000, width: 1280, height: 720, fps: 60, name: "High" },\n    { bitrate: 12000, width: 1280, height: 720, fps: 60, name: "Ultra" },\n    { bitrate: 15000, width: 1280, height: 720, fps: 60, name: "Maximum" },\n  ];\n\n  const hud = document.createElement("section");\n  hud.id = "seven-bz-hud";\n  hud.setAttribute("aria-label", "7BZ streaming controls");\n  hud.innerHTML = `\n    <div class="seven-shell">\n      <div class="seven-topbar">\n        <div class="seven-brand"><i class="seven-mark"></i><span class="seven-brand-name">7BZ Dev Cloud Game</span><span class="seven-live">LIVE</span></div>\n        <div class="seven-quick">\n          <div class="seven-chip"><b data-stat="fps">--</b><span>Stream FPS</span></div>\n          <div class="seven-chip"><b data-stat="rtt">--</b><span>WebSocket RTT</span></div>\n          <div class="seven-chip"><b data-stat="rate">--</b><span>Video rate</span></div>\n          <div class="seven-chip"><b data-stat="pad">None</b><span>Gamepad</span></div>\n          <div class="seven-chip"><b data-stat="gpu">--</b><span>GPU (G0|G1)</span></div>\n          <div class="seven-chip"><b data-stat="cpu">--</b><span>CPU load</span></div>\n        </div>\n        <button class="seven-button seven-icon-button" data-action="expand" aria-label="Open controls" title="Open controls">⌄</button>\n      </div>\n      <div class="seven-details is-collapsed">\n        <div class="seven-card">\n          <div class="seven-card-title"><span>Live performance</span><span data-stat="transport">WebSocket</span></div>\n          <div class="seven-metrics">\n            <div class="seven-metric"><strong data-stat="gpu0_metric">--</strong><small>🎮 GPU 0 Render</small></div>\n            <div class="seven-metric"><strong data-stat="gpu1_metric">--</strong><small>📡 GPU 1 Stream</small></div>\n            <div class="seven-metric"><strong data-stat="jitter">--</strong><small>RTT jitter</small></div>\n            <div class="seven-metric"><strong data-stat="loss">--</strong><small>Frame loss</small></div>\n            <div class="seven-metric"><strong data-stat="packet">N/A</strong><small>Packet loss</small></div>\n          </div>\n        </div>\n        <div class="seven-card">\n          <div class="seven-optimize">\n            <div class="seven-optimize-copy"><b>Adaptive Auto-Optimizer</b><span data-stat="profile">High · 720p · 8 Mbps</span></div>\n            <button class="seven-button" data-action="optimize">Auto-Optimize</button>\n          </div>\n          <div class="seven-actions">\n            <button class="seven-button" data-action="fullscreen">Fullscreen</button>\n            <button class="seven-button" data-action="pointer">Gaming Mode (Lock Mouse)</button>\n            <button class="seven-button" data-action="mode" data-stat="modename">🖱️ Mode: M+K</button>\n            <button class="seven-button" data-action="esc">Esc Lock</button>\n            <button class="seven-button" data-action="audio">Audio Unmute</button>\n            <button class="seven-button seven-danger" data-action="reset">Force Kill / Exit Frozen Game</button>\n          </div>\n        </div>\n        <div class="seven-notice"><span>Input tip: In FPS games (Blood West), use Gaming Mode for unconstrained 360° mouse view. If using Gamepad, press any button on the controller to connect.</span><span>Ctrl+Shift+M: HUD</span></div>\n      </div>\n    </div>\n    <div class="seven-toast" role="status"></div>`;\n  document.body.appendChild(hud);\n\n  const one = (selector) => hud.querySelector(selector);\n  const stat = (name, text) => { const target = one(`[data-stat="${name}"]`); if (target) target.textContent = text; };\n  const button = (name) => one(`[data-action="${name}"]`);\n  const toast = (message) => {\n    const target = one(".seven-toast");\n    target.textContent = message;\n    target.classList.add("is-visible");\n    clearTimeout(state.toastTimer);\n    state.toastTimer = setTimeout(() => target.classList.remove("is-visible"), 3200);\n  };\n\n  function controlBase() {\n    const replaced = window.location.pathname.replace(/\\/proxy\\/8080(?:\\/.*)?$/, "/proxy/8766");\n    return replaced === window.location.pathname ? "/proxy/8766" : replaced;\n  }\n\n  function checkGamepads() {\n    try {\n      const pads = navigator.getGamepads ? Array.from(navigator.getGamepads()).filter(Boolean) : [];\n      if (pads.length > 0) {\n        const p = pads[0];\n        const name = p.id ? p.id.split("(")[0].trim() : "Controller";\n        stat("pad", name.slice(0, 11) || "Connected");\n        if (!state.hasGamepad) {\n          state.hasGamepad = true;\n          toast(`🎮 Controller Connected: ${name.slice(0, 25)}`);\n          post({ type: "gamepadControl", enabled: true });\n        }\n      } else {\n        stat("pad", "None");\n        state.hasGamepad = false;\n      }\n    } catch (_) {}\n  }\n\n  function current() {\n    const latest = window.stream_stats?.latest || {};\n    const client = window.stream_client || {};\n    const rtt = value(latest.rtt_ms);\n    if (rtt > 0) {\n      state.rtts.push(rtt);\n      if (state.rtts.length > 14) state.rtts.shift();\n    }\n    const mean = state.rtts.reduce((sum, item) => sum + item, 0) / Math.max(1, state.rtts.length);\n    const jitter = Math.sqrt(state.rtts.reduce((sum, item) => sum + ((item - mean) ** 2), 0) / Math.max(1, state.rtts.length));\n    const lost = value(latest.lost_frames ?? latest.frames_not_shown);\n    const lostDelta = Math.max(0, lost - state.lastLost);\n    state.lastLost = lost;\n    return {\n      latest, client, rtt, jitter, lost, lostDelta,\n      fps: value(latest.fps, value(window.fps)),\n      rate: value(latest.video_mbps, value(latest.mbps)),\n    };\n  }\n\n  let lastGpuCheck = 0;\n  async function syncGpus() {\n    const now = Date.now();\n    if (now - lastGpuCheck < 2000) return;\n    lastGpuCheck = now;\n    try {\n      const response = await fetch(`${controlBase()}/gpus`, { credentials: "same-origin" });\n      if (response.ok) {\n        const data = await response.json();\n        const gpus = data.gpus || [];\n        if (gpus.length >= 2) {\n          const g0 = gpus[0], g1 = gpus[1];\n          stat("gpu", `${g0.utilization}% | ${g1.utilization}%`);\n          stat("gpu0_metric", `${g0.utilization}% · ${(g0.memory_used / 1024).toFixed(1)} GB`);\n          stat("gpu1_metric", `${g1.utilization}% · ${(g1.memory_used / 1024).toFixed(1)} GB`);\n        } else if (gpus.length === 1) {\n          const g0 = gpus[0];\n          stat("gpu", `${g0.utilization}%`);\n          stat("gpu0_metric", `${g0.utilization}% · ${(g0.memory_used / 1024).toFixed(1)} GB`);\n          stat("gpu1_metric", "Inactive");\n        }\n      }\n    } catch (_) {}\n  }\n\n  function render() {\n    const sample = current();\n    const latest = sample.latest;\n    stat("fps", sample.fps ? `${format(sample.fps)} FPS` : "--");\n    stat("rtt", sample.rtt ? `${format(sample.rtt)} ms` : "--");\n    stat("rate", sample.rate ? `${format(sample.rate, 1)} Mbps` : "--");\n    if (!lastGpuCheck) {\n      stat("gpu", latest.gpu_percent !== undefined ? `${format(latest.gpu_percent)}%` : "--");\n    }\n    stat("cpu", latest.cpu_percent !== undefined ? `${format(latest.cpu_percent)}%` : "--");\n    stat("jitter", state.rtts.length > 2 ? `${format(sample.jitter, 1)} ms` : "--");\n    stat("loss", Number.isFinite(Number(sample.lost)) ? String(sample.lost) : "--");\n    const used = value(latest.gpu_mem_used), total = value(latest.gpu_mem_total);\n    const gib = 1024 ** 3;\n    if (!lastGpuCheck && total) {\n      stat("gpu0_metric", `${format(used / gib, 1)} / ${format(total / gib, 1)} GB`);\n    }\n    const transport = sample.client.transport || "websockets";\n    stat("transport", transport === "webrtc" ? "WebRTC" : "WebSocket");\n    stat("packet", transport === "webrtc" && latest.packet_loss_percent !== undefined\n      ? `${format(latest.packet_loss_percent, 2)}%` : "N/A (WS)");\n    checkGamepads();\n    syncGpus();\n    if (state.auto) maybeTune(sample);\n  }\n\n  function applyLevel(index, reason) {\n    state.level = Math.max(0, Math.min(levels.length - 1, index));\n    const profile = levels[state.level];\n    post({ type: "settings", settings: { rate_control_mode: "cbr", video_bitrate: profile.bitrate, framerate: profile.fps } });\n    post({ type: "setManualResolution", width: profile.width, height: profile.height });\n    stat("profile", `${profile.name} · ${profile.width}×${profile.height} · ${profile.bitrate / 1000} Mbps`);\n    toast(`${reason}: ${profile.name} at ${profile.bitrate / 1000} Mbps`);\n  }\n\n  function maybeTune(sample) {\n    const now = Date.now();\n    if (now - state.lastTune < 12000) return;\n    state.lastTune = now;\n    const bad = sample.rtt > 145 || sample.jitter > 35 || sample.lostDelta > 0 || (sample.fps > 0 && sample.fps < 42);\n    const excellent = sample.rtt > 0 && sample.rtt < 55 && sample.jitter < 12 && sample.lostDelta === 0 && sample.fps >= 55;\n    if (bad && state.level > 0) applyLevel(state.level - 1, "Connection stabilized");\n    else if (excellent && state.level < levels.length - 1) applyLevel(state.level + 1, "Connection headroom detected");\n  }\n\n  function toggleExpanded() {\n    state.expanded = !state.expanded;\n    one(".seven-details").classList.toggle("is-collapsed", !state.expanded);\n    button("expand").textContent = state.expanded ? "⌃" : "⌄";\n    post({ type: "statsOpen", open: true });\n  }\n\n  async function togglePointer() {\n    if (document.pointerLockElement) {\n      document.exitPointerLock();\n      return;\n    }\n    // Activate Selkies Gaming Mode for raw relative hardware deltas\n    post({ type: "requestGamingMode" });\n    if (window.webrtcInput && typeof window.webrtcInput.enterGamingMode === "function") {\n      try { window.webrtcInput.enterGamingMode(); } catch (_) {}\n    }\n    const target = document.getElementById("videoCanvas") || document.getElementById("stream") || document.querySelector("canvas") || document.documentElement;\n    try {\n      await target.requestPointerLock({ unadjustedMovement: true });\n      toast("Gaming Mode: 360° mouse look enabled. Press Esc to exit.");\n    } catch (_error) {\n      try {\n        await target.requestPointerLock();\n        toast("Gaming Mode enabled. Press Esc to exit.");\n      } catch (error) {\n        toast(`Pointer lock unavailable: ${error.message}`);\n      }\n    }\n  }\n\n  async function toggleInputMode() {\n    const btn = button("mode");\n    if (!btn) return;\n    btn.disabled = true;\n    try {\n      const response = await fetch(`${controlBase()}/input-mode`, {\n        method: "POST",\n        credentials: "same-origin",\n        headers: { "X-7BZ-Action": "hud", "Content-Type": "application/json" },\n        body: JSON.stringify({})\n      });\n      if (!response.ok) throw new Error(`HTTP ${response.status}`);\n      const result = await response.json();\n      state.inputMode = result.mode;\n      updateInputModeUI();\n      toast(result.mode === "mouse_keyboard"\n        ? "Switched to 🖱️ Mouse & Keyboard Mode (Zero gamepad interference)"\n        : "Switched to 🎮 Gamepad Mode (Xbox 360 Controller)");\n    } catch (error) {\n      toast(`Input mode switch failed: ${error.message}`);\n    } finally {\n      btn.disabled = false;\n    }\n  }\n\n  function updateInputModeUI() {\n    const isKbm = state.inputMode === "mouse_keyboard";\n    const label = isKbm ? "🖱️ Mode: M+K" : "🎮 Mode: Gamepad";\n    stat("modename", label);\n    const btn = button("mode");\n    if (btn) {\n      btn.textContent = label;\n      btn.classList.toggle("is-active", !isKbm);\n    }\n  }\n\n  async function syncInputMode() {\n    try {\n      const response = await fetch(`${controlBase()}/input-mode`, { credentials: "same-origin" });\n      if (response.ok) {\n        const data = await response.json();\n        state.inputMode = data.mode || "mouse_keyboard";\n        updateInputModeUI();\n      }\n    } catch (_) {}\n  }\n\n  async function toggleEsc() {\n    if (!navigator.keyboard?.lock) { toast("Esc Lock requires a Chromium browser in fullscreen."); return; }\n    try {\n      if (state.esc) { navigator.keyboard.unlock(); state.esc = false; }\n      else { await navigator.keyboard.lock(["Escape"]); state.esc = true; }\n      button("esc").classList.toggle("is-active", state.esc);\n      toast(state.esc ? "Escape is captured by the remote game." : "Escape lock released.");\n    } catch (_error) { toast("Enter fullscreen first, then enable Esc Lock."); }\n  }\n\n  async function resetGame() {\n    const control = button("reset");\n    control.disabled = true;\n    control.textContent = "Stopping…";\n    try {\n      const response = await fetch(`${controlBase()}/reset`, { method: "POST", credentials: "same-origin",\n        headers: { "X-7BZ-Action": "hud" } });\n      if (!response.ok) throw new Error(`HTTP ${response.status}`);\n      const result = await response.json();\n      toast(result.message || "Dashboard restored.");\n      if (document.pointerLockElement) document.exitPointerLock();\n    } catch (error) { toast(`Reset failed: ${error.message}`); }\n    finally { control.disabled = false; control.textContent = "Force Kill / Exit Frozen Game"; }\n  }\n\n  hud.addEventListener("click", (event) => {\n    const action = event.target.closest("[data-action]")?.dataset.action;\n    if (!action) return;\n    if (action === "expand") toggleExpanded();\n    if (action === "fullscreen") post({ type: "requestFullscreen" });\n    if (action === "pointer") togglePointer();\n    if (action === "mode") toggleInputMode();\n    if (action === "esc") toggleEsc();\n    if (action === "audio") {\n      state.audio = true;\n      post({ type: "pipelineControl", pipeline: "audio", enabled: true });\n      post({ type: "setMute", value: false });\n      post({ type: "setVolume", value: 1 });\n      button("audio").classList.add("is-active");\n      toast("Audio enabled at full volume.");\n    }\n    if (action === "optimize") {\n      state.auto = !state.auto;\n      button("optimize").classList.toggle("is-active", state.auto);\n      button("optimize").textContent = state.auto ? "Auto: ON" : "Auto-Optimize";\n      if (state.auto) { state.lastTune = 0; applyLevel(state.level, "Auto-Optimizer enabled"); }\n      else toast("Auto-Optimizer paused.");\n    }\n    if (action === "reset") resetGame();\n  });\n\n  document.addEventListener("pointerlockchange", () => {\n    state.pointer = Boolean(document.pointerLockElement);\n    button("pointer").classList.toggle("is-active", state.pointer);\n  });\n  document.addEventListener("fullscreenchange", () => button("fullscreen").classList.toggle("is-active", Boolean(document.fullscreenElement)));\n  window.addEventListener("keydown", (event) => {\n    if (event.ctrlKey && event.shiftKey && event.code === "KeyM") {\n      event.preventDefault();\n      toggleExpanded();\n    }\n  }, true);\n\n  window.addEventListener("gamepadconnected", (event) => {\n    state.hasGamepad = true;\n    toast(`🎮 Gamepad Connected: ${event.gamepad.id.slice(0, 30)}`);\n    post({ type: "gamepadControl", enabled: true });\n    checkGamepads();\n  });\n  window.addEventListener("gamepaddisconnected", () => {\n    state.hasGamepad = false;\n    toast("Gamepad disconnected");\n    checkGamepads();\n  });\n\n  post({ type: "statsOpen", open: true });\n  syncInputMode();\n  setInterval(render, 1000);\n  setTimeout(render, 700);\n})();\n', encoding='utf-8')

# Write tint2 panel configuration for desktop taskbar
tint2rc = """# Minimal dark tint2 panel configuration
panel_position = bottom center horizontal
panel_size = 100% 32
panel_margin = 0 0
panel_padding = 4 2 4
panel_background_id = 1
wm_menu = 1
panel_dock = 0
panel_layer = top

taskbar_mode = single_desktop
taskbar_padding = 2 0 2
taskbar_background_id = 0
taskbar_active_background_id = 2

task_icon = 1
task_text = 1
task_width = 190
task_maximum_size = 220 28
task_padding = 6 2
task_font = DejaVu Sans 10
task_font_color = #d8e4f5 100
task_active_font_color = #4dcdf2 100

time1_font = DejaVu Sans 10
clock_font_color = #9bdcf5 100
time1_format = %H:%M:%S

rounded = 0
border_width = 1
background_color = #090e1e 100
border_color = #172b4d 100

rounded = 4
border_width = 1
background_color = #152744 100
border_color = #4dcdf2 100
"""
(BENCH / 'tint2rc').write_text(tint2rc, encoding='utf-8')

# Steam launch wrapper - executes under unprivileged 'gamer' user
steam_sh = """#!/bin/bash
export DISPLAY="${DISPLAY:-:0}"
export PATH="/usr/games:/usr/local/bin:/usr/bin:$PATH"
export SDL_AUDIODRIVER=pulse
export LD_PRELOAD='/tmp/kaggle-game-dev-bench/$LIB/selkies_input_interposer.so:/tmp/kaggle-game-dev-bench/$LIB/selkies_fake_udev.so'
export SELKIES_JS_SOCKET_PATH="/tmp"
export SDL_JOYSTICK_DISABLE_UDEV=1
export SDL_JOYSTICK_DEVICE=/dev/input/event1000:/dev/input/event1001:/dev/input/event1002:/dev/input/event1003

mkdir -p /home/gamer/logs
LOG="/home/gamer/logs/steam.log"
STEAM_BIN=$(which steam 2>/dev/null || echo /usr/games/steam)

echo "Starting Steam as user gamer at $(date)..." >> "$LOG"
exec env \
    DISPLAY="$DISPLAY" \
    PATH="$PATH" \
    SDL_AUDIODRIVER=pulse \
    LD_PRELOAD="$LD_PRELOAD" \
    SELKIES_JS_SOCKET_PATH="$SELKIES_JS_SOCKET_PATH" \
    SDL_JOYSTICK_DISABLE_UDEV=1 \
    SDL_JOYSTICK_DEVICE="$SDL_JOYSTICK_DEVICE" \
    HOME="/home/gamer" \
    "$STEAM_BIN" -no-cef-sandbox -cef-disable-gpu -cef-force-software-rendering "$@" >> "$LOG" 2>&1
"""
(BENCH / 'launch_steam.sh').write_text(steam_sh, encoding='utf-8')

# Heroic launch wrapper (Epic Games & GOG) - executes under unprivileged 'gamer' user
heroic_sh = '#!/bin/bash\nexport DISPLAY="${DISPLAY:-:0}"\nexport PATH="/usr/games:/usr/local/bin:/usr/bin:$PATH"\nexport SDL_AUDIODRIVER=pulse\nexport LD_PRELOAD=\'/tmp/kaggle-game-dev-bench/$LIB/selkies_input_interposer.so:/tmp/kaggle-game-dev-bench/$LIB/selkies_fake_udev.so\'\nexport SELKIES_JS_SOCKET_PATH="/tmp"\nexport SDL_JOYSTICK_DISABLE_UDEV=1\nexport SDL_JOYSTICK_DEVICE=/dev/input/event1000:/dev/input/event1001:/dev/input/event1002:/dev/input/event1003\n\nmkdir -p /home/gamer/logs\nLOG="/home/gamer/logs/heroic.log"\nHEROIC_BIN=$(which heroic 2>/dev/null || echo /usr/bin/heroic)\n\n# Source auto-optimizer Wine settings if available\nif [ -f /home/gamer/.config/gamelab_wine_env.sh ]; then\n    source /home/gamer/.config/gamelab_wine_env.sh\nfi\n\n# Automatically sanitize Heroic configuration and clean DXVK DLLs before launch\npython3 /tmp/kaggle-game-dev-bench/fix_heroic_config.py >> "$LOG" 2>&1\nchown -R gamer:gamer /home/gamer 2>/dev/null || true\nchmod -R u+rwX /home/gamer 2>/dev/null || true\n\necho "Starting Heroic as user gamer at $(date)..." >> "$LOG"\nHEROIC_PREFIX=()\nVGL_TARGET=$(PYTHONPATH=/tmp/kaggle-game-dev-bench python3 -c \'from gpu_runtime import diagnose; d=diagnose(); print(d["virtualgl_display"] if d["virtualgl"] and d["nvidia_32bit_gl"] else "")\' 2>/dev/null)\nif [ -n "$VGL_TARGET" ]; then\n    HEROIC_PREFIX=(/opt/VirtualGL/bin/vglrun -d "$VGL_TARGET")\nfi\nif [ "$(id -u)" -eq 0 ]; then\n    USER_PREFIX=(runuser -u gamer --)\nelse\n    USER_PREFIX=()\nfi\nexec "${USER_PREFIX[@]}" env \\\n    DISPLAY="$DISPLAY" \\\n    PATH="$PATH" \\\n    SDL_AUDIODRIVER=pulse \\\n    LD_PRELOAD="$LD_PRELOAD" \\\n    SELKIES_JS_SOCKET_PATH="$SELKIES_JS_SOCKET_PATH" \\\n    SDL_JOYSTICK_DISABLE_UDEV=1 \\\n    SDL_JOYSTICK_DEVICE="$SDL_JOYSTICK_DEVICE" \\\n    HOME="/home/gamer" \\\n    "${HEROIC_PREFIX[@]}" "$HEROIC_BIN" --no-sandbox --disable-gpu --disable-gpu-compositing "$@" >> "$LOG" 2>&1\n'
(BENCH / 'launch_heroic.sh').write_text(heroic_sh, encoding='utf-8')

# Minigalaxy launch wrapper (GOG dedicated) - executes under unprivileged 'gamer' user
mini_sh = """#!/bin/bash
export DISPLAY="${DISPLAY:-:0}"
export PATH="/usr/games:/usr/local/bin:/usr/bin:$PATH"
export SDL_AUDIODRIVER=pulse
mkdir -p /home/gamer/logs
LOG="/home/gamer/logs/minigalaxy.log"
MINI_BIN=$(which minigalaxy 2>/dev/null || echo /usr/games/minigalaxy)

echo "Starting Minigalaxy as user gamer at $(date)..." >> "$LOG"
exec env \
    DISPLAY="$DISPLAY" \
    PATH="$PATH" \
    SDL_AUDIODRIVER=pulse \
    HOME="/home/gamer" \
    "$MINI_BIN" "$@" >> "$LOG" 2>&1
"""
(BENCH / 'launch_minigalaxy.sh').write_text(mini_sh, encoding='utf-8')

# Lutris launch wrapper - executes under unprivileged 'gamer' user
lutris_sh = """#!/bin/bash
export DISPLAY="${DISPLAY:-:0}"
export PATH="/usr/games:/usr/local/bin:/usr/bin:$PATH"
export SDL_AUDIODRIVER=pulse
mkdir -p /home/gamer/logs
LOG="/home/gamer/logs/lutris.log"
LUTRIS_BIN=$(which lutris 2>/dev/null || echo /usr/games/lutris)

# Match Ubuntu noble C extension ABI (_dbus_bindings) using python3.12 if available
if [ -x /usr/bin/python3.12 ]; then
    PY_BIN="/usr/bin/python3.12"
else
    PY_BIN="python3"
fi

echo "Starting Lutris with $PY_BIN as user gamer at $(date)..." >> "$LOG"
exec env \
    DISPLAY="$DISPLAY" \
    PATH="$PATH" \
    SDL_AUDIODRIVER=pulse \
    HOME="/home/gamer" \
    "$PY_BIN" "$LUTRIS_BIN" "$@" >> "$LOG" 2>&1
"""
(BENCH / 'launch_lutris.sh').write_text(lutris_sh, encoding='utf-8')

# EA App launch wrapper - executes under unprivileged 'gamer' user in isolated wineprefix
ea_sh = """#!/bin/bash
export DISPLAY="${DISPLAY:-:0}"
export PATH="/usr/games:/usr/local/bin:/usr/bin:$PATH"
export WINEPREFIX="/home/gamer/ea-prefix"
export WINEARCH=win64
export WINEDLLOVERRIDES="mscoree="
mkdir -p "$WINEPREFIX" /tmp/ea-store /home/gamer/logs
chown -R gamer:gamer "$WINEPREFIX" /home/gamer/logs 2>/dev/null || true

LOG="/home/gamer/logs/ea.log"
echo "Starting EA App runner at $(date)..." >> "$LOG"

EA_INSTALLER="/tmp/ea-store/EAappInstaller.exe"
if [ ! -f "$EA_INSTALLER" ]; then
    echo "Downloading EAappInstaller.exe..." >> "$LOG"
    wget -q https://origin-a.akamaihd.net/EA-Desktop-Client-Download/installer-releases/EAappInstaller.exe -O "$EA_INSTALLER"
fi

EA_INSTALLED="$WINEPREFIX/drive_c/Program Files/Electronic Arts/EA Desktop/EA Desktop/EADesktop.exe"
if [ -f "$EA_INSTALLED" ]; then
    echo "Running installed EA Desktop: $EA_INSTALLED" >> "$LOG"
    TARGET="$EA_INSTALLED"
else
    echo "Running EA App Installer via Wine64: $EA_INSTALLER" >> "$LOG"
    TARGET="$EA_INSTALLER"
fi

exec env \
    DISPLAY="$DISPLAY" \
    PATH="$PATH" \
    WINEPREFIX="$WINEPREFIX" \
    WINEARCH=win64 \
    WINEDLLOVERRIDES="$WINEDLLOVERRIDES" \
    HOME="/home/gamer" \
    wine "$TARGET" "$@" >> "$LOG" 2>&1
"""
(BENCH / 'launch_ea.sh').write_text(ea_sh, encoding='utf-8')

# Fix Heroic Config Python script
(BENCH / 'fix_heroic_config.py').write_text('"""Apply Heroic settings according to the live, measured graphics backend."""\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nfrom datetime import datetime, timezone\nfrom gpu_runtime import diagnose\n\nHOME = Path(os.environ.get(\'KAGGLE_GAMER_HOME\', \'/home/gamer\'))\nCONFIG = HOME / \'.config/heroic\'\nGAME_CONFIG = CONFIG / \'GamesConfig\'\nRUNNERS = CONFIG / \'tools/wine\'\nBACKUPS = HOME / \'logs/heroic-config-backups\'\nBACKUPS.mkdir(parents=True, exist_ok=True)\nGPU_FACTS = diagnose()\nGPU_DXVK = (GPU_FACTS[\'vulkan_window\'] and GPU_FACTS[\'nvidia_32bit_gl\']\n            and GPU_FACTS[\'vulkan_32bit_loader\'] and GPU_FACTS[\'dxvk_files\'])\n\n# Select best available Wine runner (Prefer Wine-GE over system wine)\nrunner = {\'bin\': \'/usr/bin/wine\', \'wineserver\': \'/usr/bin/wineserver\',\n          \'wineboot\': \'/usr/bin/wineboot\', \'name\': \'Wine Default\', \'type\': \'wine\'}\nfor candidate in [RUNNERS / \'Wine-GE-Proton8-26/bin/wine\', *RUNNERS.glob(\'*/bin/wine\')]:\n    if (candidate.is_file() and os.access(candidate, os.X_OK) and\n            candidate.with_name(\'wineserver\').is_file()):\n        runner = {\'bin\': str(candidate), \'wineserver\': str(candidate.with_name(\'wineserver\')),\n                  \'wineboot\': str(candidate.with_name(\'wineboot\')),\n                  \'name\': candidate.parent.parent.name, \'type\': \'wine\'}\n        break\n\nGAME_SETTINGS_KEYS = (\n    \'autoInstallDxvk\', \'wineVersion\', \'winePrefix\', \'autoInstallVkd3d\',\n    \'preferSystemLibs\', \'launcherArgs\', \'enableEsync\', \'enableFsync\'\n)\n\ndef clean_prefix_dxvk(prefix: Path):\n    """Safely disarm DXVK DLL overrides so WineD3D (OpenGL) renders without Vulkan hardware crash."""\n    for sub in (\'drive_c/windows/system32\', \'drive_c/windows/syswow64\'):\n        sys_dir = prefix / sub\n        if not sys_dir.is_dir():\n            continue\n        for dll_name in (\'dxgi.dll\', \'d3d11.dll\', \'d3d10core.dll\', \'d3d9.dll\'):\n            target = sys_dir / dll_name\n            if target.is_file():\n                try:\n                    bak = target.with_name(target.name + \'.dxvk_bak\')\n                    if not bak.exists():\n                        target.rename(bak)\n                    else:\n                        target.unlink()\n                    print(f\'Disarmed DXVK dll in prefix: {target}\')\n                except OSError as err:\n                    print(f\'Notice disarming {target}: {err}\')\n\ndef repair(path, global_config=False):\n    if not path.exists():\n        return \'missing\'\n    try:\n        old_text = path.read_text(encoding=\'utf-8\')\n        data = json.loads(old_text)\n        if not isinstance(data, dict):\n            return \'unexpected JSON shape\'\n    except (OSError, json.JSONDecodeError) as exc:\n        return f\'not edited: {exc}\'\n\n    settings_objects = []\n    if global_config:\n        # Migrate flat config if found\n        if not isinstance(data.get(\'defaultSettings\'), dict):\n            keys = (\'autoInstallDxvk\', \'autoInstallVkd3d\', \'autoInstallDxvkNvapi\',\n                    \'enableEsync\', \'enableFsync\', \'disableUMU\', \'wineVersion\')\n            data[\'defaultSettings\'] = {key: data.pop(key) for key in keys if key in data}\n        if isinstance(data.get(\'defaultSettings\'), dict):\n            settings_objects.append(data[\'defaultSettings\'])\n        else:\n            settings_objects.append(data)\n    else:\n        # Per-game config: handle stem key, arbitrary single key, multiple keys, or flat\n        if path.stem in data and isinstance(data[path.stem], dict):\n            settings_objects.append(data[path.stem])\n        else:\n            child_matches = [\n                v for v in data.values()\n                if isinstance(v, dict) and any(k in v for k in GAME_SETTINGS_KEYS)\n            ]\n            if child_matches:\n                settings_objects.extend(child_matches)\n            elif any(k in data for k in GAME_SETTINGS_KEYS):\n                settings_objects.append(data)\n            elif len(data) == 1 and isinstance(list(data.values())[0], dict):\n                settings_objects.append(list(data.values())[0])\n\n    if not settings_objects:\n        return \'unexpected settings shape\'\n\n    for settings in settings_objects:\n        for key in (\'autoInstallVkd3d\', \'autoInstallDxvkNvapi\',\n                    \'enableEsync\', \'enableFsync\'):\n            settings[key] = False\n        settings[\'autoInstallDxvk\'] = bool(GPU_DXVK)\n        settings[\'disableUMU\'] = True\n\n        # Assign Wine runner\n        selected = settings.get(\'wineVersion\')\n        if (not isinstance(selected, dict) or selected.get(\'type\') != \'wine\' or\n                not os.path.isfile(selected.get(\'bin\', \'\')) or\n                not os.path.isfile(selected.get(\'wineserver\', \'\'))):\n            settings[\'wineVersion\'] = runner\n\n        # Ensure WINEDLLOVERRIDES forces WineD3D builtins\n        env_opts = settings.get(\'enviromentOptions\')\n        if not isinstance(env_opts, list):\n            env_opts = []\n        found_override = False\n        for opt in env_opts:\n            if isinstance(opt, dict) and opt.get(\'key\') == \'WINEDLLOVERRIDES\':\n                opt[\'value\'] = (\'d3d11,dxgi,d3d9,d3d10core=n,b\' if GPU_DXVK\n                                else \'d3d11,dxgi,d3d9,d3d10core=b\')\n                found_override = True\n                break\n        if not found_override:\n            env_opts.append({\'key\': \'WINEDLLOVERRIDES\', \'value\':\n                             \'d3d11,dxgi,d3d9,d3d10core=n,b\' if GPU_DXVK\n                             else \'d3d11,dxgi,d3d9,d3d10core=b\'})\n        settings[\'enviromentOptions\'] = env_opts\n\n        # Clean prefix DXVK overrides if prefix exists\n        pfx = settings.get(\'winePrefix\')\n        if pfx and os.path.isdir(pfx) and not GPU_DXVK:\n            clean_prefix_dxvk(Path(pfx))\n\n    new_text = json.dumps(data, indent=2) + \'\\n\'\n    if new_text == old_text:\n        return \'already configured\'\n    stamp = datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%SZ\')\n    backup = BACKUPS / f\'{path.name}.{stamp}.bak\'\n    shutil.copy2(path, backup)\n    tmp = path.with_suffix(path.suffix + \'.new\')\n    tmp.write_text(new_text, encoding=\'utf-8\')\n    os.replace(tmp, path)\n    return f\'updated; backup {backup}\'\n\ndef sync_legendary():\n    """Seamlessly share Epic Games auth tokens & installed library between Heroic and Legendary CLI."""\n    heroic_leg = CONFIG / \'legendaryConfig/legendary\'\n    std_leg = HOME / \'.config/legendary\'\n    heroic_leg.mkdir(parents=True, exist_ok=True)\n    std_leg.mkdir(parents=True, exist_ok=True)\n\n    synced = []\n    for fname in (\'user.json\', \'installed.json\', \'config.ini\'):\n        h_f = heroic_leg / fname\n        s_f = std_leg / fname\n        try:\n            if h_f.is_file() and not s_f.is_file():\n                shutil.copy2(h_f, s_f)\n                synced.append(f\'{fname} (Heroic -> Legendary)\')\n            elif s_f.is_file() and not h_f.is_file():\n                shutil.copy2(s_f, h_f)\n                synced.append(f\'{fname} (Legendary -> Heroic)\')\n            elif h_f.is_file() and s_f.is_file():\n                if h_f.stat().st_mtime > s_f.stat().st_mtime:\n                    shutil.copy2(h_f, s_f)\n                    synced.append(f\'{fname} (Heroic -> Legendary, updated)\')\n                elif s_f.stat().st_mtime > h_f.stat().st_mtime:\n                    shutil.copy2(s_f, h_f)\n                    synced.append(f\'{fname} (Legendary -> Heroic, updated)\')\n        except Exception as e:\n            print(f\'Notice syncing {fname}: {e}\')\n    if synced:\n        print(\'Legendary/Heroic synced:\', \', \'.join(synced))\n    else:\n        print(\'Legendary/Heroic auth status: synchronized\')\n\nprint(\'Heroic runner:\', runner[\'bin\'])\nglobal_file = CONFIG / \'config.json\'\nprint(global_file, repair(global_file, global_config=True))\nfor path in sorted(GAME_CONFIG.glob(\'*.json\')):\n    print(path, repair(path))\nfor log in sorted(GAME_CONFIG.glob(\'*-lastPlay.log\'), key=lambda p: p.stat().st_mtime, reverse=True)[:5]:\n    print(f\'Latest play log: {log}\')\n    print(\'\\n\'.join(log.read_text(errors=\'replace\').splitlines()[-25:]))\n\nsync_legendary()\n', encoding='utf-8')

# Fix Heroic Shell wrapper for launcher
fix_heroic_sh = '#!/bin/bash\nexport DISPLAY="${DISPLAY:-:0}"\nexport PATH="/usr/games:/usr/local/bin:/usr/bin:$PATH"\nmkdir -p /home/gamer/logs\nLOG="/home/gamer/logs/heroic_fix.log"\nexec xterm -geometry 95x25 -title "Heroic Config Repair" -e bash -c "echo \'=== Checking GPU and repairing Heroic settings ===\' | tee -a \'$LOG\'; python3 /tmp/kaggle-game-dev-bench/fix_heroic_config.py 2>&1 | tee -a \'$LOG\'; chown -R gamer:gamer /home/gamer 2>/dev/null || true; chmod -R u+rwX /home/gamer 2>/dev/null || true; echo \'\'; echo \'Settings updated for the detected graphics backend.\'; echo \'\'; read -p \'Press Enter to close window...\'"\n'
(BENCH / 'fix_heroic.sh').write_text(fix_heroic_sh, encoding='utf-8')

# Legendary interactive menu and graphical interface
(BENCH / 'legendary_menu.sh').write_text('#!/bin/bash\nexport DISPLAY="${DISPLAY:-:0}"\nexport PATH="/usr/games:/usr/local/bin:/usr/bin:$PATH"\nexport SDL_AUDIODRIVER=pulse\nexport HOME="/home/gamer"\nexport LEGENDARY_CONFIG_PATH="$HOME/.config/heroic/legendaryConfig/legendary"\nexport BROWSER="/bin/true"\nmkdir -p "$LEGENDARY_CONFIG_PATH" "$HOME/.config/legendary" "$HOME/logs"\nset -o pipefail\n\n# Source auto-optimizer Wine settings if available\nif [ -f "$HOME/.config/gamelab_wine_env.sh" ]; then\n    source "$HOME/.config/gamelab_wine_env.sh"\nfi\n\n# Ensure credentials and configs are synced on startup\npython3 /tmp/kaggle-game-dev-bench/fix_heroic_config.py >/dev/null 2>&1 || true\n\nwhile true; do\n    clear\n    echo "=================================================================="\n    echo "           LEGENDARY CLI - EPIC GAMES NATIVE MANAGER              "\n    echo "=================================================================="\n    echo " Epic Library Location : $LEGENDARY_CONFIG_PATH"\n    echo " Rendering Backend     : WineD3D (OpenGL software rendering / llvmpipe)"\n    echo " Shared Session        : Synced automatically with Heroic Games"\n    echo "------------------------------------------------------------------"\n    echo " [1] List installed games and launchable apps"\n    echo " [2] Quick Launch an installed game (Brotato, Wolfenstein, etc)"\n    echo " [3] Log in to Epic Games (or check login status)"\n    echo " [4] List all games in your Epic Games library"\n    echo " [5] Install a game from your Epic library"\n    echo " [6] Fix Heroic settings (Disable DXVK / Enable WineD3D)"\n    echo " [7] Open an interactive Bash shell as gamer user"\n    echo " [8] Print a game\'s launch command without starting it"\n    echo " [0] Exit"\n    echo "=================================================================="\n    read -p "Select option [0-8]: " choice\n\n    case "$choice" in\n        1)\n            echo ""\n            echo "Syncing Epic library from Heroic..."\n            python3 /tmp/kaggle-game-dev-bench/fix_heroic_config.py 2>/dev/null || true\n            echo ""\n            legendary list-installed\n            echo ""\n            read -p "Press Enter to return to menu..."\n            ;;\n        2)\n            echo ""\n            echo "Installed games:"\n            legendary list-installed\n            echo ""\n            read -p "Enter Game App Name (from the list above): " appname\n            if [ -n "$appname" ]; then\n                WINE_BIN=""\n                for candidate in "/home/gamer/.config/heroic/tools/wine/Wine-GE-Proton8-26/bin/wine" \\\n                                 /home/gamer/.config/heroic/tools/wine/*/bin/wine \\\n                                 /usr/bin/wine /usr/local/bin/wine /usr/games/wine; do\n                    if [ -f "$candidate" ] && [ -x "$candidate" ]; then\n                        WINE_BIN="$candidate"\n                        break\n                    fi\n                done\n                if [ -z "$WINE_BIN" ]; then\n                    echo "Error: No Wine runner found on system!"\n                else\n                    echo "Checking GPU 3D renderer before launching $appname using runner: $WINE_BIN"\n                    export WINEPREFIX="/home/gamer/.wine"\n                    \n                    # Auto-heal prefix if kernel32.dll is missing\n                    if [ ! -f "$WINEPREFIX/drive_c/windows/system32/kernel32.dll" ]; then\n                        echo "Initializing Wine prefix with Wine-GE..."\n                        WINEBOOT_BIN="$(dirname "$WINE_BIN")/wineboot"\n                        if [ -x "$WINEBOOT_BIN" ]; then\n                            WINEDLLOVERRIDES="mscoree,mshtml=d" "$WINEBOOT_BIN" --init 2>/dev/null || true\n                        fi\n                    fi\n                    \n                    python3 /tmp/kaggle-game-dev-bench/gpu_runtime.py launch "$appname" "$WINE_BIN" 2>&1 | tee -a /home/gamer/logs/legendary-game.log\n                    echo "Legendary exit status: ${PIPESTATUS[0]}" | tee -a /home/gamer/logs/legendary-game.log\n                fi\n            fi\n            read -p "Press Enter to return to menu..."\n            ;;\n        3)\n            echo ""\n            echo "Checking Epic Games login status..."\n            python3 /tmp/kaggle-game-dev-bench/fix_heroic_config.py 2>/dev/null || true\n            if [ -f "$HOME/.config/heroic/legendaryConfig/legendary/user.json" ] || [ -f "$HOME/.config/legendary/user.json" ]; then\n                echo "✓ Found existing Epic Games session!"\n                legendary status 2>/dev/null\n                echo ""\n                read -p "Already logged in! Do you want to re-authenticate with a new code? (y/N): " reauth\n                if [[ ! "$reauth" =~ ^[Yy]$ ]]; then\n                    read -p "Press Enter to return to menu..."\n                    continue\n                fi\n            fi\n\n            echo ""\n            echo "=================================================================="\n            echo "                   EPIC GAMES LOGIN INSTRUCTIONS                  "\n            echo "=================================================================="\n            echo " 1. Open this URL in your PC browser (Chrome/Edge/Firefox):"\n            echo "    https://legendary.gl/epiclogin"\n            echo ""\n            echo " 2. Sign in with your Epic Games account."\n            echo ""\n            echo " 3. The page will display a JSON code containing \'authorizationCode\'."\n            echo "    Example: {\\"authorizationCode\\": \\"3a7b9c1d2e...\\"}"\n            echo ""\n            echo " 4. Copy the authorizationCode value (or the whole JSON text)"\n            echo "    and paste it into the prompt below."\n            echo "=================================================================="\n            echo ""\n            legendary auth\n            python3 /tmp/kaggle-game-dev-bench/fix_heroic_config.py 2>/dev/null || true\n            echo ""\n            read -p "Press Enter to return to menu..."\n            ;;\n        4)\n            echo ""\n            legendary list\n            echo ""\n            read -p "Press Enter to return to menu..."\n            ;;\n        5)\n            echo ""\n            read -p "Enter Game App Name to install: " install_app\n            if [ -n "$install_app" ]; then\n                legendary install "$install_app"\n                python3 /tmp/kaggle-game-dev-bench/fix_heroic_config.py 2>/dev/null || true\n            fi\n            read -p "Press Enter to return to menu..."\n            ;;\n        6)\n            echo ""\n            python3 /tmp/kaggle-game-dev-bench/fix_heroic_config.py\n            echo ""\n            read -p "Press Enter to return to menu..."\n            ;;\n        7)\n            echo "Dropping into shell as user gamer. Type \'exit\' to return."\n            bash\n            ;;\n        8)\n            legendary list-installed\n            read -p "Enter Game App Name: " appname\n            if [ -n "$appname" ]; then\n                WINE_BIN=""\n                for candidate in "/home/gamer/.config/heroic/tools/wine/Wine-GE-Proton8-26/bin/wine" \\\n                                 /home/gamer/.config/heroic/tools/wine/*/bin/wine \\\n                                 /usr/bin/wine /usr/local/bin/wine /usr/games/wine; do\n                    if [ -f "$candidate" ] && [ -x "$candidate" ]; then\n                        WINE_BIN="$candidate"\n                        break\n                    fi\n                done\n                legendary launch "$appname" --wine "${WINE_BIN:-wine}" --dry-run 2>&1 | tee -a /home/gamer/logs/legendary-dry-run.log\n                echo "Legendary dry-run exit status: ${PIPESTATUS[0]}" | tee -a /home/gamer/logs/legendary-dry-run.log\n            fi\n            read -p "Press Enter to return to menu..."\n            ;;\n        0)\n            break\n            ;;\n        *)\n            echo "Invalid option."\n            sleep 1\n            ;;\n    esac\ndone\n', encoding='utf-8')
(BENCH / 'legendary_gui.py').write_text('"""Modern Graphical Interface for Legendary Epic Games Manager."""\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport threading\nimport time\nimport tkinter as tk\nfrom tkinter import ttk, messagebox, scrolledtext\n\nHOME = Path(os.environ.get(\'KAGGLE_GAMER_HOME\', \'/home/gamer\'))\nCONFIG = HOME / \'.config/heroic\'\nHEROIC_LEG = CONFIG / \'legendaryConfig/legendary\'\nSTD_LEG = HOME / \'.config/legendary\'\nBENCH = Path(\'/tmp/kaggle-game-dev-bench\')\nLOGS = HOME / \'logs\'\nLOGS.mkdir(parents=True, exist_ok=True)\n\n# Ensure config paths exist\nHEROIC_LEG.mkdir(parents=True, exist_ok=True)\nSTD_LEG.mkdir(parents=True, exist_ok=True)\n\ndef find_best_runner():\n    candidates = [\n        HOME / \'.config/heroic/tools/wine/Wine-GE-Proton8-26/bin/wine\',\n        *HOME.glob(\'.config/heroic/tools/wine/*/bin/wine\'),\n        Path(\'/usr/bin/wine\'),\n        Path(\'/usr/local/bin/wine\'),\n        Path(\'/usr/games/wine\'),\n    ]\n    for c in candidates:\n        if c.is_file() and os.access(c, os.X_OK):\n            return str(c)\n    return \'/usr/bin/wine\'\n\ndef format_size(bytes_val):\n    try:\n        b = float(bytes_val)\n        if b >= 1024 * 1024 * 1024:\n            return f"{b / (1024**3):.2f} GiB"\n        elif b >= 1024 * 1024:\n            return f"{b / (1024**2):.1f} MiB"\n        elif b > 0:\n            return f"{b / 1024:.0f} KiB"\n    except (ValueError, TypeError):\n        pass\n    return "Unknown"\n\nclass LegendaryApp(tk.Tk):\n    def __init__(self):\n        super().__init__()\n        self.title("Epic Games · Legendary Manager")\n        self.geometry("980x660")\n        self.minsize(800, 500)\n\n        # Active process tracker\n        self.active_proc = None\n        self.proc_thread = None\n\n        # Apply dark theme styling\n        self.setup_styles()\n\n        # Build UI layout\n        self.create_header()\n        self.create_tabs()\n        self.create_bottom_bar()\n\n        # Initial data load\n        self.refresh_all()\n\n    def setup_styles(self):\n        self.configure(bg="#141923")\n        self.style = ttk.Style(self)\n        try:\n            self.style.theme_use("clam")\n        except tk.TclError:\n            pass\n\n        # Colors\n        self.c_bg = "#141923"\n        self.c_panel = "#1b2230"\n        self.c_card = "#222c3e"\n        self.c_fg = "#f0f4fc"\n        self.c_muted = "#8fa0bd"\n        self.c_accent = "#3498db"\n        self.c_green = "#2ecc71"\n        self.c_red = "#e74c3c"\n        self.c_yellow = "#f39c12"\n\n        self.style.configure(".", background=self.c_bg, foreground=self.c_fg, font=("DejaVu Sans", 10))\n        self.style.configure("TNotebook", background=self.c_bg, borderwidth=0)\n        self.style.configure("TNotebook.Tab", background=self.c_panel, foreground=self.c_muted,\n                             padding=[18, 9], font=("DejaVu Sans", 10, "bold"))\n        self.style.map("TNotebook.Tab",\n                       background=[("selected", self.c_card), ("active", "#2a364d")],\n                       foreground=[("selected", "#ffffff"), ("active", self.c_fg)])\n\n        self.style.configure("Treeview", background="#1a202c", foreground=self.c_fg,\n                             fieldbackground="#1a202c", rowheight=32, font=("DejaVu Sans", 10),\n                             borderwidth=0)\n        self.style.configure("Treeview.Heading", background=self.c_panel, foreground="#ffffff",\n                             font=("DejaVu Sans", 10, "bold"), padding=[6, 6])\n        self.style.map("Treeview", background=[("selected", "#2b4a70")], foreground=[("selected", "#ffffff")])\n\n        self.style.configure("TFrame", background=self.c_bg)\n        self.style.configure("Card.TFrame", background=self.c_panel, relief="flat")\n\n    def create_header(self):\n        header = tk.Frame(self, bg=self.c_panel, height=65)\n        header.pack(fill="x", side="top", padx=12, pady=(10, 6))\n\n        # Brand badge\n        logo_frame = tk.Frame(header, bg=self.c_panel)\n        logo_frame.pack(side="left", padx=14, pady=10)\n\n        lbl_badge = tk.Label(logo_frame, text=" LEGENDARY ", bg=self.c_accent, fg="#ffffff",\n                             font=("DejaVu Sans", 10, "bold"), padx=6, pady=2)\n        lbl_badge.pack(side="left", padx=(0, 10))\n\n        lbl_title = tk.Label(logo_frame, text="Epic Games Launcher", bg=self.c_panel, fg=self.c_fg,\n                             font=("DejaVu Sans", 14, "bold"))\n        lbl_title.pack(side="left")\n\n        # Account & Runner status\n        info_frame = tk.Frame(header, bg=self.c_panel)\n        info_frame.pack(side="right", padx=14, pady=10)\n\n        self.lbl_account = tk.Label(info_frame, text="Checking login status...", bg=self.c_panel,\n                                    fg=self.c_yellow, font=("DejaVu Sans", 10, "bold"))\n        self.lbl_account.pack(anchor="e")\n\n        self.lbl_runner = tk.Label(info_frame, text="Detecting Wine runner...", bg=self.c_panel,\n                                   fg=self.c_muted, font=("DejaVu Sans", 9))\n        self.lbl_runner.pack(anchor="e")\n\n    def create_tabs(self):\n        self.notebook = ttk.Notebook(self)\n        self.notebook.pack(fill="both", expand=True, padx=12, pady=6)\n\n        # Tab 1: Installed Games\n        self.tab_installed = ttk.Frame(self.notebook)\n        self.notebook.add(self.tab_installed, text=" 🎮 Installed Games ")\n        self.build_installed_tab()\n\n        # Tab 2: Epic Library\n        self.tab_library = ttk.Frame(self.notebook)\n        self.notebook.add(self.tab_library, text=" 📥 Full Epic Library ")\n        self.build_library_tab()\n\n        # Tab 3: Console Logs\n        self.tab_console = ttk.Frame(self.notebook)\n        self.notebook.add(self.tab_console, text=" 📜 Launch Console & Logs ")\n        self.build_console_tab()\n\n    def build_installed_tab(self):\n        container = tk.Frame(self.tab_installed, bg=self.c_bg)\n        container.pack(fill="both", expand=True, padx=6, pady=6)\n\n        # Toolbar above games table\n        toolbar = tk.Frame(container, bg=self.c_bg)\n        toolbar.pack(fill="x", side="top", pady=(0, 8))\n\n        btn_launch = tk.Button(toolbar, text=" ▶  LAUNCH SELECTED GAME ", bg=self.c_green, fg="#ffffff",\n                               font=("DejaVu Sans", 11, "bold"), relief="flat", padx=16, pady=6,\n                               activebackground="#27ae60", activeforeground="#ffffff", cursor="hand2",\n                               command=self.launch_selected)\n        btn_launch.pack(side="left", padx=(0, 10))\n\n        btn_sync = tk.Button(toolbar, text=" 🔄 Sync Heroic & Refresh ", bg=self.c_card, fg=self.c_fg,\n                             font=("DejaVu Sans", 10), relief="flat", padx=12, pady=6,\n                             activebackground="#2a364d", cursor="hand2", command=self.sync_and_refresh)\n        btn_sync.pack(side="left", padx=5)\n\n        btn_fix = tk.Button(toolbar, text=" 🛠️ Check GPU & Fix Prefixes ", bg=self.c_card, fg=self.c_fg,\n                            font=("DejaVu Sans", 10), relief="flat", padx=12, pady=6,\n                            activebackground="#2a364d", cursor="hand2", command=self.run_sanitizer)\n        btn_fix.pack(side="left", padx=5)\n\n        btn_login = tk.Button(toolbar, text=" 🔑 Login to Epic Games ", bg=self.c_card, fg="#ffffff",\n                              font=("DejaVu Sans", 10), relief="flat", padx=12, pady=6,\n                              activebackground="#2a364d", cursor="hand2", command=self.open_login_dialog)\n        btn_login.pack(side="right")\n\n        # Table of installed games\n        columns = ("title", "app_name", "version", "size", "install_path")\n        self.tree_installed = ttk.Treeview(container, columns=columns, show="headings", selectmode="browse")\n        self.tree_installed.heading("title", text="Game Title")\n        self.tree_installed.heading("app_name", text="App Name ID")\n        self.tree_installed.heading("version", text="Version")\n        self.tree_installed.heading("size", text="Size")\n        self.tree_installed.heading("install_path", text="Installed Location")\n\n        self.tree_installed.column("title", width=220, anchor="w")\n        self.tree_installed.column("app_name", width=240, anchor="w")\n        self.tree_installed.column("version", width=120, anchor="center")\n        self.tree_installed.column("size", width=90, anchor="center")\n        self.tree_installed.column("install_path", width=250, anchor="w")\n\n        scroll = ttk.Scrollbar(container, orient="vertical", command=self.tree_installed.yview)\n        self.tree_installed.configure(yscrollcommand=scroll.set)\n\n        self.tree_installed.pack(side="left", fill="both", expand=True)\n        scroll.pack(side="right", fill="y")\n\n        self.tree_installed.bind("<Double-1>", lambda e: self.launch_selected())\n\n    def build_library_tab(self):\n        container = tk.Frame(self.tab_library, bg=self.c_bg)\n        container.pack(fill="both", expand=True, padx=6, pady=6)\n\n        toolbar = tk.Frame(container, bg=self.c_bg)\n        toolbar.pack(fill="x", side="top", pady=(0, 8))\n\n        btn_load = tk.Button(toolbar, text=" 🔍 Fetch Library from Epic ", bg=self.c_accent, fg="#ffffff",\n                             font=("DejaVu Sans", 10, "bold"), relief="flat", padx=14, pady=6,\n                             cursor="hand2", command=self.fetch_remote_library)\n        btn_load.pack(side="left", padx=(0, 10))\n\n        btn_install = tk.Button(toolbar, text=" 📥 Install Selected Game ", bg=self.c_card, fg=self.c_fg,\n                                font=("DejaVu Sans", 10), relief="flat", padx=12, pady=6,\n                                cursor="hand2", command=self.install_selected)\n        btn_install.pack(side="left")\n\n        # Filter\n        tk.Label(toolbar, text="Filter:", bg=self.c_bg, fg=self.c_muted, font=("DejaVu Sans", 10)).pack(side="left", padx=(20, 5))\n        self.ent_filter = tk.Entry(toolbar, bg="#1a202c", fg="#ffffff", insertbackground="#ffffff", relief="flat", font=("DejaVu Sans", 10))\n        self.ent_filter.pack(side="left", fill="x", expand=True, padx=(0, 10))\n        self.ent_filter.bind("<KeyRelease>", self.filter_library)\n\n        columns = ("title", "app_name", "status")\n        self.tree_library = ttk.Treeview(container, columns=columns, show="headings", selectmode="browse")\n        self.tree_library.heading("title", text="Game Title")\n        self.tree_library.heading("app_name", text="App Name ID")\n        self.tree_library.heading("status", text="Local Status")\n\n        self.tree_library.column("title", width=400, anchor="w")\n        self.tree_library.column("app_name", width=300, anchor="w")\n        self.tree_library.column("status", width=140, anchor="center")\n\n        scroll = ttk.Scrollbar(container, orient="vertical", command=self.tree_library.yview)\n        self.tree_library.configure(yscrollcommand=scroll.set)\n\n        self.tree_library.pack(side="left", fill="both", expand=True)\n        scroll.pack(side="right", fill="y")\n\n    def build_console_tab(self):\n        container = tk.Frame(self.tab_console, bg=self.c_bg)\n        container.pack(fill="both", expand=True, padx=6, pady=6)\n\n        toolbar = tk.Frame(container, bg=self.c_bg)\n        toolbar.pack(fill="x", side="top", pady=(0, 6))\n\n        btn_stop = tk.Button(toolbar, text=" ⏹  STOP RUNNING GAME ", bg=self.c_red, fg="#ffffff",\n                             font=("DejaVu Sans", 10, "bold"), relief="flat", padx=14, pady=5,\n                             cursor="hand2", command=self.stop_active_game)\n        btn_stop.pack(side="left", padx=(0, 10))\n\n        btn_clear = tk.Button(toolbar, text=" 🧹 Clear Log ", bg=self.c_card, fg=self.c_fg,\n                              font=("DejaVu Sans", 9), relief="flat", padx=10, pady=5,\n                              cursor="hand2", command=lambda: self.log_text.delete("1.0", tk.END))\n        btn_clear.pack(side="left", padx=5)\n\n        self.lbl_status = tk.Label(toolbar, text="Idle", bg=self.c_bg, fg=self.c_muted, font=("DejaVu Sans", 10))\n        self.lbl_status.pack(side="right", padx=10)\n\n        self.log_text = scrolledtext.ScrolledText(container, bg="#0d1117", fg="#c9d1d9",\n                                                 insertbackground="#ffffff", font=("DejaVu Sans Mono", 9),\n                                                 relief="flat", borderwidth=0)\n        self.log_text.pack(fill="both", expand=True)\n\n    def create_bottom_bar(self):\n        bar = tk.Frame(self, bg=self.c_panel, height=32)\n        bar.pack(fill="x", side="bottom", padx=12, pady=(0, 8))\n\n        lbl_tip = tk.Label(bar, text="💡 Game launches require a verified NVIDIA 3D renderer. Run gpu_runtime.py diagnose for details.",\n                           bg=self.c_panel, fg=self.c_muted, font=("DejaVu Sans", 9))\n        lbl_tip.pack(side="left", padx=12, pady=6)\n\n    def log(self, text):\n        def _append():\n            self.log_text.insert(tk.END, text + "\\n")\n            self.log_text.see(tk.END)\n        self.after(0, _append)\n\n    def refresh_all(self):\n        self.refresh_account_status()\n        self.refresh_runner_status()\n        self.refresh_installed_games()\n\n    def refresh_account_status(self):\n        user_file = None\n        for p in [HEROIC_LEG / \'user.json\', STD_LEG / \'user.json\']:\n            if p.is_file():\n                user_file = p\n                break\n\n        if user_file:\n            try:\n                data = json.loads(user_file.read_text(encoding=\'utf-8\'))\n                display_name = data.get(\'displayName\', \'Epic User\')\n                acc_id = data.get(\'account_id\', \'\')\n                short_id = f" ({acc_id[:8]}...)" if acc_id else ""\n                self.lbl_account.config(text=f"🟢 Logged In: {display_name}{short_id}", fg=self.c_green)\n                return\n            except Exception:\n                pass\n        self.lbl_account.config(text="🔴 Not Logged In", fg=self.c_red)\n\n    def refresh_runner_status(self):\n        runner = find_best_runner()\n        name = "Wine-GE-Proton8-26" if "Wine-GE" in runner else Path(runner).name\n        self.lbl_runner.config(text=f"🍷 Runner: {name} (WineD3D OpenGL active)")\n\n    def refresh_installed_games(self):\n        for item in self.tree_installed.get_children():\n            self.tree_installed.delete(item)\n\n        installed_file = None\n        for p in [HEROIC_LEG / \'installed.json\', STD_LEG / \'installed.json\']:\n            if p.is_file():\n                installed_file = p\n                break\n\n        games = {}\n        if installed_file:\n            try:\n                games = json.loads(installed_file.read_text(encoding=\'utf-8\'))\n            except Exception as e:\n                self.log(f"Notice reading installed.json: {e}")\n\n        if not games:\n            # Fallback: Query legendary list-installed --json\n            try:\n                res = subprocess.run(["legendary", "list-installed", "--json"], capture_output=True, text=True, timeout=5)\n                if res.returncode == 0 and res.stdout.strip():\n                    data = json.loads(res.stdout)\n                    if isinstance(data, list):\n                        for g in data:\n                            games[g.get(\'app_name\', \'\')] = g\n                    elif isinstance(data, dict):\n                        games = data\n            except Exception:\n                pass\n\n        for app_name, g in sorted(games.items(), key=lambda x: str(x[1].get(\'title\', \'\')).lower()):\n            title = g.get(\'title\') or app_name\n            version = g.get(\'version\', \'Unknown\')\n            size_str = format_size(g.get(\'install_size\', 0))\n            path_str = g.get(\'install_path\', \'N/A\')\n            self.tree_installed.insert("", "end", values=(title, app_name, version, size_str, path_str))\n\n        # Select first item if available\n        children = self.tree_installed.get_children()\n        if children:\n            self.tree_installed.selection_set(children[0])\n            self.tree_installed.focus(children[0])\n\n    def sync_and_refresh(self):\n        self.log("▶ Syncing Heroic and Legendary configurations...")\n        try:\n            fix_script = BENCH / \'fix_heroic_config.py\'\n            if fix_script.is_file():\n                res = subprocess.run([sys.executable, str(fix_script)], capture_output=True, text=True, timeout=10)\n                self.log(res.stdout)\n                if res.stderr:\n                    self.log(res.stderr)\n            self.refresh_all()\n            self.log("✓ Sync completed successfully.")\n        except Exception as exc:\n            self.log(f"Sync error: {exc}")\n\n    def run_sanitizer(self):\n        self.notebook.select(self.tab_console)\n        self.log("▶ Checking live GPU renderer and updating Wine settings...")\n        threading.Thread(target=self._run_sanitizer_worker, daemon=True).start()\n\n    def _run_sanitizer_worker(self):\n        try:\n            fix_script = BENCH / \'fix_heroic_config.py\'\n            if fix_script.is_file():\n                res = subprocess.run([sys.executable, str(fix_script)], capture_output=True, text=True, timeout=15)\n                self.log(res.stdout)\n            subprocess.run(["chown", "-R", "gamer:gamer", "/home/gamer"], check=False)\n            subprocess.run(["chmod", "-R", "u+rwX", "/home/gamer"], check=False)\n            self.log("✓ Wine settings updated for the detected graphics backend.")\n            self.after(0, self.refresh_all)\n        except Exception as e:\n            self.log(f"Error during sanitization: {e}")\n\n    def launch_selected(self):\n        selected = self.tree_installed.selection()\n        if not selected:\n            messagebox.showinfo("No Selection", "Please select a game from the list to launch.")\n            return\n\n        values = self.tree_installed.item(selected[0], "values")\n        title = values[0]\n        app_name = values[1]\n\n        self.notebook.select(self.tab_console)\n        self.log(f"\\n==================================================================")\n        self.log(f"🚀 Launching game: {title} ({app_name})")\n        self.log(f"==================================================================")\n\n        runner = find_best_runner()\n        self.log(f"Runner: {runner}")\n        self.log("Checking for NVIDIA OpenGL before launch...")\n\n        threading.Thread(target=self._launch_worker, args=(app_name, title, runner), daemon=True).start()\n\n    def _launch_worker(self, app_name, title, runner):\n        env = dict(os.environ)\n        env["DISPLAY"] = os.environ.get("DISPLAY", ":0")\n        env["HOME"] = str(HOME)\n        env["WINEPREFIX"] = str(HOME / ".wine")\n        env["CUDA_VISIBLE_DEVICES"] = "0"\n        direct_xorg = (env["DISPLAY"].split(".", 1)[0] == ":1" and\n                       (BENCH / "gpu-xorg-ready").exists())\n        if direct_xorg:\n            env["GAMELAB_DIRECT_XORG"] = "1"\n            for key in ("VGL_DISPLAY", "MESA_GL_VERSION_OVERRIDE",\n                        "MESA_GLSL_VERSION_OVERRIDE", "__NV_PRIME_RENDER_OFFLOAD"):\n                env.pop(key, None)\n        else:\n            env["VGL_DISPLAY"] = "egl0"\n\n        # Automatically verify and heal Wine prefix before launch if kernel32.dll is missing\n        kernel32 = HOME / ".wine/drive_c/windows/system32/kernel32.dll"\n        if not kernel32.is_file():\n            self.log("▶ Detected incomplete or uninitialized Wine prefix. Auto-initializing with Wine-GE...")\n            wineboot_bin = Path(runner).with_name("wineboot")\n            if wineboot_bin.is_file():\n                try:\n                    init_env = dict(env)\n                    init_env["WINEDLLOVERRIDES"] = "mscoree,mshtml=d"\n                    subprocess.run([str(wineboot_bin), "--init"], env=init_env, check=False, timeout=30)\n                    self.log("✓ Wine prefix auto-initialized successfully.")\n                except Exception as wb_err:\n                    self.log(f"Notice auto-initializing prefix: {wb_err}")\n\n        from gpu_runtime import find_game_binary\n        game_binary = find_game_binary(app_name)\n        if game_binary and game_binary.name.lower() == "wolfneworder_x64.exe":\n            cmd = ["python3", "/tmp/kaggle-game-dev-bench/launch_idtech5.py",\n                   *(["--direct-xorg"] if direct_xorg else []),\n                   "--wine", runner, str(game_binary)]\n        else:\n            cmd = ["python3", "/tmp/kaggle-game-dev-bench/gpu_runtime.py",\n                   "launch", app_name, runner]\n        self.lbl_status.config(text=f"▶ Running: {title}", fg=self.c_green)\n\n        try:\n            self.active_proc = subprocess.Popen(\n                cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env\n            )\n            for line in self.active_proc.stdout:\n                self.log(line.rstrip())\n            self.active_proc.wait()\n            self.log(f"\\nGame process exited with status: {self.active_proc.returncode}")\n        except Exception as exc:\n            self.log(f"Launch exception: {exc}")\n        finally:\n            self.active_proc = None\n            self.lbl_status.config(text="Idle", fg=self.c_muted)\n\n    def stop_active_game(self):\n        if self.active_proc and self.active_proc.poll() is None:\n            self.log("Terminating active game process...")\n            self.active_proc.terminate()\n            time.sleep(1)\n            if self.active_proc.poll() is None:\n                self.active_proc.kill()\n            self.log("Process stopped.")\n        else:\n            self.log("No active game process running.")\n\n    def fetch_remote_library(self):\n        self.log("▶ Fetching Epic Games library via legendary list...")\n        self.ent_filter.delete(0, tk.END)\n        for item in self.tree_library.get_children():\n            self.tree_library.delete(item)\n\n        threading.Thread(target=self._fetch_library_worker, daemon=True).start()\n\n    def _fetch_library_worker(self):\n        try:\n            res = subprocess.run(["legendary", "list", "--json"], capture_output=True, text=True, timeout=25)\n            if res.returncode == 0 and res.stdout.strip():\n                items = json.loads(res.stdout)\n                installed_ids = {self.tree_installed.item(i, "values")[1] for i in self.tree_installed.get_children()}\n\n                self.all_library_items = []\n                for item in items:\n                    app = item.get("app_name", "")\n                    title = item.get("app_title") or item.get("title") or app\n                    status = "✓ Installed" if app in installed_ids else "Available"\n                    self.all_library_items.append((title, app, status))\n\n                self.all_library_items.sort(key=lambda x: x[0].lower())\n                self.after(0, self._populate_library_table, self.all_library_items)\n                self.log(f"✓ Found {len(self.all_library_items)} games in your Epic library.")\n            else:\n                self.log(f"Legendary list returned code {res.returncode}:\\n{res.stderr or res.stdout}")\n        except Exception as e:\n            self.log(f"Library fetch error: {e}")\n\n    def _populate_library_table(self, items):\n        for item in self.tree_library.get_children():\n            self.tree_library.delete(item)\n        for row in items:\n            self.tree_library.insert("", "end", values=row)\n\n    def filter_library(self, event=None):\n        query = self.ent_filter.get().strip().lower()\n        if not hasattr(self, \'all_library_items\'):\n            return\n        if not query:\n            self._populate_library_table(self.all_library_items)\n        else:\n            filtered = [r for r in self.all_library_items if query in r[0].lower() or query in r[1].lower()]\n            self._populate_library_table(filtered)\n\n    def install_selected(self):\n        sel = self.tree_library.selection()\n        if not sel:\n            messagebox.showinfo("No Selection", "Please select a game to install from the library.")\n            return\n\n        values = self.tree_library.item(sel[0], "values")\n        title = values[0]\n        app_name = values[1]\n\n        if not messagebox.askyesno("Install Game", f"Install \'{title}\' ({app_name}) via Legendary?"):\n            return\n\n        self.notebook.select(self.tab_console)\n        self.log(f"\\n▶ Starting installation of {title} ({app_name})...")\n        threading.Thread(target=self._install_worker, args=(app_name,), daemon=True).start()\n\n    def _install_worker(self, app_name):\n        try:\n            cmd = ["legendary", "install", app_name, "-y"]\n            proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)\n            for line in proc.stdout:\n                self.log(line.rstrip())\n            proc.wait()\n            self.log(f"Installation finished with code {proc.returncode}.")\n            self.sync_and_refresh()\n        except Exception as e:\n            self.log(f"Install error: {e}")\n\n    def open_login_dialog(self):\n        dlg = tk.Toplevel(self)\n        dlg.title("Epic Games Authentication")\n        dlg.geometry("560x380")\n        dlg.resizable(False, False)\n        dlg.configure(bg=self.c_bg)\n        dlg.transient(self)\n        dlg.grab_set()\n\n        tk.Label(dlg, text="Log In to Epic Games", bg=self.c_bg, fg="#ffffff",\n                 font=("DejaVu Sans", 13, "bold")).pack(pady=(16, 10))\n\n        frame_steps = tk.Frame(dlg, bg=self.c_panel, padx=14, pady=12)\n        frame_steps.pack(fill="x", padx=16, pady=6)\n\n        msg = (\n            "1. Open this URL in your PC browser (Chrome/Edge/Firefox):\\n"\n            "   https://legendary.gl/epiclogin\\n\\n"\n            "2. Sign in with your Epic account.\\n\\n"\n            "3. The browser will show text containing \'authorizationCode\'.\\n"\n            "4. Copy and paste the code (or the entire JSON string) below:"\n        )\n        tk.Label(frame_steps, text=msg, bg=self.c_panel, fg=self.c_fg, justify="left",\n                 font=("DejaVu Sans", 9)).pack(anchor="w")\n\n        # Copy URL button\n        def _copy_url():\n            self.clipboard_clear()\n            self.clipboard_append("https://legendary.gl/epiclogin")\n            lbl_info.config(text="✓ Link copied to clipboard!", fg=self.c_green)\n\n        btn_copy = tk.Button(frame_steps, text=" 📋 Copy Login Link ", bg=self.c_card, fg=self.c_fg,\n                             font=("DejaVu Sans", 9), relief="flat", cursor="hand2", command=_copy_url)\n        btn_copy.pack(anchor="w", pady=(6, 0))\n\n        # Input entry\n        lbl_code = tk.Label(dlg, text="Authorization Code or JSON Text:", bg=self.c_bg, fg=self.c_muted,\n                            font=("DejaVu Sans", 9, "bold"))\n        lbl_code.pack(anchor="w", padx=18, pady=(10, 2))\n\n        ent_code = tk.Entry(dlg, bg="#1a202c", fg="#ffffff", insertbackground="#ffffff",\n                            relief="flat", font=("DejaVu Sans Mono", 10))\n        ent_code.pack(fill="x", padx=18, ipady=4)\n\n        lbl_info = tk.Label(dlg, text="", bg=self.c_bg, fg=self.c_yellow, font=("DejaVu Sans", 9))\n        lbl_info.pack(pady=4)\n\n        def _submit():\n            raw_input = ent_code.get().strip()\n            if not raw_input:\n                lbl_info.config(text="Please paste an authorization code first.", fg=self.c_red)\n                return\n\n            # Extract code if full JSON was pasted\n            code = raw_input\n            if "{" in raw_input and "authorizationCode" in raw_input:\n                try:\n                    data = json.loads(raw_input)\n                    code = data.get("authorizationCode", raw_input)\n                except Exception:\n                    m = re.search(r\'"authorizationCode"\\s*:\\s*"([a-f0-9]+)"\', raw_input, re.I)\n                    if m:\n                        code = m.group(1)\n\n            lbl_info.config(text="Authenticating with Epic Games...", fg=self.c_yellow)\n            dlg.update()\n\n            def _auth_worker():\n                res = subprocess.run(["legendary", "auth", "--code", code], capture_output=True, text=True)\n                if res.returncode == 0:\n                    self.sync_and_refresh()\n                    self.after(0, lambda: (\n                        messagebox.showinfo("Success", "Epic Games authentication succeeded!"),\n                        dlg.destroy()\n                    ))\n                else:\n                    err_msg = res.stderr or res.stdout\n                    self.after(0, lambda: lbl_info.config(text=f"Auth failed: {err_msg[:60]}", fg=self.c_red))\n\n            threading.Thread(target=_auth_worker, daemon=True).start()\n\n        btn_frame = tk.Frame(dlg, bg=self.c_bg)\n        btn_frame.pack(fill="x", padx=18, pady=10)\n\n        tk.Button(btn_frame, text=" Cancel ", bg=self.c_card, fg=self.c_muted, relief="flat",\n                  cursor="hand2", command=dlg.destroy).pack(side="left")\n\n        tk.Button(btn_frame, text=" 🚀 Log In to Epic Games ", bg=self.c_green, fg="#ffffff",\n                  font=("DejaVu Sans", 10, "bold"), relief="flat", padx=14, pady=5,\n                  cursor="hand2", command=_submit).pack(side="right")\n\nif __name__ == "__main__":\n    app = LegendaryApp()\n    app.mainloop()\n', encoding='utf-8')
(BENCH / 'gpu_runtime.py').write_text('"""Choose a measured graphics path for Legendary/Wine games.\n\nNVENC and nvidia-smi are deliberately not treated as proof of 3D rendering.\nRun ``python3 gpu_runtime.py diagnose`` inside the live desktop session.\n"""\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport re\nimport shutil\nimport subprocess\nimport sys\n\n\ndef run(command, env=None, timeout=12):\n    try:\n        result = subprocess.run(command, env=env, text=True, errors="replace",\n                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT,\n                                timeout=timeout, check=False)\n        return result.returncode, result.stdout\n    except (OSError, subprocess.TimeoutExpired) as exc:\n        return 1, str(exc)\n\n\ndef clean_env():\n    env = os.environ.copy()\n    for key in ("LIBGL_ALWAYS_SOFTWARE", "GALLIUM_DRIVER", "MESA_LOADER_DRIVER_OVERRIDE",\n                "MESA_GL_VERSION_OVERRIDE", "LIBGL_DRIVERS_PATH",\n                "__GLX_VENDOR_LIBRARY_NAME"):\n        env.pop(key, None)\n    # The Selkies input interposer must not affect graphics probes.\n    env.pop("LD_PRELOAD", None)\n    return env\n\n\ndef nvidia_renderer(output):\n    vendor = re.search(r"OpenGL vendor string:\\s*(.+)", output, re.I)\n    renderer = re.search(r"OpenGL renderer string:\\s*(.+)", output, re.I)\n    # Tesla renderers often say "Tesla T4/PCIe/SSE2", without "NVIDIA".\n    return bool(vendor and "NVIDIA" in vendor.group(1).upper() and renderer\n                and not re.search(r"llvmpipe|softpipe|software", renderer.group(1), re.I))\n\n\ndef has_32bit_library(name):\n    injected = os.environ.get("GAMELAB_NVIDIA_LIB32", "")\n    if injected:\n        candidate = Path(injected) / name\n        try:\n            with candidate.open("rb") as library:\n                if library.read(5) == b"\\x7fELF\\x01":\n                    return True\n        except OSError:\n            pass\n    rc, out = run(["ldconfig", "-p"], timeout=4)\n    return rc == 0 and any(f"{name} (libc6)" in line\n                           for line in out.splitlines())\n\n\ndef dxvk_available():\n    root = Path("/opt/gamelab-dxvk/dxvk-2.6.2")\n    return all((root / arch / (name + ".dll")).is_file()\n               for arch in ("x32", "x64")\n               for name in ("d3d9", "d3d10core", "d3d11", "dxgi"))\n\n\ndef egl_device_probe(env):\n    """Test the NVIDIA EGL device directly, independent of Xorg and Xvfb."""\n    eglinfo = Path("/opt/VirtualGL/bin/eglinfo")\n    if not eglinfo.is_file():\n        return False, "VirtualGL eglinfo is not installed", ""\n    rc, listing = run([str(eglinfo), "-e"], env, timeout=8)\n    candidates = list(dict.fromkeys(re.findall(r"\\begl\\d+\\b", listing)))[:8] if rc == 0 else []\n    if not candidates:\n        candidates = ["egl0"]\n    last_result = listing[-240:]\n    for target in candidates:\n        rc, out = run([str(eglinfo), target], env, timeout=12)\n        renderer = next((line.strip() for line in out.splitlines()\n                         if line.strip().startswith("OpenGL renderer string:")), "")\n        vendor = next((line.strip() for line in out.splitlines()\n                       if line.strip().startswith("OpenGL vendor string:")), "")\n        if (rc == 0 and "NVIDIA" in vendor.upper() and renderer\n                and not re.search(r"llvmpipe|softpipe|software", renderer, re.I)):\n            return True, renderer, target\n        last_result = renderer or out[-240:]\n    return False, last_result, ""\n\n\ndef cpu_quota_count():\n    """Return the cgroup CPU quota as a whole-core budget, if one exists."""\n    try:\n        quota, period = Path("/sys/fs/cgroup/cpu.max").read_text().split()[:2]\n        if quota != "max":\n            return max(1, math.floor(int(quota) / int(period)))\n    except (OSError, ValueError, ZeroDivisionError):\n        pass\n    try:\n        quota = int(Path("/sys/fs/cgroup/cpu/cpu.cfs_quota_us").read_text())\n        period = int(Path("/sys/fs/cgroup/cpu/cpu.cfs_period_us").read_text())\n        if quota > 0:\n            return max(1, math.floor(quota / period))\n    except (OSError, ValueError, ZeroDivisionError):\n        pass\n    return None\n\n\ndef software_threads():\n    override = os.environ.get("LP_NUM_THREADS", "")\n    if override.isdigit() and 1 <= int(override) <= 32:\n        return int(override)\n    try:\n        cores = len(os.sched_getaffinity(0))\n    except AttributeError:\n        cores = os.cpu_count() or 1\n    quota = cpu_quota_count()\n    if quota is not None:\n        cores = min(cores, quota)\n    reserve = 2 if cores >= 4 else 1\n    return max(1, min(8, cores - reserve))\n\n\ndef lower_software_resolution(env):\n    """Best-effort temporary Xvfb mode change; game settings may differ."""\n    requested = os.environ.get("GAMELAB_SOFTWARE_RESOLUTION", "854x480")\n    if requested == "off" or not shutil.which("xrandr") or not env.get("DISPLAY"):\n        return None\n    if requested not in ("640x360", "854x480", "1024x576", "1280x720"):\n        print(f"Unsupported GAMELAB_SOFTWARE_RESOLUTION={requested!r}; keeping desktop mode.",\n              file=sys.stderr)\n        return None\n    rc, out = run(["xrandr", "--current"], env, timeout=5)\n    match = re.search(r"\\bcurrent (\\d+) x (\\d+)\\b", out)\n    if rc != 0 or not match:\n        return None\n    previous = f"{match.group(1)}x{match.group(2)}"\n    if previous == requested:\n        return None\n    rc, out = run(["xrandr", "-s", requested], env, timeout=5)\n    if rc != 0:\n        print(f"Could not set {requested} desktop mode: {out[-200:]}", file=sys.stderr)\n        return None\n    print(f"Software desktop mode: {requested} (restore {previous} on exit)", flush=True)\n    return previous\n\n\ndef install_dxvk(prefix):\n    """Install both bitnesses without following Wine DLL symlinks."""\n    root = Path("/opt/gamelab-dxvk/dxvk-2.6.2")\n    for arch, subdir in (("x64", "system32"), ("x32", "syswow64")):\n        destination = Path(prefix) / "drive_c/windows" / subdir\n        if not destination.is_dir():\n            raise RuntimeError(f"Wine prefix is incomplete: {destination}")\n        for name in ("d3d9", "d3d10core", "d3d11", "dxgi"):\n            target = destination / (name + ".dll")\n            backup = destination / (name + ".dll.gamelab-original")\n            temporary = destination / (name + ".dll.gamelab-new")\n            shutil.copyfile(root / arch / target.name, temporary)\n            if (target.exists() or target.is_symlink()) and not backup.exists():\n                os.replace(target, backup)\n            os.replace(temporary, target)\n\n\ndef install_vkd3d(prefix):\n    """Install VKD3D-Proton (DirectX 12) if available."""\n    candidates = list(Path("/opt/gamelab-vkd3d").glob("vkd3d-proton-*"))\n    if not candidates:\n        return\n    root = candidates[0]\n    for arch, subdir in (("x64", "system32"), ("x86", "syswow64")):\n        src_dir = root / arch\n        if not src_dir.is_dir():\n            continue\n        dst_dir = Path(prefix) / "drive_c/windows" / subdir\n        if not dst_dir.is_dir():\n            continue\n        for name in ("d3d12.dll", "d3d12core.dll"):\n            src_file = src_dir / name\n            if src_file.is_file():\n                shutil.copyfile(src_file, dst_dir / name)\n\n\ndef diagnose():\n    env = clean_env()\n    facts = {\n        "display": env.get("DISPLAY", ""),\n        "nvidia_control": Path("/dev/nvidiactl").exists(),\n        "nvidia_gpu": Path("/dev/nvidia0").exists(),\n        "nvidia_modeset": Path("/dev/nvidia-modeset").exists(),\n        "nvidia_icd": any(Path(p).exists() for p in (\n            "/etc/vulkan/icd.d/nvidia_icd.json",\n            "/usr/share/vulkan/icd.d/nvidia_icd.json")) or any(\n                Path(p).is_file() and "nvidia" in Path(p).name.lower()\n                for p in (env.get("VK_DRIVER_FILES") or env.get("VK_ICD_FILENAMES") or "").split(":") if p),\n        "direct_gl": False, "virtualgl": False, "virtualgl_display": "",\n        "vulkan_device": False,\n        "vulkan_window": False,\n        "nvidia_32bit_gl": has_32bit_library("libGLX_nvidia.so.0"),\n        "vulkan_32bit_loader": has_32bit_library("libvulkan.so.1"),\n        "dxvk_files": dxvk_available(),\n    }\n    facts["egl_device"], facts["egl_renderer"], facts["egl_device_display"] = egl_device_probe(env)\n    if shutil.which("glxinfo") and env.get("DISPLAY"):\n        rc, out = run(["glxinfo", "-B"], env)\n        facts["direct_gl"] = rc == 0 and nvidia_renderer(out)\n        facts["gl_renderer"] = next((line.strip() for line in out.splitlines()\n                                     if "OpenGL renderer string:" in line), out[-240:])\n        vgl = shutil.which("vglrun") or "/opt/VirtualGL/bin/vglrun"\n        if Path(vgl).is_file() and facts["nvidia_control"] and facts["nvidia_gpu"]:\n            candidates = [facts["egl_device_display"]] if facts["egl_device"] else ["egl0"]\n            if Path("/tmp/kaggle-game-dev-bench/gpu-xorg-ready").exists():\n                candidates.append(":1")\n            for target in candidates:\n                rc, out = run([vgl, "-d", target, "glxinfo", "-B"], env)\n                if rc == 0 and nvidia_renderer(out):\n                    facts["virtualgl"] = True\n                    facts["virtualgl_display"] = target\n                    facts["vgl_renderer"] = next((line.strip() for line in out.splitlines()\n                                                   if "OpenGL renderer string:" in line), out[-240:])\n                    break\n    if facts["nvidia_control"] and facts["nvidia_gpu"] and shutil.which("vulkaninfo"):\n        rc, out = run(["vulkaninfo", "--summary"], env)\n        facts["vulkan_device"] = rc == 0 and bool(re.search(\n            r"deviceName\\s*=.*(?:NVIDIA|Tesla T4)", out, re.I))\n        facts["vulkan_summary"] = out[-500:]\n    if facts["vulkan_device"] and env.get("DISPLAY"):\n        rc, out = run([sys.executable, "-m", "selkies.gpu_probe", "--vulkan-window"], env)\n        facts["vulkan_window"] = rc == 0 and out.strip() == "true"\n    return facts\n\n\ndef legendary_install_path(app):\n    """Return an installed Epic title\'s directory from Legendary metadata."""\n    home = Path(os.environ.get("KAGGLE_GAMER_HOME", "/home/gamer"))\n    candidates = (\n        home / ".config/heroic/legendaryConfig/legendary/installed.json",\n        home / ".config/legendary/installed.json",\n    )\n    for installed in candidates:\n        try:\n            record = json.loads(installed.read_text(encoding="utf-8")).get(app, {})\n            path = Path(record.get("install_path") or record.get("installPath") or "")\n            if path.is_dir():\n                return path\n        except (OSError, ValueError, AttributeError):\n            continue\n    return None\n\n\ndef unreal_shipping_binary(app):\n    """Find a UE Shipping executable while excluding prerequisite launchers."""\n    install = legendary_install_path(app)\n    if install is None:\n        return None\n    matches = []\n    for path in install.rglob("*-Win64-Shipping.exe"):\n        lowered = str(path).lower()\n        if "prereq" not in lowered and "redist" not in lowered and "crashreport" not in lowered:\n            matches.append(path)\n    matches.sort(key=lambda path: ("binaries\\\\win64" not in str(path).lower().replace("/", "\\\\"), len(str(path))))\n    return matches[0] if matches else None\n\n\ndef find_game_binary(app):\n    """Find the game\'s actual executable binary (supporting UE, id Tech, Unity, custom engines)."""\n    ue_shipping = unreal_shipping_binary(app)\n    if ue_shipping and ue_shipping.is_file():\n        return ue_shipping\n\n    home = Path(os.environ.get("KAGGLE_GAMER_HOME", "/home/gamer"))\n    install_candidates = []\n\n    # Check installed.json metadata (exact key match, then fuzzy/case-insensitive match)\n    for installed_file in (\n        home / ".config/heroic/legendaryConfig/legendary/installed.json",\n        home / ".config/legendary/installed.json",\n    ):\n        try:\n            data = json.loads(installed_file.read_text(encoding="utf-8"))\n            if app in data:\n                record = data[app]\n                ip = Path(record.get("install_path") or record.get("installPath") or "")\n                ex = record.get("executable") or ""\n                if ip.is_dir():\n                    if ex and (ip / ex).is_file():\n                        return ip / ex\n                    for p in ip.rglob("*.exe"):\n                        low = p.name.lower()\n                        if any(skip in low for skip in ("prereq", "redist", "crash", "unins", "setup", "dxsetup", "vcredist", "epicgameslauncher")):\n                            continue\n                        if ex and p.name.lower() == Path(ex).name.lower():\n                            return p\n                        if "x64" in low or "wolf" in low:\n                            return p\n                    install_candidates.append(ip)\n            for key, record in data.items():\n                if not isinstance(record, dict):\n                    continue\n                k_low = key.lower()\n                a_low = app.lower()\n                t_low = str(record.get("title", "")).lower()\n                if a_low in k_low or k_low in a_low or a_low in t_low:\n                    ip = Path(record.get("install_path") or record.get("installPath") or "")\n                    ex = record.get("executable") or ""\n                    if ip.is_dir():\n                        if ex and (ip / ex).is_file():\n                            return ip / ex\n                        for p in ip.rglob("*.exe"):\n                            low = p.name.lower()\n                            if any(skip in low for skip in ("prereq", "redist", "crash", "unins", "setup", "dxsetup", "vcredist", "epicgameslauncher")):\n                                continue\n                            if ex and p.name.lower() == Path(ex).name.lower():\n                                return p\n                            if "x64" in low or "wolf" in low:\n                                return p\n                        install_candidates.append(ip)\n        except Exception:\n            pass\n\n    install_candidates.extend([\n        home / "game-builds",\n        home / ".wine/drive_c/Program Files",\n        home / ".wine/drive_c/Program Files (x86)",\n        Path("/kaggle/working"),\n    ])\n\n    found_exes = []\n    for root in install_candidates:\n        if not root.is_dir():\n            continue\n        try:\n            for p in root.rglob("*.exe"):\n                low = p.name.lower()\n                if any(skip in low for skip in ("prereq", "redist", "crash", "unins", "setup", "dxsetup", "vcredist", "epicgameslauncher")):\n                    continue\n                found_exes.append(p)\n        except Exception:\n            pass\n\n    if found_exes:\n        a_low = app.lower()\n        found_exes.sort(key=lambda p: (\n            0 if a_low in p.name.lower() else 1,\n            0 if ("x64" in p.name.lower() or "64" in p.name.lower()) else 1,\n            len(p.parts)\n        ))\n        return found_exes[0]\n    return None\n\n\ndef get_input_mode() -> str:\n    """Read the active input mode (mouse_keyboard or gamepad)."""\n    mode_file = Path("/tmp/kaggle-game-dev-bench/input-mode.json")\n    try:\n        if mode_file.is_file():\n            data = json.loads(mode_file.read_text(encoding="utf-8"))\n            return data.get("mode", "mouse_keyboard")\n    except Exception:\n        pass\n    return os.environ.get("SEVEN_BZ_INPUT_MODE", "mouse_keyboard")\n\n\ndef configure_mouse(wine_bin, env):\n    """Apply verified Wine DirectInput, X11 pointer capture, and raw sensitivity."""\n    if not wine_bin:\n        return\n    values = (\n        ("HKEY_CURRENT_USER\\\\Software\\\\Wine\\\\DirectInput", "MouseWarpOverride", "force"),\n        ("HKEY_CURRENT_USER\\\\Software\\\\Wine\\\\X11 Driver", "GrabPointer", "Y"),\n        ("HKEY_CURRENT_USER\\\\Software\\\\Wine\\\\X11 Driver", "UseTakeFocus", "N"),\n        ("HKEY_CURRENT_USER\\\\Control Panel\\\\Mouse", "MouseSpeed", "0"),\n        ("HKEY_CURRENT_USER\\\\Control Panel\\\\Mouse", "MouseThreshold1", "0"),\n        ("HKEY_CURRENT_USER\\\\Control Panel\\\\Mouse", "MouseThreshold2", "0"),\n        ("HKEY_CURRENT_USER\\\\Control Panel\\\\Mouse", "MouseSensitivity", "10"),\n    )\n    for key, name, value in values:\n        try:\n            subprocess.run([wine_bin, "reg", "add", key, "/v", name, "/t", "REG_SZ",\n                            "/d", value, "/f"], env=env, check=False,\n                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=8)\n        except (OSError, subprocess.TimeoutExpired):\n            pass\n\n\ndef select_graphics_backend(facts, preference="auto", is_64bit=True):\n    """Select only paths whose required presentation probe succeeded."""\n    if preference not in ("auto", "dxvk", "wined3d"):\n        raise ValueError("GAMELAB_GRAPHICS_BACKEND must be auto, dxvk, or wined3d")\n    dxvk_ready = (facts["vulkan_window"] and (facts["vulkan_32bit_loader"] or is_64bit)\n                  and facts["dxvk_files"])\n    gl_ready = facts["nvidia_32bit_gl"] or is_64bit\n    if preference == "dxvk":\n        if not dxvk_ready:\n            raise RuntimeError("DXVK requires a working Vulkan X11 swapchain, 32-bit loader, and both DLL sets")\n        return "dxvk"\n    if preference == "auto" and dxvk_ready:\n        return "dxvk"\n    if facts["direct_gl"] and gl_ready:\n        return "direct_gl"\n    if facts["virtualgl"] and gl_ready:\n        return "virtualgl"\n    if preference == "wined3d":\n        if facts["virtualgl"] and gl_ready:\n            return "virtualgl"\n        raise RuntimeError("WineD3D requires verified NVIDIA GL and 32-bit GL libraries")\n    return "software"\n\n\ndef launch(app, wine):\n    facts = diagnose()\n    env = clean_env()\n    direct_xorg = (env.get("DISPLAY", "").split(".", 1)[0] == ":1" and\n                   facts["direct_gl"] and\n                   os.environ.get("GAMELAB_DIRECT_XORG") == "1")\n    env["WINEPREFIX"] = os.environ.get("WINEPREFIX", "/home/gamer/.wine")\n    env["WINEDLLOVERRIDES"] = "d3d11,dxgi,d3d9,d3d10core=b;mscoree,mshtml=d"\n    env["SDL_AUDIODRIVER"] = "pulse"\n    env["HOME"] = os.environ.get("KAGGLE_GAMER_HOME", "/home/gamer")\n    env["__GL_SYNC_TO_VBLANK"] = "0"\n\n    # Dedicate GPU 0 exclusively to 3D rendering and physics simulation\n    env["CUDA_VISIBLE_DEVICES"] = "0"\n    env["DRI_PRIME"] = "0"\n    if not direct_xorg:\n        env["VGL_DISPLAY"] = "egl0"\n    env["DXVK_FILTER_DEVICE_NAME"] = "Tesla T4"\n    env["WINE_FULLSCREEN_FSR"] = "1"\n    env["WINE_FULLSCREEN_FSR_STRENGTH"] = "2"\n    env["VKD3D_CONFIG"] = "dxr11,dxr"\n    env["VKD3D_FEATURE_LEVEL"] = "12_1"\n\n    # Guarantee full OpenGL 4.5 Compatibility Profile & WGL extensions for Wine/idTech\n    if not direct_xorg:\n        env["MESA_GL_VERSION_OVERRIDE"] = "4.5COMPAT"\n        env["MESA_GLSL_VERSION_OVERRIDE"] = "450"\n        env["__NV_PRIME_RENDER_OFFLOAD"] = "1"\n        env["__GLX_VENDOR_LIBRARY_NAME"] = "nvidia"\n    if not env.get("DISPLAY"):\n        env["DISPLAY"] = ":0"\n\n    # Propagate hardware profile to Legendary configuration\n    try:\n        leg_cfg_path = Path(env["HOME"]) / ".config/legendary/config.ini"\n        leg_cfg_path.parent.mkdir(parents=True, exist_ok=True)\n        current_cfg = leg_cfg_path.read_text(encoding="utf-8") if leg_cfg_path.is_file() else ""\n        if "[default.env]" not in current_cfg:\n            with leg_cfg_path.open("a", encoding="utf-8") as f:\n                f.write("\\n[default.env]\\nVGL_DISPLAY = egl0\\nMESA_GL_VERSION_OVERRIDE = 4.5COMPAT\\nMESA_GLSL_VERSION_OVERRIDE = 450\\n__NV_PRIME_RENDER_OFFLOAD = 1\\n__GLX_VENDOR_LIBRARY_NAME = nvidia\\n")\n    except Exception:\n        pass\n\n    if not env.get("PULSE_SERVER"):\n        for candidate in ("/run/user/1000/pulse/native", "/tmp/pulse-PKdhtXMmr18n/native"):\n            if Path(candidate).exists():\n                env["PULSE_SERVER"] = f"unix:{candidate}"\n                break\n        if not env.get("PULSE_SERVER"):\n            try:\n                found = list(Path("/tmp").glob("pulse-*/native")) + list(Path("/run/user").glob("*/pulse/native"))\n                if found:\n                    env["PULSE_SERVER"] = f"unix:{found[0]}"\n            except Exception:\n                pass\n\n    wine_bin = wine or shutil.which("wine")\n    configure_mouse(wine_bin, env)\n\n    input_mode = get_input_mode()\n    print(f"Active input mode: {input_mode}", flush=True)\n    if input_mode == "mouse_keyboard":\n        # Pure Mouse & Keyboard: Remove interposer and disable Wine virtual joystick\n        # to prevent ghost controller conflict (locks Pitch / Yaw fighting).\n        env.pop("LD_PRELOAD", None)\n        env["SDL_JOYSTICK_DISABLE_UDEV"] = "1"\n        env["SDL_JOYSTICK_DEVICE"] = ""\n        env["WINEDLLOVERRIDES"] = env.get("WINEDLLOVERRIDES", "") + ";winebus.sys=d"\n    else:\n        # Gamepad Mode: Bind Xbox 360 controller sockets via interposers\n        env["SELKIES_JS_SOCKET_PATH"] = ("/tmp/kaggle-game-dev-bench/direct-stream-sockets"\n                                         if direct_xorg else "/tmp")\n        env["SDL_JOYSTICK_DISABLE_UDEV"] = "1"\n        env["SDL_JOYSTICK_DEVICE"] = "/dev/input/event1000:/dev/input/event1001:/dev/input/event1002:/dev/input/event1003"\n        interposer = "/tmp/kaggle-game-dev-bench/$LIB/selkies_input_interposer.so:/tmp/kaggle-game-dev-bench/$LIB/selkies_fake_udev.so"\n        env["LD_PRELOAD"] = os.environ.get("LD_PRELOAD", interposer)\n\n    target_bin = find_game_binary(app)\n    is_64bit = True\n    if target_bin and "32" in target_bin.name.lower() and "64" not in target_bin.name.lower():\n        is_64bit = False\n    launch_cwd = str(target_bin.parent) if target_bin else None\n\n    # Switch to 64-bit Wine binary for 64-bit binaries to prevent 32-bit wrapper dropping 64-bit VGL faker\n    if is_64bit and wine_bin:\n        wine64_candidate = Path(wine_bin).with_name("wine64")\n        if wine64_candidate.is_file():\n            wine_bin = str(wine64_candidate)\n\n    vgl = shutil.which("vglrun") or "/opt/VirtualGL/bin/vglrun"\n    vgl_ready = (not direct_xorg and Path(vgl).is_file() and\n                 (facts.get("virtualgl") or facts.get("egl_device")))\n    vgl_display = facts.get("virtualgl_display") or facts.get("egl_device_display") or "egl0"\n\n    # vglrun enables its dlopen interposer by default.  -dl is not a valid\n    # option in current VirtualGL and is interpreted as the executable.\n    wrap_script = Path("/tmp/kaggle-game-dev-bench/vgl-wrap.sh")\n    try:\n        wrap_script.write_text(f"#!/bin/bash\\nexec {vgl} -d {vgl_display} \\"$@\\"\\n", encoding="utf-8")\n        wrap_script.chmod(0o755)\n    except Exception:\n        pass\n\n    if vgl_ready:\n        env["VGL_DISPLAY"] = vgl_display\n        env.update(VGL_READBACK="pbo", VGL_SPOIL="1", VGL_SYNC="0", VGL_COMPRESS="proxy")\n        vgl_fps = os.environ.get("GAMELAB_VGL_FPS", "60")\n        try:\n            if 0 <= float(vgl_fps) <= 240:\n                env["VGL_FPS"] = vgl_fps\n        except ValueError:\n            pass\n\n    if target_bin and wine_bin:\n        print(f"Target game binary detected: {target_bin}", flush=True)\n        command = [wine_bin, str(target_bin)]\n    else:\n        command = ["legendary", "launch", app, "--wine", wine]\n        if vgl_ready:\n            command.extend(["--wrapper", str(wrap_script)])\n\n    # Prepend vglrun so the process tree inherits its OpenGL interposers.\n    if vgl_ready:\n        command = [vgl, "-d", vgl_display, *command]\n\n    try:\n        selected = select_graphics_backend(\n            facts, os.environ.get("GAMELAB_GRAPHICS_BACKEND", "auto"), is_64bit=is_64bit)\n    except (ValueError, RuntimeError) as exc:\n        print(f"Graphics backend unavailable: {exc}", file=sys.stderr)\n        return 78\n    if selected == "dxvk":\n        try:\n            install_dxvk(env["WINEPREFIX"])\n        except (OSError, RuntimeError) as exc:\n            print(f"DXVK setup failed: {exc}", file=sys.stderr)\n            return 78\n        try:\n            install_vkd3d(env["WINEPREFIX"])\n        except Exception as exc:\n            print(f"VKD3D setup notice: {exc}", file=sys.stderr)\n        env["WINEDLLOVERRIDES"] = "d3d12,d3d12core,d3d11,dxgi,d3d9,d3d10core=n,b;mscoree,mshtml=d"\n        env["DXVK_HUD"] = "devinfo,fps"\n        backend = "NVIDIA Vulkan through DXVK & VKD3D + VirtualGL (GPU 0)"\n    elif selected == "direct_gl":\n        backend = "NVIDIA OpenGL"\n    elif selected == "virtualgl":\n        display = facts.get("virtualgl_display") or "egl0"\n        backend = f"NVIDIA OpenGL through VirtualGL ({display})"\n    elif os.environ.get("GAMELAB_ALLOW_SOFTWARE") == "1":\n        backend = "Mesa software (explicit opt-in)"\n        env["WINEDLLOVERRIDES"] = "d3d11,dxgi,d3d9,d3d10core=b;mscoree,mshtml=d"\n        env["LIBGL_ALWAYS_SOFTWARE"] = "1"\n        env["GALLIUM_DRIVER"] = "llvmpipe"\n        env["LP_NUM_THREADS"] = str(software_threads())\n        if shutil.which("nice"):\n            command = ["nice", "-n", "5", *command]\n    else:\n        backend = "Mesa software (fallback)"\n        env["WINEDLLOVERRIDES"] = "d3d11,dxgi,d3d9,d3d10core=b;mscoree,mshtml=d"\n        env["LIBGL_ALWAYS_SOFTWARE"] = "1"\n        env["GALLIUM_DRIVER"] = "llvmpipe"\n        env["LP_NUM_THREADS"] = str(software_threads())\n        if shutil.which("nice"):\n            command = ["nice", "-n", "5", *command]\n    print(f"Rendering backend: {backend}", flush=True)\n    if backend.startswith("Mesa software"):\n        print(f"llvmpipe worker threads: {env[\'LP_NUM_THREADS\']}", flush=True)\n        previous_resolution = lower_software_resolution(env)\n        try:\n            return subprocess.call(command, env=env, cwd=launch_cwd)\n        finally:\n            if previous_resolution:\n                run(["xrandr", "-s", previous_resolution], env, timeout=5)\n    return subprocess.call(command, env=env, cwd=launch_cwd)\n\n\nif __name__ == "__main__":\n    if len(sys.argv) == 2 and sys.argv[1] == "diagnose":\n        print(json.dumps(diagnose(), indent=2))\n    elif len(sys.argv) == 4 and sys.argv[1] == "launch":\n        sys.exit(launch(sys.argv[2], sys.argv[3]))\n    else:\n        print("Usage: gpu_runtime.py diagnose | launch APP WINE_BIN", file=sys.stderr)\n        sys.exit(2)\n', encoding='utf-8')
(BENCH / 'launch_idtech5.py').write_text('"""Launch a 64-bit id Tech OpenGL game on the Selkies X desktop.\n\nThe GLX extension must exist on the captured X server.  VirtualGL sends\nOpenGL work to the selected NVIDIA GLX server or EGL device and copies\ncompleted frames into the Wine window on DISPLAY. Selkies captures :0.\n"""\n\nimport argparse\nimport os\nfrom pathlib import Path\nimport re\nimport subprocess\nimport sys\n\n\nVGL = Path("/opt/VirtualGL/bin/vglrun")\n\n\ndef probe(args, env, timeout=15):\n    result = subprocess.run(args, env=env, text=True, errors="replace",\n                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,\n                            timeout=timeout, check=False)\n    return result.returncode, result.stdout\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("game", type=Path,\n                        help="Path to a 64-bit Windows OpenGL game executable")\n    parser.add_argument("--wine", type=Path, required=True,\n                        help="Path to Wine-GE\'s bin/wine or bin/wine64")\n    parser.add_argument("--vgl-display", default="auto",\n                        help="VirtualGL target: auto, :1, or egl0 (default: auto)")\n    parser.add_argument("--direct-xorg", action="store_true",\n                        help="Run directly on the NVIDIA Xorg display :1 (use its Selkies stream)")\n    parser.add_argument("--egl-device", default=None,\n                        help="Deprecated alias for --vgl-display")\n    parser.add_argument("--prefix", type=Path,\n                        default=Path("/home/gamer/.wine"))\n    parser.add_argument("game_args", nargs=argparse.REMAINDER)\n    args = parser.parse_args()\n\n    game = args.game.resolve()\n    wine = args.wine.resolve()\n    if not game.is_file() or not wine.is_file() or (not args.direct_xorg and not VGL.is_file()):\n        parser.error("game, Wine binary, and selected graphics launcher must exist")\n\n    env = os.environ.copy()\n    env["DISPLAY"] = ":1" if args.direct_xorg else (env.get("DISPLAY") or ":0")\n    env["WINEPREFIX"] = str(args.prefix)\n    env["VGL_COMPRESS"] = "proxy"\n    env["SDL_AUDIODRIVER"] = "pulse"\n    overrides = env.get("WINEDLLOVERRIDES", "")\n    env["WINEDLLOVERRIDES"] = (overrides + ";" if overrides else "") + "opengl32=b"\n    # These variables cannot add GLX to Xvfb or select the VirtualGL GPU.\n    for key in ("LIBGL_ALWAYS_SOFTWARE", "GALLIUM_DRIVER",\n                "MESA_LOADER_DRIVER_OVERRIDE", "MESA_GL_VERSION_OVERRIDE",\n                "MESA_GLSL_VERSION_OVERRIDE", "__NV_PRIME_RENDER_OFFLOAD",\n                "__GLX_VENDOR_LIBRARY_NAME"):\n        env.pop(key, None)\n\n    if args.direct_xorg:\n        # A direct GLX context and the captured window must live on the same\n        # NVIDIA X server. VirtualGL\'s cross-server drawable mapping is not\n        # reliable for this game\'s multi-context WGL path.\n        env.pop("VGL_DISPLAY", None)\n        for key in ("VGL_SYNC", "VGL_READBACK", "VGL_LOGO", "VGL_PROFILE"):\n            env.pop(key, None)\n        socket_dir = Path("/tmp/kaggle-game-dev-bench/direct-stream-sockets")\n        input_hook = Path("/tmp/kaggle-game-dev-bench/lib64/selkies_input_interposer.so")\n        if socket_dir.is_dir() and input_hook.is_file():\n            env["SELKIES_JS_SOCKET_PATH"] = str(socket_dir)\n            env["LD_PRELOAD"] = (str(input_hook) + ":" + env.get("LD_PRELOAD", "")).rstrip(":")\n            env["SDL_JOYSTICK_DISABLE_UDEV"] = "1"\n        try:\n            rc, info = probe(["glxinfo", "-B"], env)\n            if rc or not re.search(r"OpenGL vendor string:\\s*NVIDIA", info):\n                raise RuntimeError("NVIDIA GLX is unavailable on :1: " + info[-500:])\n        except (OSError, subprocess.TimeoutExpired, RuntimeError) as exc:\n            print(f"Graphics prerequisite failed: {exc}", file=sys.stderr)\n            return 78\n        wine_bin = wine.with_name("wine64") if wine.with_name("wine64").is_file() else wine\n        command = [str(wine_bin), str(game)]\n        command.extend(args.game_args or\n                       ["+set", "com_skipIntroVideo", "1",\n                        "+set", "com_skipBootSequence", "1",\n                        "+set", "r_fullscreen", "0", "+set", "r_mode", "-1",\n                        "+set", "r_windowedWidth", "1280",\n                        "+set", "r_windowedHeight", "720"])\n        print("OpenGL target: direct NVIDIA Xorg :1", flush=True)\n        print("Stream this display using start_direct_stream.sh", flush=True)\n        return subprocess.call(command, cwd=game.parent, env=env)\n\n    try:\n        rc, extensions = probe(["xdpyinfo", "-queryExtensions"], env)\n        if rc or not re.search(r"(?m)^\\s*GLX\\s+\\(opcode:", extensions):\n            raise RuntimeError(\n                f"{env[\'DISPLAY\']} does not advertise GLX. Wine\'s winex11.drv "\n                "will disable WGL. Restart the captured Xvfb with GLX enabled "\n                "or use a GLX-capable captured X server."\n            )\n\n        selected = args.egl_device or args.vgl_display\n        if selected == "auto":\n            candidates = ([":1"] if Path("/tmp/kaggle-game-dev-bench/gpu-xorg-ready").exists()\n                          else []) + ["egl0"]\n        else:\n            candidates = [selected]\n        failures = []\n        for candidate in candidates:\n            rc, info = probe([str(VGL), "-d", candidate, "glxinfo", "-B"], env)\n            if (not rc and re.search(r"OpenGL vendor string:\\s*NVIDIA", info)\n                    and not re.search(r"OpenGL renderer string:.*(?:llvmpipe|softpipe)", info, re.I)):\n                selected = candidate\n                break\n            failures.append(f"{candidate}: {info[-500:]}")\n        else:\n            raise RuntimeError("VirtualGL NVIDIA probe failed:\\n" + "\\n".join(failures))\n    except (OSError, subprocess.TimeoutExpired, RuntimeError) as exc:\n        print(f"Graphics prerequisite failed: {exc}", file=sys.stderr)\n        return 78\n\n    # vglrun already preloads libdlfaker and libvglfaker by default.\n    wine_bin = wine.with_name("wine64") if wine.with_name("wine64").is_file() else wine\n    env["VGL_DISPLAY"] = selected\n    command = [str(VGL), "-d", selected, str(wine_bin), str(game)]\n    if not args.game_args:\n        command.extend(["-window", "+com_skipIntroVideo", "1", "+r_fullscreen", "0"])\n    else:\n        command.extend(args.game_args)\n    print("OpenGL target:", selected, flush=True)\n    print("Wine window:", env["DISPLAY"], flush=True)\n    return subprocess.call(command, cwd=game.parent, env=env)\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n', encoding='utf-8')
(BENCH / 'start_direct_stream.sh').write_text('#!/bin/bash\n# Stream the NVIDIA Xorg display directly. Run as root after the notebook\'s\n# ordinary desktop session has created PulseAudio and gpu_xorg.sh has run.\nset -euo pipefail\nBENCH=/tmp/kaggle-game-dev-bench\nPORT="${DIRECT_STREAM_PORT:-8081}"\nWIDTH="${DIRECT_STREAM_WIDTH:-1280}"\nHEIGHT="${DIRECT_STREAM_HEIGHT:-720}"\nORIGIN="${SELKIES_ALLOWED_ORIGINS:-https://kkb-production.jupyter-proxy.kaggle.net}"\nif [ "$(id -u)" -ne 0 ]; then\n    echo \'This Xorg installation requires root for Selkies MIT-SHM capture.\' >&2\n    exit 1\nfi\nbash "$BENCH/gpu_xorg.sh"\ntest -e "$BENCH/gpu-xorg-ready"\nDISPLAY=:1 XAUTHORITY="$BENCH/nvidia-Xauthority" \\\n    glxinfo -B | grep -q \'OpenGL vendor string: NVIDIA\'\nif command -v xrandr >/dev/null; then\n    DISPLAY=:1 XAUTHORITY="$BENCH/nvidia-Xauthority" \\\n        xrandr --output DVI-D-0 --mode "${WIDTH}x${HEIGHT}"\nfi\nPULSE_SERVER="$(cat "$BENCH/pulse-server-path" 2>/dev/null || true)"\nexport DISPLAY=:1 XAUTHORITY="$BENCH/nvidia-Xauthority" PULSE_SERVER\nENCODER_DEVICE=0\nif [ -e /dev/nvidia1 ]; then ENCODER_DEVICE=1; fi\nexport CUDA_VISIBLE_DEVICES="$ENCODER_DEVICE"\nexport SELKIES_ALLOWED_ORIGINS="$ORIGIN"\nexport SELKIES_JS_SOCKET_PATH="$BENCH/direct-stream-sockets"\nmkdir -p "$SELKIES_JS_SOCKET_PATH" "$BENCH/logs"\ntest -f "$BENCH/web-root/index.html"\n# Keep the system Openbox settings, with a rule for the game\'s Wine window.\n# EWMH fullscreen removes both the title bar and tint2\'s reserved screen area.\npython3 - "$BENCH/direct-openbox-rc.xml" <<\'PY\'\nfrom pathlib import Path\nimport sys\n\nsource = Path(\'/etc/xdg/openbox/rc.xml\').read_text(encoding=\'utf-8\')\nrule = \'\'\'<applications>\n  <application class="wolfneworder_x64.exe">\n    <decor>no</decor>\n    <fullscreen>yes</fullscreen>\n    <focus>yes</focus>\n  </application>\'\'\'\nif source.count(\'<applications>\') != 1:\n    raise SystemExit(\'Cannot locate Openbox application rules\')\nPath(sys.argv[1]).write_text(source.replace(\'<applications>\', rule, 1), encoding=\'utf-8\')\nPY\nstart_desktop_program() {\n    local label="$1"\n    shift\n    local pidfile="$BENCH/direct-${label}.pid"\n    if [ -f "$pidfile" ] && kill -0 "$(cat "$pidfile")" 2>/dev/null; then\n        return\n    fi\n    runuser -u gamer -- env DISPLAY=:1 HOME=/home/gamer \\\n        PULSE_SERVER="$PULSE_SERVER" SDL_AUDIODRIVER=pulse \\\n        CUDA_VISIBLE_DEVICES=0 GAMELAB_DIRECT_XORG=1 \\\n        SELKIES_JS_SOCKET_PATH="$SELKIES_JS_SOCKET_PATH" \\\n        PYTHONPATH=/root/.local/lib/python3.13/site-packages:/usr/local/lib/python3.13/dist-packages \\\n        "$@" > "$BENCH/logs/direct-${label}.log" 2>&1 < /dev/null &\n    echo $! > "$pidfile"\n}\nstart_desktop() {\n    chmod 666 "$SELKIES_JS_SOCKET_PATH"/*.sock 2>/dev/null || true\n    start_desktop_program openbox openbox --config-file "$BENCH/direct-openbox-rc.xml"\n    if command -v tint2 >/dev/null; then\n        start_desktop_program tint2 tint2 -c "$BENCH/tint2rc"\n    fi\n    start_desktop_program dashboard python3 "$BENCH/dev_launcher.py"\n    start_desktop_program legendary python3 "$BENCH/legendary_gui.py"\n}\nif curl -fsS "http://127.0.0.1:${PORT}/api/health" >/dev/null 2>&1; then\n    start_desktop\n    echo "Direct desktop already streaming on port $PORT"\n    exit 0\nfi\n# This process runs as root because Xorg rejected the gamer\'s MIT-SHM attach.\n# Disable command execution and transfers on this privileged stream.\nnohup env SELKIES_ENCODER_DEVICE="$ENCODER_DEVICE" SELKIES_ENCODER=h264enc \\\n    python3 -m selkies --port="$PORT" --enable-basic-auth=false \\\n    --command-enabled=false --file-transfers=none --printing-enabled=false \\\n    --web-root="$BENCH/web-root" --ui-title=\'7BZ Dev Cloud Game\' \\\n    --ui-show-sidebar=true --ui-show-logo=false \\\n    --manual-resolution=true --manual-width="$WIDTH" --manual-height="$HEIGHT" \\\n    > "$BENCH/logs/direct-stream.log" 2>&1 < /dev/null &\necho $! > "$BENCH/direct-stream.pid"\nfor attempt in $(seq 1 20); do\n    if curl -fsS "http://127.0.0.1:${PORT}/api/health" >/dev/null 2>&1; then\n        start_desktop\n        echo "Direct NVIDIA stream ready on port $PORT"\n        exit 0\n    fi\n    sleep 0.5\ndone\necho "Direct stream did not start; see $BENCH/logs/direct-stream.log" >&2\nexit 1\n', encoding='utf-8')
(BENCH / 'bench_runtime.py').write_text('"""Sample a live game session without confusing stream FPS with game FPS.\n\nRun after entering repeatable gameplay. Save one JSON report per configuration.\nThe HUD\'s stream FPS and NVIDIA encoder usage are deliberately not game FPS.\n"""\nimport argparse\nfrom datetime import datetime, timezone\nimport json\nimport os\nfrom pathlib import Path\nimport statistics\nimport subprocess\nimport time\n\n\ndef cpu_stat():\n    path = Path("/sys/fs/cgroup/cpu.stat")\n    try:\n        return {key: int(value) for key, value in\n                (line.split() for line in path.read_text().splitlines())}\n    except (OSError, ValueError):\n        return {}\n\n\ndef gpus():\n    try:\n        output = subprocess.check_output([\n            "nvidia-smi", "--query-gpu=index,utilization.gpu,utilization.encoder,memory.used",\n            "--format=csv,noheader,nounits"], text=True, timeout=5,\n            stderr=subprocess.DEVNULL)\n    except (OSError, subprocess.SubprocessError):\n        return []\n    result = []\n    for line in output.splitlines():\n        fields = [part.strip() for part in line.split(",")]\n        if len(fields) != 4:\n            continue\n        try:\n            result.append(dict(index=int(fields[0]), gpu_percent=int(fields[1]),\n                               encoder_percent=int(fields[2]), memory_mib=int(fields[3])))\n        except ValueError:\n            continue\n    return result\n\n\ndef summarize(samples):\n    if not samples:\n        return None\n    return dict(mean=round(statistics.mean(samples), 2),\n                maximum=max(samples))\n\n\ndef benchmark(seconds, interval):\n    start = time.monotonic()\n    previous_time = start\n    previous_cpu = cpu_stat()\n    cpu_cores = []\n    throttled_ms_per_second = []\n    gpu_samples = {}\n    while True:\n        remaining = seconds - (time.monotonic() - start)\n        if remaining <= 0:\n            break\n        time.sleep(min(interval, remaining))\n        now = time.monotonic()\n        current_cpu = cpu_stat()\n        elapsed = now - previous_time\n        if "usage_usec" in current_cpu and "usage_usec" in previous_cpu:\n            cpu_cores.append((current_cpu["usage_usec"] - previous_cpu["usage_usec"])\n                             / (elapsed * 1_000_000))\n        if "throttled_usec" in current_cpu and "throttled_usec" in previous_cpu:\n            throttled_ms_per_second.append(\n                (current_cpu["throttled_usec"] - previous_cpu["throttled_usec"])\n                / (elapsed * 1000))\n        previous_cpu, previous_time = current_cpu, now\n        for gpu in gpus():\n            bucket = gpu_samples.setdefault(gpu["index"], dict(gpu_percent=[],\n                                    encoder_percent=[], memory_mib=[]))\n            for metric in ("gpu_percent", "encoder_percent", "memory_mib"):\n                bucket[metric].append(gpu[metric])\n    return dict(cpu_cores=summarize(cpu_cores),\n                cpu_throttled_ms_per_second=summarize(throttled_ms_per_second),\n                gpus={index: {name: summarize(values) for name, values in metrics.items()}\n                      for index, metrics in gpu_samples.items()})\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--label", required=True, help="game/backend/settings for this run")\n    parser.add_argument("--seconds", type=int, default=30)\n    parser.add_argument("--interval", type=float, default=1)\n    parser.add_argument("--game-fps", type=float,\n                        help="measured in-game FPS, never Selkies stream FPS")\n    parser.add_argument("--input-latency-ms", type=float,\n                        help="client HUD latency measured during this run")\n    parser.add_argument("--output", type=Path)\n    args = parser.parse_args()\n    if args.seconds < 1 or args.interval <= 0:\n        parser.error("seconds and interval must be positive")\n    report = dict(timestamp=datetime.now(timezone.utc).isoformat(), label=args.label,\n                  duration_seconds=args.seconds,\n                  graphics_backend=os.environ.get("GAMELAB_GRAPHICS_BACKEND", "auto"),\n                  vgl_fps=os.environ.get("GAMELAB_VGL_FPS", "60"),\n                  websocket_queue=os.environ.get("GAMELAB_WS_QUEUE", "12"),\n                  game_fps=args.game_fps, input_latency_ms=args.input_latency_ms,\n                  measurements=benchmark(args.seconds, args.interval))\n    encoded = json.dumps(report, indent=2)\n    if args.output:\n        args.output.parent.mkdir(parents=True, exist_ok=True)\n        args.output.write_text(encoded + "\\n", encoding="utf-8")\n    print(encoded)\n', encoding='utf-8')
(BENCH / 'gpu_xorg.sh').write_text('#!/bin/bash\n# Optional NVIDIA 3D server for VirtualGL. Selkies keeps its Xvfb desktop.\nset -u\nBENCH=/tmp/kaggle-game-dev-bench\nREADY="$BENCH/gpu-xorg-ready"\nLOG="$BENCH/logs/gpu-xorg.log"\nrm -f "$READY"\nmkdir -p "$BENCH/logs"\n\nif [ ! -c /dev/nvidiactl ] || [ ! -c /dev/nvidia0 ] || [ ! -c /dev/nvidia-modeset ]; then\n    echo \'NVIDIA display devices were not mounted by the container host.\' | tee -a "$LOG"\n    exit 0\nfi\nif ! command -v Xorg >/dev/null || ! find /usr/lib -name nvidia_drv.so -print -quit 2>/dev/null | grep -q .; then\n    echo \'NVIDIA Xorg display driver or Xorg is missing. Host graphics/display capability is required.\' | tee -a "$LOG"\n    exit 0\nfi\n# Identical T4 renderer strings cannot identify the card. Pin this X server\n# to nvidia-smi GPU 0\'s PCI address instead of relying on Xorg probe order.\npci_address=$(nvidia-smi -i 0 --query-gpu=pci.bus_id --format=csv,noheader 2>/dev/null | head -n 1 | tr -d \'[:space:]\')\nif [[ ! "$pci_address" =~ ^([[:xdigit:]]+):([[:xdigit:]]+):([[:xdigit:]]+)\\.([0-7])$ ]]; then\n    echo "Cannot determine GPU 0 PCI bus ID: $pci_address" | tee -a "$LOG"\n    exit 0\nfi\ndomain=$((16#${BASH_REMATCH[1]}))\nbus=$((16#${BASH_REMATCH[2]}))\ndevice=$((16#${BASH_REMATCH[3]}))\nfunc_num=${BASH_REMATCH[4]}\nbus_id="PCI:${bus}:${device}:${func_num}"\nif [ "$domain" -ne 0 ]; then bus_id="PCI:${bus}@${domain}:${device}:${func_num}"; fi\nif [ -S /tmp/.X11-unix/X1 ]; then\n    if grep -Fq "BusID \\"$bus_id\\"" "$BENCH/nvidia-headless.conf" 2>/dev/null && \\\n       DISPLAY=:1 XAUTHORITY="$BENCH/nvidia-Xauthority" glxinfo -B 2>&1 | grep -Eiq \'OpenGL vendor string:.*NVIDIA\'; then\n        DISPLAY=:1 XAUTHORITY="$BENCH/nvidia-Xauthority" xhost +si:localuser:gamer >> "$LOG" 2>&1\n        touch "$READY"\n        echo \'Reusing verified NVIDIA Xorg display :1.\' | tee -a "$LOG"\n    else\n        echo \'Display :1 is occupied by an unverified X server.\' | tee -a "$LOG"\n    fi\n    exit 0\nfi\n\ncat > "$BENCH/nvidia-headless.conf" <<EOF\nSection "ServerLayout"\n    Identifier "HeadlessNVIDIA"\n    Screen 0 "Screen0"\nEndSection\nSection "Device"\n    Identifier "GPU0"\n    Driver "nvidia"\n    BusID "$bus_id"\n    Option "AllowEmptyInitialConfiguration" "true"\n    Option "UseDisplayDevice" "none"\nEndSection\nSection "Screen"\n    Identifier "Screen0"\n    Device "GPU0"\n    DefaultDepth 24\n    SubSection "Display"\n        Depth 24\n        Virtual 1280 720\n    EndSubSection\nEndSection\nEOF\n\nAUTH="$BENCH/nvidia-Xauthority"\ntouch "$AUTH"\nchmod 600 "$AUTH"\nxauth -f "$AUTH" add :1 . "$(od -An -N16 -tx1 /dev/urandom | tr -d \' \\n\')"\nXorg :1 -config "$BENCH/nvidia-headless.conf" -auth "$AUTH" -nolisten tcp -noreset >> "$LOG" 2>&1 &\nPID=$!\nfor attempt in $(seq 1 20); do\n    if [ -S /tmp/.X11-unix/X1 ]; then break; fi\n    if ! kill -0 "$PID" 2>/dev/null; then break; fi\n    sleep 0.5\ndone\nif [ ! -S /tmp/.X11-unix/X1 ]; then\n    echo \'NVIDIA Xorg failed; see gpu-xorg.log.\' | tee -a "$LOG"\n    exit 0\nfi\nDISPLAY=:1 XAUTHORITY="$AUTH" xhost +si:localuser:gamer >> "$LOG" 2>&1\nif DISPLAY=:1 XAUTHORITY="$AUTH" glxinfo -B 2>&1 | grep -Eiq \'OpenGL renderer string:.*NVIDIA\'; then\n    touch "$READY"\n    echo \'Verified NVIDIA OpenGL on display :1.\' | tee -a "$LOG"\nelse\n    echo \'Display :1 is not NVIDIA accelerated.\' | tee -a "$LOG"\nfi\n', encoding='utf-8')
(BENCH / 'probe_nvidia_userspace.sh').write_text('#!/usr/bin/env bash\n# High-Performance NVIDIA Userspace 3D Driver & Vulkan ICD Provisioner for Kaggle\n# Offloads 3D rendering to NVIDIA GPU hardware (OpenGL & Vulkan) without modifying the kernel.\nset -uo pipefail\n\nbench=/tmp/kaggle-game-dev-bench\nwork="$bench/nvidia-userspace-probe"\nmkdir -p "$work" "$work/lib64" "$work/lib32"\n\necho "=========================================================="\necho "    NVIDIA 3D HARDWARE ACCELERATION PROVISIONER           "\necho "=========================================================="\n\nif ! command -v nvidia-smi >/dev/null 2>&1; then\n    echo "Notice: nvidia-smi not found. Running in CPU-only environment."\n    exit 0\nfi\n\nversion=$(nvidia-smi --query-gpu=driver_version --format=csv,noheader 2>/dev/null | head -n 1 | tr -d \'[:space:]\')\nif [[ ! "$version" =~ ^[0-9]+(\\.[0-9]+)+$ ]]; then\n    echo "Notice: Cannot determine loaded NVIDIA driver version ($version)."\n    exit 0\nfi\n\nmajor=$(echo "$version" | cut -d. -f1)\necho "▶ Host NVIDIA Driver Version detected: $version (Major Branch: $major)"\n\ninstalled_ok=false\n\n# Tier 1: Fast APT package installation (~10 seconds)\necho "▶ Tier 1: Checking official NVIDIA 3D graphics packages for driver $major..."\nif [ "$(id -u)" -eq 0 ]; then\n    apt-get update -o DPkg::Lock::Timeout=60 -qq 2>/dev/null || true\n    for pkg in "libnvidia-gl-$major" "libnvidia-gl-${major}-server"; do\n        if apt-cache show "$pkg" >/dev/null 2>&1; then\n            echo "▶ Found matching repository package: $pkg. Installing userspace libraries..."\n            apt-get install -y --no-install-recommends \\\n                -o DPkg::Lock::Timeout=120 -o APT::Get::Assume-Yes=true \\\n                "$pkg" "${pkg}:i386" vulkan-tools libgl1-mesa-dri mesa-utils 2>/dev/null || \\\n            apt-get install -y --no-install-recommends \\\n                -o DPkg::Lock::Timeout=120 -o APT::Get::Assume-Yes=true \\\n                "$pkg" vulkan-tools libgl1-mesa-dri mesa-utils 2>/dev/null || true\n            break\n        fi\n    done\nfi\n\n# Verify if 64-bit GLX/EGL library is now present in standard paths\nif ldconfig -p | grep -q "libGLX_nvidia.so.0" || [ -f "/usr/lib/x86_64-linux-gnu/libGLX_nvidia.so.0" ]; then\n    echo "✓ NVIDIA 3D graphics libraries verified via system package manager."\n    installed_ok=true\nfi\n\n# Tier 2: Direct matching NVIDIA archive extraction if APT package was unavailable\nif [ "$installed_ok" = false ]; then\n    echo "▶ Tier 2: Fetching matching NVIDIA userspace binary archive for $version..."\n    runfile="$work/NVIDIA-Linux-x86_64-$version.run"\n    if [ ! -f "$runfile" ] || [ $(stat -c%s "$runfile" 2>/dev/null || echo 0) -lt 10000000 ]; then\n        urls=(\n            "https://us.download.nvidia.com/tesla/$version/NVIDIA-Linux-x86_64-$version.run"\n            "https://us.download.nvidia.com/XFree86/Linux-x86_64/$version/NVIDIA-Linux-x86_64-$version.run"\n            "https://download.nvidia.com/XFree86/Linux-x86_64/$version/NVIDIA-Linux-x86_64-$version.run"\n        )\n        for u in "${urls[@]}"; do\n            echo "Attempting download from: $u"\n            if curl -fL --connect-timeout 10 --max-time 120 "$u" -o "$runfile" 2>/dev/null; then\n                if [ -f "$runfile" ] && [ $(stat -c%s "$runfile") -gt 10000000 ]; then\n                    echo "✓ Downloaded matching NVIDIA installer: $(stat -c%s "$runfile") bytes"\n                    break\n                fi\n            fi\n            rm -f "$runfile"\n        done\n    fi\n\n    driver="$work/NVIDIA-Linux-x86_64-$version"\n    if [ -f "$runfile" ] && [ ! -d "$driver" ]; then\n        echo "▶ Extracting userspace graphics components from archive..."\n        (cd "$work" && sh "$runfile" --extract-only) 2>/dev/null || true\n    fi\n\n    if [ -d "$driver" ]; then\n        link_libraries() {\n            local src=$1 dst=$2\n            mkdir -p "$dst"\n            find "$src" -maxdepth 1 -type f \\( -name \'libnvidia-*.so*\' -o -name \'libGLX_nvidia.so*\' -o -name \'libEGL_nvidia.so*\' \\) -exec cp -sf {} "$dst/" \\; 2>/dev/null || true\n        }\n        link_libraries "$driver" "$work/lib64"\n        [ -d "$driver/32" ] && link_libraries "$driver/32" "$work/lib32"\n\n        for libdir in "$work/lib64" "$work/lib32"; do\n            for name in libGLX_nvidia libEGL_nvidia; do\n                vlib=$(find "$libdir" -name "${name}.so.*" 2>/dev/null | head -n1)\n                [ -n "$vlib" ] && ln -sfn "$(basename "$vlib")" "$libdir/${name}.so.0" 2>/dev/null || true\n            done\n        done\n        installed_ok=true\n    fi\nfi\n\n# Step 3: Register GLVND and Vulkan ICD manifests\nmkdir -p /usr/share/glvnd/egl_vendor.d /etc/vulkan/icd.d /usr/share/vulkan/icd.d "$work"\n\ncat > "$work/10_nvidia.json" <<\'EOF\'\n{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}\nEOF\n\ncat > "$work/nvidia_icd.json" <<\'EOF\'\n{"file_format_version":"1.0.0","ICD":{"library_path":"libGLX_nvidia.so.0","api_version":"1.3.0"}}\nEOF\n\ncat > "$work/nvidia_egl_icd.json" <<\'EOF\'\n{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0","api_version":"1.3.0"}}\nEOF\n\ncp -f "$work/10_nvidia.json" /usr/share/glvnd/egl_vendor.d/10_nvidia.json 2>/dev/null || true\ncp -f "$work/nvidia_icd.json" /etc/vulkan/icd.d/nvidia_icd.json 2>/dev/null || true\ncp -f "$work/nvidia_icd.json" /usr/share/vulkan/icd.d/nvidia_icd.json 2>/dev/null || true\n\n# Step 4: Register libraries with dynamic linker\nif [ -d "$work/lib64" ] && [ "$(ls -A "$work/lib64" 2>/dev/null)" ]; then\n    echo "$work/lib64" > /etc/ld.so.conf.d/nvidia-gpu.conf\n    [ -d "$work/lib32" ] && [ "$(ls -A "$work/lib32" 2>/dev/null)" ] && echo "$work/lib32" >> /etc/ld.so.conf.d/nvidia-gpu.conf\nfi\nldconfig 2>/dev/null || true\n\n# Step 5: Write environment exports\n{\n    printf \'export __EGL_VENDOR_LIBRARY_FILENAMES="/usr/share/glvnd/egl_vendor.d/10_nvidia.json"\\n\'\n    printf \'export VK_DRIVER_FILES="/etc/vulkan/icd.d/nvidia_icd.json"\\n\'\n    printf \'export VK_ICD_FILENAMES="/etc/vulkan/icd.d/nvidia_icd.json"\\n\'\n    if [ -d "$work/lib64" ] && [ "$(ls -A "$work/lib64" 2>/dev/null)" ]; then\n        printf \'export LD_LIBRARY_PATH="%s:%s${LD_LIBRARY_PATH:+:$LD_LIBRARY_PATH}"\\n\' "$work/lib64" "$work/lib32"\n        printf \'export GAMELAB_NVIDIA_LIB32="%s"\\n\' "$work/lib32"\n    fi\n} > "$work/env.sh"\n\necho "✓ NVIDIA userspace graphics bridge successfully configured."\necho "  Vulkan ICD: /etc/vulkan/icd.d/nvidia_icd.json"\necho "  EGL Vendor: /usr/share/glvnd/egl_vendor.d/10_nvidia.json"\n', encoding='utf-8')

legendary_sh = """#!/bin/bash
export DISPLAY="${DISPLAY:-:0}"
export PATH="/usr/games:/usr/local/bin:/usr/bin:$PATH"
export HOME="/home/gamer"
mkdir -p /home/gamer/logs
LOG="/home/gamer/logs/legendary_gui.log"

for w in /home/gamer/.config/heroic/tools/wine/*/bin/wine; do
    if [ -f "$w" ]; then
        if [ ! -f /usr/bin/wine ]; then
            ln -sf "$w" /usr/bin/wine 2>/dev/null || true
            ln -sf "$(dirname "$w")/wineserver" /usr/bin/wineserver 2>/dev/null || true
            ln -sf "$(dirname "$w")/wineboot" /usr/bin/wineboot 2>/dev/null || true
        fi
        if [ ! -f /usr/local/bin/wine ]; then
            ln -sf "$w" /usr/local/bin/wine 2>/dev/null || true
        fi
        break
    fi
done

if python3 -c "import tkinter" >/dev/null 2>&1; then
    exec python3 /tmp/kaggle-game-dev-bench/legendary_gui.py >> "$LOG" 2>&1
else
    if [ "$(id -u)" -eq 0 ]; then
        apt-get update -qq && apt-get install -y -qq python3-tk >> "$LOG" 2>&1 || true
        if python3 -c "import tkinter" >/dev/null 2>&1; then
            exec python3 /tmp/kaggle-game-dev-bench/legendary_gui.py >> "$LOG" 2>&1
        fi
    fi
    exec xterm -geometry 105x32 -title "Legendary Epic Games Manager" -e /bin/bash /tmp/kaggle-game-dev-bench/legendary_menu.sh >> "$LOG" 2>&1
fi
"""
(BENCH / 'launch_legendary.sh').write_text(legendary_sh, encoding='utf-8')

# Auto-Optimizer script and launch wrapper
(BENCH / 'auto_optimize.py').write_text('"""Legacy desktop quality tuner for 7BZ Dev Cloud Game."""\nimport json\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport threading\nimport time\nimport tkinter as tk\nfrom tkinter import ttk, messagebox\nimport urllib.request\n\nHOME = Path(os.environ.get(\'KAGGLE_GAMER_HOME\', \'/home/gamer\'))\nBENCH = Path(\'/tmp/kaggle-game-dev-bench\')\nWINE_ENV_FILE = HOME / \'.config/gamelab_wine_env.sh\'\nSETTINGS_FILE = HOME / \'.config/gamelab_optimize.json\'\n\n# ---------------------------------------------------------------------------\n# Quality presets\n# ---------------------------------------------------------------------------\nPRESETS = {\n    \'low\': {\n        \'label\': \'📶  Low\',\n        \'resolution\': \'854x480\',\n        \'wine_csmt\': False,\n        \'wine_extra\': \'renderer=gl\',\n        \'desc\': \'Best for slow or mobile connections (< 5 Mbps).\\nLower resolution but very smooth streaming.\',\n    },\n    \'medium\': {\n        \'label\': \'📶📶  Medium\',\n        \'resolution\': \'1024x576\',\n        \'wine_csmt\': False,\n        \'wine_extra\': \'renderer=gl\',\n        \'desc\': \'Balanced quality for average connections (5–15 Mbps).\\nGood balance between clarity and smoothness.\',\n    },\n    \'high\': {\n        \'label\': \'📶📶📶  High\',\n        \'resolution\': \'1280x720\',\n        \'wine_csmt\': False,\n        \'wine_extra\': \'\',\n        \'desc\': \'Full 720p HD at 60 FPS (15–30 Mbps).\\nRecommended for broadband connections.\',\n    },\n    \'ultra\': {\n        \'label\': \'📶📶📶📶  Ultra\',\n        \'resolution\': \'1280x720\',\n        \'wine_csmt\': True,\n        \'wine_extra\': \'\',\n        \'desc\': \'Maximum quality with all renderer features (30+ Mbps).\\nRequires fast, stable connection.\',\n    },\n}\nPRESET_ORDER = [\'low\', \'medium\', \'high\', \'ultra\']\n\n# ---------------------------------------------------------------------------\n# Keyboard Lock JavaScript (injected into Selkies HTML)\n# ---------------------------------------------------------------------------\nKEYBOARD_LOCK_JS = """\n<!-- 7BZ Dev Cloud Game: Keyboard Lock – prevents Esc from exiting browser fullscreen -->\n<script>\n(function(){\n  var _kl=false;\n  function _lock(){\n    if(document.fullscreenElement && navigator.keyboard && navigator.keyboard.lock){\n      navigator.keyboard.lock([\'Escape\']).then(function(){_kl=true;\n        console.log(\'[GameLab] Keyboard locked – Esc captured for gaming\');\n      }).catch(function(e){console.log(\'[GameLab] Keyboard lock:\',e.message);});\n    }\n  }\n  document.addEventListener(\'fullscreenchange\',function(){\n    if(document.fullscreenElement){_lock();}\n    else if(_kl && navigator.keyboard && navigator.keyboard.unlock){navigator.keyboard.unlock();_kl=false;}\n  });\n  _lock();\n})();\n</script>\n"""\n\n# ---------------------------------------------------------------------------\n# Helpers\n# ---------------------------------------------------------------------------\n\ndef get_current_resolution():\n    try:\n        r = subprocess.run([\'xrandr\', \'--query\'], capture_output=True, text=True, timeout=5)\n        for line in r.stdout.splitlines():\n            if \'*\' in line:\n                return line.strip().split()[0]\n    except Exception:\n        pass\n    return \'1280x720\'\n\n\ndef set_resolution(res):\n    try:\n        w, h = res.split(\'x\')\n        # Try to set directly first\n        r = subprocess.run([\'xrandr\', \'-s\', res], capture_output=True, text=True, timeout=5)\n        if r.returncode != 0:\n            # Create the mode and try again\n            cvt = subprocess.run([\'cvt\', w, h], capture_output=True, text=True, timeout=5)\n            for line in cvt.stdout.splitlines():\n                if \'Modeline\' in line:\n                    parts = line.split(\'"\')\n                    if len(parts) >= 3:\n                        mode_name = parts[1]\n                        mode_args = parts[2].strip()\n                        subprocess.run([\'xrandr\', \'--newmode\', mode_name] + mode_args.split(), check=False, timeout=5)\n                        for output in (\'screen\', \'SCREEN\', \'default\', \'VGA-0\', \'Virtual-1\'):\n                            subprocess.run([\'xrandr\', \'--addmode\', output, mode_name], check=False, timeout=5)\n                        subprocess.run([\'xrandr\', \'-s\', mode_name], check=False, timeout=5)\n        return True\n    except Exception as e:\n        print(f\'Resolution change error: {e}\')\n        return False\n\n\ndef write_wine_env(preset_key):\n    p = PRESETS[preset_key]\n    csmt = \'1\' if p[\'wine_csmt\'] else \'0\'\n    d3d = f"csmt={csmt}"\n    if p.get(\'wine_extra\'):\n        d3d += \';\' + p[\'wine_extra\']\n\n    content = f"""#!/bin/bash\n# Auto-generated by 7BZ Auto-Optimizer – preset: {preset_key}\nexport WINE_D3D_CONFIG="{d3d}"\nexport MESA_GL_VERSION_OVERRIDE="4.3"\nexport __GL_SYNC_TO_VBLANK=0\nexport vblank_mode=0\nexport WINEDLLOVERRIDES="d3d11,dxgi,d3d9,d3d10core=b"\n"""\n    WINE_ENV_FILE.parent.mkdir(parents=True, exist_ok=True)\n    WINE_ENV_FILE.write_text(content, encoding=\'utf-8\')\n\n\ndef save_settings(preset_key):\n    SETTINGS_FILE.parent.mkdir(parents=True, exist_ok=True)\n    SETTINGS_FILE.write_text(json.dumps({\'preset\': preset_key, \'ts\': time.time()}, indent=2), encoding=\'utf-8\')\n\n\ndef load_settings():\n    try:\n        return json.loads(SETTINGS_FILE.read_text(encoding=\'utf-8\')).get(\'preset\', \'high\')\n    except Exception:\n        return \'high\'\n\n\ndef inject_keyboard_lock():\n    """Inject keyboard lock JS into every Selkies HTML file found on disk."""\n    marker = \'7BZ Dev Cloud Game: Keyboard Lock\'\n    patched = []\n    search_roots = [Path(\'/usr/local/lib\'), Path(\'/usr/lib\')]\n    for root in search_roots:\n        if not root.is_dir():\n            continue\n        for html in root.rglob(\'index.html\'):\n            if \'selkies\' not in str(html).lower():\n                continue\n            try:\n                text = html.read_text(encoding=\'utf-8\')\n                if marker in text:\n                    patched.append(f\'{html} (already patched)\')\n                    continue\n                if \'</head>\' in text:\n                    html.write_text(text.replace(\'</head>\', KEYBOARD_LOCK_JS + \'\\n</head>\'), encoding=\'utf-8\')\n                    patched.append(str(html))\n                elif \'</body>\' in text:\n                    html.write_text(text.replace(\'</body>\', KEYBOARD_LOCK_JS + \'\\n</body>\'), encoding=\'utf-8\')\n                    patched.append(str(html))\n            except Exception as e:\n                patched.append(f\'{html} (error: {e})\')\n    return patched\n\n\ndef test_bandwidth():\n    """Quick bandwidth measurement (server-side egress estimate)."""\n    urls = [\n        \'https://speed.cloudflare.com/__down?bytes=2000000\',\n        \'https://www.google.com/images/branding/googlelogo/2x/googlelogo_color_272x92dp.png\',\n    ]\n    for url in urls:\n        try:\n            req = urllib.request.Request(url, headers={\'User-Agent\': \'GameLab/1.0\'})\n            t0 = time.time()\n            with urllib.request.urlopen(req, timeout=8) as resp:\n                data = resp.read()\n            elapsed = time.time() - t0\n            if elapsed > 0.01:\n                return (len(data) * 8) / (elapsed * 1_000_000)\n        except Exception:\n            continue\n    return 0.0\n\n\ndef recommend_preset(speed_mbps):\n    if speed_mbps >= 30:\n        return \'ultra\'\n    elif speed_mbps >= 15:\n        return \'high\'\n    elif speed_mbps >= 5:\n        return \'medium\'\n    return \'low\'\n\n\n# ---------------------------------------------------------------------------\n# GUI\n# ---------------------------------------------------------------------------\n\nclass AutoOptimizer(tk.Tk):\n    def __init__(self):\n        super().__init__()\n        self.title("7BZ Dev Cloud Game · Auto-Optimizer")\n        self.geometry("720x560")\n        self.minsize(640, 480)\n\n        # Dark theme\n        self.c_bg = "#141923"\n        self.c_panel = "#1b2230"\n        self.c_card = "#222c3e"\n        self.c_fg = "#f0f4fc"\n        self.c_muted = "#8fa0bd"\n        self.c_accent = "#3498db"\n        self.c_green = "#2ecc71"\n        self.c_red = "#e74c3c"\n        self.c_yellow = "#f39c12"\n        self.c_orange = "#e67e22"\n\n        self.configure(bg=self.c_bg)\n        self.current_preset = load_settings()\n\n        self.build_header()\n        self.build_status_panel()\n        self.build_presets_panel()\n        self.build_actions_panel()\n        self.build_footer()\n\n        self.refresh_status()\n\n    # ---- Header ----\n    def build_header(self):\n        hdr = tk.Frame(self, bg=self.c_panel)\n        hdr.pack(fill=\'x\', padx=10, pady=(8, 4))\n\n        tk.Label(hdr, text=" ⚡ AUTO-OPTIMIZE ", bg=self.c_orange, fg=\'#fff\',\n                 font=(\'DejaVu Sans\', 10, \'bold\'), padx=6, pady=2).pack(side=\'left\', padx=(10, 12), pady=8)\n        tk.Label(hdr, text="Adaptive Streaming & Rendering Quality", bg=self.c_panel, fg=self.c_fg,\n                 font=(\'DejaVu Sans\', 13, \'bold\')).pack(side=\'left\', pady=8)\n\n    # ---- Status panel ----\n    def build_status_panel(self):\n        frm = tk.Frame(self, bg=self.c_card, padx=14, pady=10)\n        frm.pack(fill=\'x\', padx=10, pady=6)\n\n        tk.Label(frm, text="Current Status", bg=self.c_card, fg=\'#fff\',\n                 font=(\'DejaVu Sans\', 11, \'bold\')).grid(row=0, column=0, columnspan=4, sticky=\'w\', pady=(0, 6))\n\n        self.lbl_resolution = self._stat(frm, 1, 0, "Resolution:")\n        self.lbl_preset = self._stat(frm, 1, 2, "Active Preset:")\n        self.lbl_wine = self._stat(frm, 2, 0, "Wine Renderer:")\n        self.lbl_keyboard = self._stat(frm, 2, 2, "Esc Key Lock:")\n\n    def _stat(self, parent, row, col, label):\n        tk.Label(parent, text=label, bg=self.c_card, fg=self.c_muted,\n                 font=(\'DejaVu Sans\', 9, \'bold\')).grid(row=row, column=col, sticky=\'w\', padx=(0, 6), pady=2)\n        val = tk.Label(parent, text="...", bg=self.c_card, fg=self.c_fg, font=(\'DejaVu Sans\', 9))\n        val.grid(row=row, column=col + 1, sticky=\'w\', padx=(0, 30), pady=2)\n        return val\n\n    # ---- Presets panel ----\n    def build_presets_panel(self):\n        outer = tk.Frame(self, bg=self.c_bg)\n        outer.pack(fill=\'both\', expand=True, padx=10, pady=6)\n\n        tk.Label(outer, text="Select Quality Preset:", bg=self.c_bg, fg=\'#fff\',\n                 font=(\'DejaVu Sans\', 11, \'bold\')).pack(anchor=\'w\', pady=(0, 6))\n\n        self.preset_btns = {}\n        for key in PRESET_ORDER:\n            p = PRESETS[key]\n            card = tk.Frame(outer, bg=self.c_card, padx=12, pady=8, cursor=\'hand2\',\n                            highlightthickness=2, highlightbackground=self.c_card)\n            card.pack(fill=\'x\', pady=3)\n\n            lbl = tk.Label(card, text=p[\'label\'], bg=self.c_card, fg=self.c_fg,\n                           font=(\'DejaVu Sans\', 11, \'bold\'), anchor=\'w\')\n            lbl.pack(side=\'left\', padx=(0, 12))\n\n            desc_short = p[\'desc\'].split(\'\\n\')[0]\n            tk.Label(card, text=desc_short, bg=self.c_card, fg=self.c_muted,\n                     font=(\'DejaVu Sans\', 9), anchor=\'w\').pack(side=\'left\', fill=\'x\', expand=True)\n\n            res_lbl = tk.Label(card, text=p[\'resolution\'], bg=self.c_card, fg=self.c_accent,\n                               font=(\'DejaVu Sans Mono\', 10, \'bold\'))\n            res_lbl.pack(side=\'right\', padx=8)\n\n            self.preset_btns[key] = card\n            for widget in (card, lbl, res_lbl):\n                widget.bind(\'<Button-1>\', lambda e, k=key: self.select_preset(k))\n\n    def select_preset(self, key):\n        self.current_preset = key\n        for k, card in self.preset_btns.items():\n            if k == key:\n                card.configure(highlightbackground=self.c_accent, bg=\'#1a3a52\')\n                for w in card.winfo_children():\n                    w.configure(bg=\'#1a3a52\')\n            else:\n                card.configure(highlightbackground=self.c_card, bg=self.c_card)\n                for w in card.winfo_children():\n                    w.configure(bg=self.c_card)\n\n    # ---- Actions panel ----\n    def build_actions_panel(self):\n        frm = tk.Frame(self, bg=self.c_bg)\n        frm.pack(fill=\'x\', padx=10, pady=8)\n\n        btn_auto = tk.Button(frm, text=" 🚀  AUTO-DETECT & APPLY ",\n                             bg=self.c_green, fg=\'#fff\', font=(\'DejaVu Sans\', 11, \'bold\'),\n                             relief=\'flat\', padx=16, pady=8, cursor=\'hand2\',\n                             activebackground=\'#27ae60\', command=self.auto_detect_and_apply)\n        btn_auto.pack(side=\'left\', padx=(0, 10))\n\n        btn_apply = tk.Button(frm, text=" ✅  Apply Selected Preset ",\n                              bg=self.c_accent, fg=\'#fff\', font=(\'DejaVu Sans\', 10, \'bold\'),\n                              relief=\'flat\', padx=14, pady=8, cursor=\'hand2\',\n                              activebackground=\'#2980b9\', command=self.apply_preset)\n        btn_apply.pack(side=\'left\', padx=5)\n\n        btn_esc = tk.Button(frm, text=" 🔒  Fix Esc Key ",\n                            bg=self.c_card, fg=self.c_fg, font=(\'DejaVu Sans\', 10),\n                            relief=\'flat\', padx=12, pady=8, cursor=\'hand2\',\n                            command=self.fix_esc_key)\n        btn_esc.pack(side=\'left\', padx=5)\n\n        self.lbl_action_status = tk.Label(frm, text="", bg=self.c_bg, fg=self.c_yellow,\n                                          font=(\'DejaVu Sans\', 9))\n        self.lbl_action_status.pack(side=\'right\', padx=10)\n\n    # ---- Footer ----\n    def build_footer(self):\n        bar = tk.Frame(self, bg=self.c_panel, height=28)\n        bar.pack(fill=\'x\', side=\'bottom\', padx=10, pady=(0, 6))\n        tk.Label(bar, text="💡 Tip: Press AUTO-DETECT for the best settings, or pick a preset and click Apply.",\n                 bg=self.c_panel, fg=self.c_muted, font=(\'DejaVu Sans\', 9)).pack(padx=10, pady=5)\n\n    # ---- Logic ----\n\n    def refresh_status(self):\n        cur_res = get_current_resolution()\n        self.lbl_resolution.config(text=cur_res)\n        self.lbl_preset.config(text=PRESETS.get(self.current_preset, {}).get(\'label\', self.current_preset))\n\n        # Wine renderer info\n        env_csmt = \'Enabled\' if PRESETS.get(self.current_preset, {}).get(\'wine_csmt\') else \'Disabled\'\n        self.lbl_wine.config(text=f"WineD3D (OpenGL) · CSMT {env_csmt}")\n\n        # Keyboard lock status\n        kl_status = "Not checked"\n        try:\n            for root in [Path(\'/usr/local/lib\'), Path(\'/usr/lib\')]:\n                for html in root.rglob(\'index.html\'):\n                    if \'selkies\' in str(html).lower():\n                        if \'7BZ Dev Cloud Game: Keyboard Lock\' in html.read_text(encoding=\'utf-8\'):\n                            kl_status = "✅ Active (Esc captured)"\n                        else:\n                            kl_status = "❌ Not injected"\n                        break\n        except Exception:\n            kl_status = "Unknown"\n        self.lbl_keyboard.config(text=kl_status)\n\n        self.select_preset(self.current_preset)\n\n    def apply_preset(self):\n        self.lbl_action_status.config(text="Applying...", fg=self.c_yellow)\n        self.update()\n        threading.Thread(target=self._apply_worker, daemon=True).start()\n\n    def _apply_worker(self):\n        key = self.current_preset\n        p = PRESETS[key]\n        msgs = []\n\n        # 1. Resolution\n        if set_resolution(p[\'resolution\']):\n            msgs.append(f"Resolution → {p[\'resolution\']}")\n        else:\n            msgs.append(f"Resolution change to {p[\'resolution\']} failed (may already be set)")\n\n        # 2. Wine env\n        write_wine_env(key)\n        msgs.append(f"Wine config → csmt={\'on\' if p[\'wine_csmt\'] else \'off\'}")\n\n        # 3. Save\n        save_settings(key)\n        msgs.append(f"Preset saved: {key}")\n\n        summary = " · ".join(msgs)\n        self.after(0, lambda: self.lbl_action_status.config(text=f"✅ {summary}", fg=self.c_green))\n        self.after(0, self.refresh_status)\n\n    def auto_detect_and_apply(self):\n        self.lbl_action_status.config(text="🔍 Testing bandwidth...", fg=self.c_yellow)\n        self.update()\n        threading.Thread(target=self._auto_worker, daemon=True).start()\n\n    def _auto_worker(self):\n        speed = test_bandwidth()\n        recommended = recommend_preset(speed)\n\n        self.current_preset = recommended\n        self.after(0, lambda: self.select_preset(recommended))\n        self.after(0, lambda: self.lbl_action_status.config(\n            text=f"📶 Speed: {speed:.1f} Mbps → Recommended: {PRESETS[recommended][\'label\']}",\n            fg=self.c_accent))\n        self.after(100, lambda: None)  # small delay\n\n        # Apply\n        p = PRESETS[recommended]\n        set_resolution(p[\'resolution\'])\n        write_wine_env(recommended)\n        save_settings(recommended)\n\n        # Also inject keyboard lock\n        patched = inject_keyboard_lock()\n\n        self.after(200, lambda: self.lbl_action_status.config(\n            text=f"✅ Applied: {PRESETS[recommended][\'label\']} ({p[\'resolution\']}) · {speed:.1f} Mbps" +\n                 (" · Esc Lock ✅" if patched else ""),\n            fg=self.c_green))\n        self.after(200, self.refresh_status)\n\n    def fix_esc_key(self):\n        self.lbl_action_status.config(text="🔒 Injecting keyboard lock...", fg=self.c_yellow)\n        self.update()\n\n        def _worker():\n            patched = inject_keyboard_lock()\n            if patched:\n                msg = f"✅ Keyboard lock injected into {len(patched)} file(s). Reload your browser tab (F5) to activate."\n            else:\n                msg = "⚠️ Could not find Selkies HTML files. Use the browser bookmarklet instead."\n            self.after(0, lambda: self.lbl_action_status.config(text=msg, fg=self.c_green if patched else self.c_yellow))\n            self.after(0, self.refresh_status)\n\n            if not patched:\n                self.after(500, self.show_bookmarklet_fallback)\n\n        threading.Thread(target=_worker, daemon=True).start()\n\n    def show_bookmarklet_fallback(self):\n        dlg = tk.Toplevel(self)\n        dlg.title("🔒 Esc Key Fix – Browser Bookmarklet")\n        dlg.geometry("540x260")\n        dlg.configure(bg=self.c_bg)\n        dlg.transient(self)\n\n        tk.Label(dlg, text="Alternative: Browser Bookmarklet", bg=self.c_bg, fg=\'#fff\',\n                 font=(\'DejaVu Sans\', 12, \'bold\')).pack(pady=(12, 8))\n\n        tk.Label(dlg, text=(\n            "If the automatic injection didn\'t work, you can fix the Esc key\\n"\n            "by creating a browser bookmark with this code.\\n"\n            "Click the bookmark after entering fullscreen (F11)."\n        ), bg=self.c_bg, fg=self.c_muted, font=(\'DejaVu Sans\', 9), justify=\'left\').pack(padx=14)\n\n        code = "javascript:void(navigator.keyboard&&navigator.keyboard.lock([\'Escape\']).then(()=>document.title+=\' 🔒\').catch(e=>alert(\'Keyboard lock: \'+e)))"\n        ent = tk.Entry(dlg, bg=\'#1a202c\', fg=\'#0f0\', insertbackground=\'#fff\',\n                       font=(\'DejaVu Sans Mono\', 8), relief=\'flat\')\n        ent.pack(fill=\'x\', padx=14, pady=8, ipady=4)\n        ent.insert(0, code)\n        ent.select_range(0, tk.END)\n\n        def _copy():\n            self.clipboard_clear()\n            self.clipboard_append(code)\n            btn_copy.config(text="✓ Copied!")\n\n        btn_copy = tk.Button(dlg, text=" 📋 Copy to Clipboard ", bg=self.c_card, fg=self.c_fg,\n                             relief=\'flat\', cursor=\'hand2\', command=_copy)\n        btn_copy.pack(pady=6)\n\n\nif __name__ == "__main__":\n    app = AutoOptimizer()\n    app.mainloop()\n', encoding='utf-8')

optimizer_sh = """#!/bin/bash
export DISPLAY="${DISPLAY:-:0}"
export PATH="/usr/games:/usr/local/bin:/usr/bin:$PATH"
export HOME="/home/gamer"
mkdir -p /home/gamer/logs
LOG="/home/gamer/logs/auto_optimize.log"

if ! python3 -c "import tkinter" >/dev/null 2>&1; then
    apt-get update -qq && apt-get install -y -qq python3-tk >> "$LOG" 2>&1 || true
fi

exec python3 /tmp/kaggle-game-dev-bench/auto_optimize.py >> "$LOG" 2>&1
"""
(BENCH / 'launch_optimizer.sh').write_text(optimizer_sh, encoding='utf-8')

# Make all shell scripts executable
for p in BENCH.glob('*.sh'):
    p.chmod(0o755)

print('Launcher scripts, storefront wrappers, and desktop configuration saved successfully.', flush=True)


In [ ]:
from pathlib import Path
import json, os, shutil, signal, subprocess, sys, time, urllib.request

BENCH = Path('/tmp/kaggle-game-dev-bench')
BUILDS = Path('/tmp/game-builds')
BUILDS.mkdir(parents=True, exist_ok=True)
SCREENSHOTS = Path('/kaggle/working/screenshots')
SCREENSHOTS.mkdir(parents=True, exist_ok=True)

# Master desktop session wrapper
wrapper = BENCH / 'start-desktop.sh'
wrapper.write_text('#!/bin/bash\n# Asymmetric Dual-GPU Allocation:\n# Force all desktop session applications, game runners, and Wine to use GPU 0.\nexport CUDA_VISIBLE_DEVICES=0\nexport DRI_PRIME=0\nexport VGL_DISPLAY=egl0\nexport MESA_GL_VERSION_OVERRIDE=4.5COMPAT\nexport MESA_GLSL_VERSION_OVERRIDE=450\n\nexport LD_PRELOAD=\'/tmp/kaggle-game-dev-bench/$LIB/selkies_input_interposer.so:/tmp/kaggle-game-dev-bench/$LIB/selkies_fake_udev.so\'\nexport SDL_AUDIODRIVER=pulse\nexport SELKIES_JS_SOCKET_PATH="/tmp"\nexport SDL_JOYSTICK_DISABLE_UDEV=1\nexport SDL_JOYSTICK_DEVICE=/dev/input/event1000:/dev/input/event1001:/dev/input/event1002:/dev/input/event1003\nexport DISPLAY="${DISPLAY:-:0}"\nexport PATH="/usr/games:/usr/local/bin:/usr/bin:$PATH"\n\n# Ensure /dev/input nodes and permissions exist\nmkdir -p /dev/input 2>/dev/null || true\nfor i in 0 1 2 3; do touch /dev/input/js$i 2>/dev/null || true; done\nfor i in 1000 1001 1002 1003; do touch /dev/input/event$i 2>/dev/null || true; done\nchmod -R 666 /dev/input 2>/dev/null || true\n\n# Continuous socket permission and mirroring monitor so gamer user can connect to gamepad sockets\n(while true; do\n    chmod 755 /run/user /run/user/0 2>/dev/null || true\n    chmod 666 /tmp/selkies_*.sock /run/user/*/selkies_*.sock 2>/dev/null || true\n    ln -sf /run/user/0/selkies_*.sock /tmp/ 2>/dev/null || true\n    ln -sf /tmp/selkies_*.sock /run/user/1000/ 2>/dev/null || true\n    sleep 2\ndone) &\n\n# Load NVIDIA userspace driver environment if available\nprobe_env=/tmp/kaggle-game-dev-bench/nvidia-userspace-probe/env.sh\nif [ -f "$probe_env" ]; then\n    # shellcheck disable=SC1090\n    . "$probe_env"\nfi\nprintf \'%s\' "${XAUTHORITY:-}" > /tmp/kaggle-game-dev-bench/xauthority-path\nprintf \'%s\' "${PULSE_SERVER:-}" > /tmp/kaggle-game-dev-bench/pulse-server-path\n\nif [ -z "${PULSE_SERVER:-}" ]; then\n    for pulse_socket in /run/user/1000/pulse/native /tmp/pulse-*/native; do\n        if [ -S "$pulse_socket" ]; then\n            export PULSE_SERVER="unix:$pulse_socket"\n            break\n        fi\n    done\nfi\n\n# Xvfb remains the Selkies capture desktop. A separate NVIDIA Xorg display is\n# used only when the host mounted the display driver and a renderer test passes.\nif [ "$(id -u)" -eq 0 ]; then\n    /bin/bash /tmp/kaggle-game-dev-bench/gpu_xorg.sh || true\nfi\n\n# Grant unprivileged gamer user access to X11 display and root packages\nxhost + 2>/dev/null || true\nchmod 755 /root 2>/dev/null || true\nchmod -R 755 /root/.local 2>/dev/null || true\nchmod -R 777 /home/gamer 2>/dev/null || true\nxset m 0 0 2>/dev/null || true\n\n# Grant unprivileged gamer user access to PulseAudio socket and runtime directory\nchmod 777 "$XDG_RUNTIME_DIR" 2>/dev/null || true\nchmod -R 777 "$XDG_RUNTIME_DIR/pulse" 2>/dev/null || true\n\ninstall -d -o gamer -g gamer -m 700 /home/gamer/.config/pulse\nif [ -f "$HOME/.config/pulse/cookie" ]; then\n    install -o gamer -g gamer -m 600 "$HOME/.config/pulse/cookie" /home/gamer/.config/pulse/cookie\nfi\n\n# Ensure /usr/bin/wine exists by symlinking Wine-GE if system wine is absent\nfor w_candidate in /home/gamer/.config/heroic/tools/wine/*/bin/wine; do\n    if [ -f "$w_candidate" ]; then\n        if [ ! -f /usr/bin/wine ]; then\n            ln -sf "$w_candidate" /usr/bin/wine 2>/dev/null || true\n            ln -sf "$(dirname "$w_candidate")/wineserver" /usr/bin/wineserver 2>/dev/null || true\n            ln -sf "$(dirname "$w_candidate")/wineboot" /usr/bin/wineboot 2>/dev/null || true\n        fi\n        if [ ! -f /usr/local/bin/wine ]; then\n            ln -sf "$w_candidate" /usr/local/bin/wine 2>/dev/null || true\n        fi\n        break\n    fi\ndone\n\n# Start Openbox window manager in background for window frames, moving & resizing\nopenbox &\n\n# Start tint2 taskbar panel in background for window switching\nif which tint2 >/dev/null 2>&1; then\n    tint2 -c /tmp/kaggle-game-dev-bench/tint2rc &\nfi\n\n# Keep the emergency HUD control plane separate from Selkies and the desktop.\ncontrol_pidfile=/tmp/kaggle-game-dev-bench/session-control.pid\nif [ -f "$control_pidfile" ]; then\n    control_pid=$(cat "$control_pidfile" 2>/dev/null || true)\n    if [ -n "$control_pid" ] && [ -r "/proc/$control_pid/cmdline" ] && \\\n       tr \'\\0\' \' \' < "/proc/$control_pid/cmdline" | grep -q \'/tmp/kaggle-game-dev-bench/session_control.py\'; then\n        kill "$control_pid" 2>/dev/null || true\n    fi\nfi\nrunuser -u gamer -- env DISPLAY="$DISPLAY" HOME=/home/gamer PATH="$PATH" \\\n    PULSE_SERVER="${PULSE_SERVER:-}" SDL_AUDIODRIVER=pulse \\\n    python3 /tmp/kaggle-game-dev-bench/session_control.py \\\n    >> /home/gamer/logs/session-control.log 2>&1 &\necho $! > "$control_pidfile"\n\n# Run the controller-friendly launcher UI as the main session process\nif [ "$(id -u)" -eq 0 ]; then\n    exec runuser -u gamer -- env -u DBUS_SESSION_BUS_ADDRESS DISPLAY="$DISPLAY" HOME=/home/gamer PATH="$PATH" \\\n        XDG_RUNTIME_DIR="${XDG_RUNTIME_DIR:-}" SDL_AUDIODRIVER=pulse \\\n        PULSE_SERVER="${PULSE_SERVER:-}" SELKIES_JS_SOCKET_PATH="${SELKIES_JS_SOCKET_PATH:-}" \\\n        GAMELAB_ALLOW_SOFTWARE="${GAMELAB_ALLOW_SOFTWARE:-0}" \\\n        GAMELAB_SOFTWARE_RESOLUTION="${GAMELAB_SOFTWARE_RESOLUTION:-854x480}" \\\n        LP_NUM_THREADS="${LP_NUM_THREADS:-}" \\\n        LD_LIBRARY_PATH="${LD_LIBRARY_PATH:-}" \\\n        __EGL_VENDOR_LIBRARY_FILENAMES="${__EGL_VENDOR_LIBRARY_FILENAMES:-}" \\\n        VK_DRIVER_FILES="${VK_DRIVER_FILES:-}" VK_ICD_FILENAMES="${VK_ICD_FILENAMES:-}" \\\n        GAMELAB_NVIDIA_LIB32="${GAMELAB_NVIDIA_LIB32:-}" \\\n        PYTHONPATH="/root/.local/lib/python3.13/site-packages:/usr/local/lib/python3.13/dist-packages:${PYTHONPATH:-}" \\\n        LD_PRELOAD="$LD_PRELOAD" SDL_JOYSTICK_DISABLE_UDEV=1 \\\n        SDL_JOYSTICK_DEVICE="$SDL_JOYSTICK_DEVICE" \\\n        bash -c "while true; do python3 /tmp/kaggle-game-dev-bench/dev_launcher.py >> /home/gamer/logs/launcher.log 2>&1 || python3 /tmp/kaggle-game-dev-bench/legendary_gui.py >> /home/gamer/logs/gui.log 2>&1 || sleep 2; done"\nfi\nexec bash -c "while true; do python3 /tmp/kaggle-game-dev-bench/dev_launcher.py >> /home/gamer/logs/launcher.log 2>&1 || python3 /tmp/kaggle-game-dev-bench/legendary_gui.py >> /home/gamer/logs/gui.log 2>&1 || sleep 2; done"\n', encoding='utf-8')
wrapper.chmod(0o755)

# Build a custom Selkies web root without changing the installed streaming core.
subprocess.run([sys.executable, str(BENCH / 'install_web_client.py')], check=True)

# Stop any previous session
pidfile = BENCH / 'selkies.pid'
if pidfile.exists():
    try:
        previous = int(pidfile.read_text())
        command = Path(f'/proc/{previous}/cmdline').read_bytes().replace(b'\0', b' ')
        if b'selkies-session' in command and b'kaggle-game-dev-bench' in command:
            os.killpg(previous, signal.SIGTERM)
            time.sleep(2)
    except (ValueError, OSError, ProcessLookupError):
        pass

env = os.environ.copy()
env['PATH'] = f"/usr/local/bin:/usr/games:/usr/bin:{env.get('PATH', '')}"

# Asymmetric Dual-GPU Pipeline:
# If two GPUs are detected (/dev/nvidia0 and /dev/nvidia1):
# GPU 1 is dedicated exclusively to Selkies desktop capture & NVENC H.264 hardware encoding.
# GPU 0 is dedicated exclusively to 3D game rendering (Vulkan/DXVK/VKD3D/VirtualGL).
if Path('/dev/nvidia1').exists() and Path('/dev/nvidia0').exists():
    print('▶ Dual-GPU environment detected: Dedicating GPU 1 to NVENC encoding and GPU 0 to 3D Game Rendering.', flush=True)
    env['CUDA_VISIBLE_DEVICES'] = '1'
    env['SELKIES_ENCODER_DEVICE'] = '1'
else:
    print('▶ Single-GPU environment detected: GPU 0 active for rendering and encoding.', flush=True)
    env['CUDA_VISIBLE_DEVICES'] = '0'
    env['SELKIES_ENCODER_DEVICE'] = '0'

queue_size = os.environ.get('GAMELAB_WS_QUEUE', '12')
if not queue_size.isdigit() or not 1 <= int(queue_size) <= 120:
    raise ValueError('GAMELAB_WS_QUEUE must be an integer from 1 to 120')
env.update(SELKIES_ENABLE_HTTPS='false', SELKIES_MODE='websockets',
           SELKIES_ALLOWED_ORIGINS='https://kkb-production.jupyter-proxy.kaggle.net',
           SELKIES_ENCODER='h264enc', SELKIES_FRAMERATE='60,30-60',
           SELKIES_VIDEO_FULLCOLOR='false|locked',
           SELKIES_VIDEO_10BIT='false|locked',
           SELKIES_BACKPRESSURE_QUEUE_SIZE=queue_size,
           SELKIES_VIDEO_BITRATE='8000,2000-15000',
           SELKIES_RATE_CONTROL_MODE='cbr', SELKIES_ENABLE_RESIZE='true',
           SELKIES_AUDIO_ENABLED='true', SELKIES_AUDIO_CHANNELS='2',
           SELKIES_AUDIO_BITRATE='128000', SELKIES_GAMEPAD_ENABLED='true',
           SELKIES_UINPUT_GAMEPAD='false',
           SELKIES_JS_SOCKET_PATH='/tmp',
           SELKIES_UI_TITLE='7BZ Dev Cloud Game',
           SELKIES_UI_SHOW_SIDEBAR='true', SELKIES_UI_SHOW_LOGO='false',
           SELKIES_MANUAL_WIDTH='1280', SELKIES_MANUAL_HEIGHT='720',
           SELKIES_WEB_ROOT=str(BENCH / 'web-root'),
           FILE_MANAGER_PATH=str(BUILDS),
           SDL_AUDIODRIVER='pulse')

# Enforce GLX extension on Xvfb in selkies package with Mesa isolation to prevent NVIDIA EGL-GBM crash
try:
    import selkies
    s_session = Path(selkies.__file__).parent / 'session.py'
    if s_session.is_file():
        s_code = s_session.read_text(encoding='utf-8')
        s_target = 'for extra in gpu + [glx, ["-extension", "GLX"]]:\n            read, write = os.pipe()\n            proc = self.spawn(["Xvfb", "-displayfd", str(write), "-auth", auth, *XVFB_ARGS, *extra],\n                              pass_fds=(write,))'
        s_target_alt = 'for extra in [glx]:\n            read, write = os.pipe()\n            proc = self.spawn(["Xvfb", "-displayfd", str(write), "-auth", auth, *XVFB_ARGS, *extra],\n                              pass_fds=(write,))'
        s_repl = ('xvfb_env = os.environ.copy()\n'
                  '        xvfb_env.update(LIBGL_ALWAYS_SOFTWARE="1", __GLX_VENDOR_LIBRARY_NAME="mesa", __EGL_VENDOR_LIBRARY_FILENAMES="", GBM_BACKENDS_PATH="/dev/null")\n'
                  '        for extra in [glx]:\n'
                  '            read, write = os.pipe()\n'
                  '            proc = self.spawn(["Xvfb", "-displayfd", str(write), "-auth", auth, *XVFB_ARGS, *extra],\n'
                  '                              pass_fds=(write,), env=xvfb_env)')
        if s_target in s_code:
            s_code = s_code.replace(s_target, s_repl)
        elif s_target_alt in s_code:
            s_code = s_code.replace(s_target_alt, s_repl)
        s_session.write_text(s_code, encoding='utf-8')
except Exception:
    pass

log = (BENCH / 'selkies.log').open('a', buffering=1)
server = subprocess.Popen(['selkies-session', '--session=' + str(wrapper),
                           '--port=8080', '--enable-basic-auth=false'],
                          stdout=log, stderr=subprocess.STDOUT,
                          stdin=subprocess.DEVNULL, env=env,
                          start_new_session=True)
pidfile.write_text(str(server.pid))

health = None
for _ in range(75):
    if server.poll() is not None:
        break
    try:
        with urllib.request.urlopen('http://127.0.0.1:8080/api/health', timeout=2) as response:
            if response.status == 200:
                health = response.read().decode(errors='replace')
                break
    except Exception:
        time.sleep(1)

if health != 'OK':
    raise RuntimeError('Selkies did not start. Log tail:\n' +
                       (BENCH / 'selkies.log').read_text(errors='replace')[-5000:])

with urllib.request.urlopen('http://127.0.0.1:8080/api/status', timeout=5) as response:
    status = json.load(response)

time.sleep(2)
for sock in Path('/tmp').glob('selkies_*.sock'):
    try: sock.chmod(0o666)
    except Exception: pass
serverlog = (BENCH / 'selkies.log').read_text(errors='replace')

# =========================================================================
# EMPIRICAL STOREFRONT GUI LAUNCH & WINDOW VERIFICATION PROBE
# =========================================================================
print('▶ Starting empirical GUI launch tests for storefronts...', flush=True)
auth_path_file = BENCH / 'xauthority-path'
probe_env = dict(os.environ, DISPLAY=':0')
if auth_path_file.exists():
    probe_env['XAUTHORITY'] = auth_path_file.read_text().strip()

def get_wm_windows():
    try:
        res = subprocess.run(['wmctrl', '-l'], capture_output=True, text=True, timeout=5,
                             env=probe_env)
        return res.stdout.splitlines() if res.returncode == 0 else []
    except Exception:
        return []

def capture_screenshot(out_path):
    try:
        result = subprocess.run(['ffmpeg', '-f', 'x11grab', '-video_size', '1280x720', '-i', ':0',
                        '-vframes', '1', '-y', str(out_path)],
                       capture_output=True, timeout=10, env=probe_env)
        if result.returncode != 0:
            print('Screenshot failed:', result.stderr.decode(errors='replace')[-350:], flush=True)
        return out_path.exists() and out_path.stat().st_size > 1000
    except Exception:
        return False

smoke_tests = [
    {
        'id': 'minigalaxy',
        'title': 'Minigalaxy GOG Client',
        'script': str(BENCH / 'launch_minigalaxy.sh'),
        'match': 'minigalaxy',
        'log': '/home/gamer/logs/minigalaxy.log'
    },
    {
        'id': 'heroic',
        'title': 'Heroic Games Launcher',
        'script': str(BENCH / 'launch_heroic.sh'),
        'match': 'heroic',
        'log': '/home/gamer/logs/heroic.log'
    },
    {
        'id': 'lutris',
        'title': 'Lutris Gaming Hub',
        'script': str(BENCH / 'launch_lutris.sh'),
        'match': 'lutris',
        'log': '/home/gamer/logs/lutris.log'
    },
    {
        'id': 'steam',
        'title': 'Steam Native Client',
        'script': str(BENCH / 'launch_steam.sh'),
        'match': 'steam',
        'log': '/home/gamer/logs/steam.log'
    },
    {
        'id': 'ea_app',
        'title': 'EA App Wine Runner',
        'script': str(BENCH / 'launch_ea.sh'),
        'match': 'ea',
        'log': '/home/gamer/logs/ea.log'
    }
]

smoke_results = {}
for test in smoke_tests:
    tid = test['id']
    script = test['script']
    match_str = test['match'].lower()
    log_file = Path(test['log'])
    shot_file = SCREENSHOTS / f'{tid}_smoke.png'
    
    print(f'  Testing storefront GUI: {test["title"]}...', flush=True)
    if not os.path.exists(script):
        smoke_results[tid] = {'tested': False, 'error': 'Launch script missing'}
        continue
        
    proc = subprocess.Popen(['runuser', '-u', 'gamer', '--', 'env', '-u', 'DBUS_SESSION_BUS_ADDRESS',
                             'PULSE_SERVER=' + (BENCH / 'pulse-server-path').read_text().strip(),
                             'dbus-run-session', '--', '/bin/bash', script], start_new_session=True)
    window_found = False
    matching_titles = []
    
    # Steam cold setup often takes longer than storefront startup.
    max_polls = 60 if tid == 'steam' else 20 if tid == 'heroic' else 12
    for _ in range(max_polls):
        time.sleep(1.5)
        wins = get_wm_windows()
        for w in wins:
            if match_str in w.lower():
                window_found = True
                matching_titles.append(w.strip())
        if window_found:
            break
            
    # Capture screen regardless of window detection to document actual virtual display state
    all_windows = get_wm_windows()
    error_shot = None
    if tid == 'steam':
        error_windows = [line for line in all_windows if 'error' in line.lower()]
        if error_windows:
            subprocess.run(['wmctrl', '-ia', error_windows[-1].split()[0]],
                           capture_output=True, timeout=5, env=probe_env)
            time.sleep(1)
            error_path = SCREENSHOTS / 'steam_error.png'
            if capture_screenshot(error_path):
                error_shot = str(error_path)
    shot_captured = capture_screenshot(shot_file)
    
    # Read process log tail
    log_tail = []
    if log_file.exists():
        lines = [l.strip() for l in log_file.read_text(encoding='utf-8', errors='replace').splitlines() if l.strip()]
        log_tail = lines[-6:]
        
    smoke_results[tid] = {
        'tested': True,
        'pid': proc.pid,
        'window_mapped': window_found,
        'detected_windows': matching_titles,
        'all_windows': all_windows,
        'error_screenshot': error_shot,
        'screenshot_saved': shot_captured,
        'screenshot_path': str(shot_file) if shot_captured else None,
        'process_poll': proc.poll(),
        'log_tail': log_tail
    }
    
    # Terminate smoke test process so desktop stays clean
    try:
        os.killpg(proc.pid, signal.SIGTERM)
        time.sleep(1)
        if proc.poll() is None:
            os.killpg(proc.pid, signal.SIGKILL)
    except Exception:
        pass

subprocess.run(['wmctrl', '-a', '7BZ Dev Cloud Game'], capture_output=True, timeout=5,
               env=probe_env)
time.sleep(1)

# Compile comprehensive health and verification audit
def check_bin(name):
    path = shutil.which(name) or (f'/usr/games/{name}' if os.path.exists(f'/usr/games/{name}') else None)
    return {'installed': path is not None, 'path': path}

def epic_installed_apps():
    installed = Path('/home/gamer/.config/heroic/legendaryConfig/legendary/installed.json')
    try:
        data = json.loads(installed.read_text())
        return sorted(data) if isinstance(data, dict) else []
    except (OSError, ValueError):
        return []

try:
    gpu_facts = json.loads(subprocess.run(
        [sys.executable, str(BENCH / 'gpu_runtime.py'), 'diagnose'],
        capture_output=True, text=True, timeout=50, env=probe_env,
        check=True).stdout)
except (OSError, ValueError, subprocess.SubprocessError):
    gpu_facts = {}

report = {
    'health': health,
    'status': status,
    'nvenc_detected': 'encodes H264, H265 on nvenc' in serverlog,
    'gamepad_sockets_ready': '4 virtual gamepads ready' in serverlog,
    'server_pid': server.pid,
    'launcher_processes': subprocess.run(['pgrep', '-u', 'gamer', '-f', '/tmp/kaggle-game-dev-bench/dev_launcher.py'],
                                         capture_output=True, text=True).stdout.splitlines(),
    'launcher_screenshot': str(SCREENSHOTS / 'game_lab_home.png') if capture_screenshot(SCREENSHOTS / 'game_lab_home.png') else None,
    'desktop_processes': [line for line in subprocess.run(['ps', '-eo', 'user,pid,ppid,args'],
                                      capture_output=True, text=True).stdout.splitlines()
                          if any(name in line for name in ('Xvfb', 'openbox', 'tint2', 'dev_launcher', 'selkies-session'))][:20],
    'x11_probe': subprocess.run(['xdpyinfo', '-display', ':0'], capture_output=True,
                                text=True, timeout=10, env=probe_env).stderr[-800:],
    'xauthority_path': probe_env.get('XAUTHORITY'),
    'gamer_audio_probe': subprocess.run(['runuser', '-u', 'gamer', '--', 'env',
                                         'PULSE_SERVER=' + (BENCH / 'pulse-server-path').read_text().strip(),
                                         'pactl', 'info'], capture_output=True,
                                         text=True, timeout=10).returncode,
    'selkies_log_tail': (BENCH / 'selkies.log').read_text(errors='replace')[-3000:],
    'desktop_components': {
        'openbox': check_bin('openbox'),
        'tint2': check_bin('tint2'),
        'xterm': check_bin('xterm'),
        'pulseaudio': check_bin('pulseaudio')
    },
    'installed_executables': {
        'steam': check_bin('steam'),
        'heroic': check_bin('heroic'),
        'legendary': check_bin('legendary'),
        'minigalaxy': check_bin('minigalaxy'),
        'lutris': check_bin('lutris'),
        'wine': check_bin('wine'),
        'wine_ge': {'installed': any(os.path.exists(p) for p in ['/home/gamer/.config/heroic/tools/wine/Wine-GE-Proton8-26/bin/wine', *Path('/home/gamer/.config/heroic/tools/wine').glob('*/bin/wine')]), 'path': next((str(p) for p in ['/home/gamer/.config/heroic/tools/wine/Wine-GE-Proton8-26/bin/wine', *Path('/home/gamer/.config/heroic/tools/wine').glob('*/bin/wine')] if os.path.exists(str(p))), None)}
    },
    'empirical_gui_smoke_tests': smoke_results,
    'epic_library_probe': {
        'heroic_legendary_config': '/home/gamer/.config/heroic/legendaryConfig/legendary',
        'heroic_installed_apps': epic_installed_apps(),
        'standalone_legendary_config': '/home/gamer/.config/legendary',
        'interactive_account_test_required': True
    },
    'compatibility_matrix': {
        'native_linux_2d_and_cpu_games': 'Fully supported (Mesa llvmpipe software rasterizer)',
        'native_linux_opengl_games': 'NVIDIA OpenGL through VirtualGL' if gpu_facts.get('virtualgl') else 'OpenGL renderer requires live verification',
        'native_vulkan_and_dxvk_games': 'Vulkan X11 presentation verified' if gpu_facts.get('vulkan_window') else 'Vulkan X11 presentation unavailable in this session',
        'windows_games_under_wine_proton': 'DXVK and WineD3D available' if gpu_facts.get('vulkan_window') else 'WineD3D on NVIDIA OpenGL' if gpu_facts.get('virtualgl') else 'Graphics backend requires live verification',
        'epic_games_support': 'Heroic and Legendary installed; individual game launch needs authenticated interactive testing',
        'controller_support': '4 Xbox 360 userspace virtual gamepads via interposer',
        'audio_support': 'PulseAudio stereo monitor -> Selkies Opus 128 kbps',
        'video_support': 'Tesla T4 NVENC hardware H.264 4:2:0 encode at 60 FPS'
    },
    'gpu_runtime': gpu_facts,
    'stream_config': {'codec': 'H.264 4:2:0 8-bit', 'websocket_queue': os.environ.get('GAMELAB_WS_QUEUE', '12')}
}

try:
    listing = subprocess.run([sys.executable, '-m', 'jupyter', 'server', 'list', '--json'],
                             capture_output=True, text=True, timeout=15).stdout
    for line in listing.splitlines():
        if not line.lstrip().startswith('{'):
            continue
        item = json.loads(line)
        base = item.get('url', '').rstrip('/') + '/'
        token = item.get('token', '')
        if not base.startswith(('http://127.0.0.1:', 'http://localhost:')):
            continue
        candidate = base + 'proxy/8080/api/health'
        if token:
            candidate += '?token=' + token
        with urllib.request.urlopen(candidate, timeout=5) as response:
            report['jupyter_proxy_local'] = response.status == 200 and response.read() == b'OK'
        break
except Exception as exc:
    report['jupyter_proxy_local'] = False
    report['jupyter_proxy_error'] = type(exc).__name__

# Persist reports into output directory
Path('/kaggle/working/game_dev_bench_health.json').write_text(json.dumps(report, indent=2))
print('================== HEALTH & DIAGNOSTIC REPORT ==================')
print(json.dumps(report, indent=2))
print('================================================================')
preferred_port = '8080'
direct_script = BENCH / 'start_direct_stream.sh'
if direct_script.is_file() and (BENCH / 'gpu-xorg-ready').exists():
    direct = subprocess.run(['bash', str(direct_script)], capture_output=True,
                            text=True, timeout=30)
    if direct.returncode == 0:
        preferred_port = '8081'
        print('Direct NVIDIA desktop ready on port 8081.', flush=True)
    else:
        print('Direct NVIDIA desktop unavailable:', (direct.stdout + direct.stderr)[-700:], flush=True)
(BENCH / 'preferred-stream-port').write_text(preferred_port, encoding='utf-8')
print('7BZ Dev Cloud Game is running in the active session.')


In [ ]:
from IPython.display import display, HTML
from pathlib import Path
port_file = Path('/tmp/kaggle-game-dev-bench/preferred-stream-port')
stream_port = port_file.read_text().strip() if port_file.is_file() else '8080'
html = r"""
<div style="font:16px system-ui, -apple-system, sans-serif;padding:24px;border-radius:16px;background:#0d1729;color:#f4f8ff;max-width:760px;box-shadow:0 8px 24px rgba(0,0,0,0.5)">
  <h2 style="margin-top:0;color:#4dcdf2;display:flex;align-items:center;gap:10px">
    <span>🎮</span> 7BZ Dev Cloud Game
  </h2>
  <p style="color:#b0c4de;line-height:1.5">
    Controller-first cloud game library with <b>Steam</b>, <b>Heroic</b>, <b>Legendary</b>, <b>Minigalaxy</b>, and <b>Lutris</b>.
  </p>

  <div style="background:#132238;padding:16px;border-radius:12px;margin:16px 0;border-left:4px solid #4dcdf2">
    <b>Connection Steps:</b>
    <ol style="margin:8px 0 0 18px;padding:0;color:#c8daf0;line-height:1.6">
      <li>In Kaggle, open <b>Run → Kaggle Jupyter Server</b> and copy the <b>VSCode Compatible URL</b>.</li>
      <li>Paste it into the field below and click <b>Connect to Stream</b>.</li>
      <li>Open the 7BZ HUD and select <b>Audio Unmute</b> for Opus stereo audio.</li>
      <li>Set your local keyboard layout to English (EN), then use mouse, keyboard, or gamepad navigation.</li>
    </ol>
  </div>

  <div style="margin:16px 0">
    <input id="kaggle-base" type="text" style="width:96%;padding:12px;border-radius:8px;border:1px solid #233b5d;background:#080e1a;color:#fff;font-size:14px" placeholder="Paste VSCode Compatible URL (https://...)">
    <button id="kaggle-open" style="margin-top:12px;padding:12px 24px;border-radius:8px;border:none;background:#4dcdf2;color:#081325;font-weight:bold;font-size:15px;cursor:pointer">Connect to Stream</button>
  </div>
  <p id="kaggle-result" style="font-size:13px;color:#9bdcf5;margin-top:6px"></p>

  <div style="border-top:1px solid #1a2a44;padding-top:14px;margin-top:18px;font-size:13px;color:#8ba2c4">
    <b>Empirical Hardware & Security Reality:</b>
    <ul style="margin:6px 0 0 16px;padding:0;line-height:1.5">
      <li><b>User state:</b> Storefronts run as <code>gamer</code>; their temporary credentials and caches are under <code>/home/gamer</code>.</li>
      <li><b>Storefront EULA:</b> Valve's Subscriber Agreement is presented to the user directly in the Steam GUI on first launch.</li>
      <li><b>GPU path:</b> T4 NVENC H.264 streaming and VirtualGL <code>egl0</code> offscreen 3D are verified. DXVK remains capability-gated.</li>
      <li><b>Custom Builds:</b> Upload test Linux builds into <code style="color:#4dcdf2">/tmp/game-builds</code> via Selkies Files panel.</li>
    </ul>
  </div>
</div>
<script>
document.getElementById('kaggle-open').onclick = function () {
  try {
    const raw = document.getElementById('kaggle-base').value.trim();
    if (!raw) throw new Error('Please paste your URL.');
    const u = new URL(raw);
    let p = u.pathname.replace(/\/(tree|lab|notebooks)(\/.*)?$/, '/');
    if (!p.endsWith('/')) p += '/';
    u.pathname = p + 'proxy/__STREAM_PORT__/';
    const target = u.toString();
    document.getElementById('kaggle-result').textContent = 'Opening authenticated Jupyter proxy route...';
    window.open(target, '_blank', 'noopener,noreferrer');
  } catch (e) {
    document.getElementById('kaggle-result').textContent = 'Please enter a valid https:// Kaggle Jupyter Server URL.';
  }
};
</script>
"""
display(HTML(html.replace('__STREAM_PORT__', stream_port)))
